# EVTEAS-Py 2.1.0 — Estudo de Viabilidade Técnica, Econômica, Ambiental e Social de projetos de piscicultura

Notebook autocontido do *framework* EVTEAS-Py (APÊNDICE A da dissertação). Executa no Google Colab sem
instalação local: a Seção 1 grava o código-fonte e as demais seções conduzem o estudo.

**Sequência de uso**

1. **Seção 1** — instala as dependências e grava o código do EVTEAS-Py (execute uma vez por sessão).
2. **Seção 2** — informe as entradas do projeto. **Nenhuma entrada vem preenchida**: o estudo só é executado
   quando todas as entradas obrigatórias estiverem informadas. Escolha o modo de entrada na célula 2.0:
   formulários desta seção, planilha `.xlsx`, arquivo de premissas `.json` de uma execução anterior ou
   assistente interativo.
3. **Seção 3** — valida as entradas e executa o estudo (determinístico, Monte Carlo, sensibilidade,
   valores críticos, cenários, decisão e, se declaradas, alternativas por TOPSIS).
4. **Seção 4** — verificação (invariantes e testes automatizados).
5. **Seção 5** — tabela de marcadores: cada `[E-xx]` (entrada) e `[R-xx]` (resultado) do texto da
   dissertação recebe o valor desta execução.
6. **Seção 6** — baixa todas as saídas em um `.zip` (Excel, JSON, Markdown, CSV de marcadores e figuras).
7. **Seção 7** (opcional) — preenche automaticamente os marcadores do `.docx` da dissertação.
8. **Seção 8** — somente **após** os questionários de validação (Apêndices C e D): pesos Likert dos especialistas.

Menu *Ambiente de execução → Executar tudo* executa as seções em ordem; a execução para na Seção 3 e
lista as entradas pendentes enquanto houver campos obrigatórios em branco.

## 1 Preparação do ambiente e código-fonte

In [ ]:
#@title 1.1 Instalar dependências { display-mode: "form" }
%pip install -q python-docx openpyxl

In [ ]:
#@title 1.2 Criar a pasta do pacote { display-mode: "form" }
import os
os.makedirs("evteas_py", exist_ok=True)
print("Pasta evteas_py pronta.")

### 1.3 Código-fonte do EVTEAS-Py

Cada célula grava um módulo do pacote (`%%writefile`). O código pode ser lido e auditado aqui; não é necessário editá-lo para realizar o estudo.

#### Módulo `config` — estrutura das entradas, parâmetros do método, conversão de taxas e formatação

In [ ]:
%%writefile evteas_py/config.py
"""EVTEAS-Py — módulo config.

Premissas do estudo organizadas em blocos (técnico, econômico, ambiental, social, governança,
pesos, decisão e Monte Carlo), registro da origem de cada premissa (Seção 3.6) e utilitários
de acesso por caminho ("bloco.campo"), conversão de taxas e formatação em português do Brasil.
"""
from __future__ import annotations

import copy
import json
import math
from dataclasses import asdict, dataclass, field, fields, is_dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np

VERSAO = "2.1.0"

# Categorias de origem das premissas (Seção 3.6)
CATEGORIAS_FONTE = [
    "dado histórico", "dado secundário", "cotação", "valor normativo", "parâmetro bibliográfico",
    "empreendimento análogo", "especialista", "estimativa do autor",
]


# ---------------------------------------------------------------------------------------------
# Utilitários numéricos
# ---------------------------------------------------------------------------------------------
def safe_div(a, b, padrao=0.0):
    """Divisão protegida (escalar ou vetorizada): devolve ``padrao`` quando o divisor é zero."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(b != 0, a / np.where(b != 0, b, 1.0), padrao)
    return r if r.ndim else float(r)


def taxa_anual_para_mensal(taxa_aa):
    """Taxa efetiva anual (fração) -> taxa efetiva mensal equivalente."""
    return (1.0 + np.asarray(taxa_aa, dtype=float)) ** (1.0 / 12.0) - 1.0


def taxa_mensal_para_anual(taxa_am):
    """Taxa efetiva mensal (fração) -> taxa efetiva anual equivalente."""
    return (1.0 + np.asarray(taxa_am, dtype=float)) ** 12.0 - 1.0


# ---------------------------------------------------------------------------------------------
# Formatação em português do Brasil
# ---------------------------------------------------------------------------------------------
def fmt_num(v, casas: int = 2) -> str:
    """1234567.891 -> '1.234.567,89'; valores ausentes -> 'n/d'."""
    if v is None:
        return "n/d"
    try:
        v = float(v)
    except (TypeError, ValueError):
        return str(v)
    if not math.isfinite(v):
        return "n/d"
    s = f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "−" + s[1:] if s.startswith("-") else s


def fmt_rs(v, casas: int = 0) -> str:
    """Valor monetário: 'R$ 1.234' ou '−R$ 1.234'."""
    if v is None or not math.isfinite(float(v)):
        return "n/d"
    s = "R$ " + fmt_num(abs(v), casas)
    return ("−" + s) if v < 0 else s


def fmt_pct(v, casas: int = 1) -> str:
    """Fração -> percentual: 0.1234 -> '12,3%'."""
    if v is None or not math.isfinite(float(v)):
        return "n/d"
    return fmt_num(float(v) * 100, casas) + "%"


# ---------------------------------------------------------------------------------------------
# Origem das premissas e distribuições de incerteza
# ---------------------------------------------------------------------------------------------
@dataclass
class Fonte:
    """Origem declarada de uma premissa (rastreabilidade, Seção 3.6)."""
    categoria: str = "estimativa do autor"
    referencia: str = ""
    observacao: str = ""

    def __post_init__(self):
        if self.categoria not in CATEGORIAS_FONTE:
            raise ValueError(f"Categoria de fonte inválida: {self.categoria!r}; use uma de {CATEGORIAS_FONTE}")


@dataclass
class Distribuicao:
    """Distribuição de uma premissa incerta (Oliveira; Medeiros Neto, 2012).

    tipo: 'triangular' (mínimo, mais provável, máximo) ou 'uniforme' (mínimo, máximo).
    """
    tipo: str = "triangular"
    minimo: float = 0.0
    mais_provavel: Optional[float] = None
    maximo: float = 0.0

    def __post_init__(self):
        if self.tipo not in ("triangular", "uniforme"):
            raise ValueError("tipo deve ser 'triangular' ou 'uniforme'")
        if self.tipo == "triangular":
            if self.mais_provavel is None:
                raise ValueError("A distribuição triangular exige o valor mais provável")
            if not (self.minimo <= self.mais_provavel <= self.maximo):
                raise ValueError(f"Triangular inválida: {self.minimo} <= {self.mais_provavel} <= {self.maximo}")
        elif self.minimo > self.maximo:
            raise ValueError("Uniforme inválida: mínimo > máximo")

    def media(self) -> float:
        if self.tipo == "triangular":
            return (self.minimo + self.mais_provavel + self.maximo) / 3.0
        return (self.minimo + self.maximo) / 2.0


# ---------------------------------------------------------------------------------------------
# Blocos de configuração
# ---------------------------------------------------------------------------------------------
CAPACIDADE_SUPORTE_PADRAO = {  # kg/m³ por sistema produtivo (Kubitza, 2017; valores de referência)
    "extensivo": 0.3, "semi-intensivo": 1.2, "intensivo": 8.0, "superintensivo": 40.0,
}


# Convenção: campos com valor None são ENTRADAS DO ESTUDO e não têm valor predefinido; devem ser
# informados pelo usuário (formulário do Colab, planilha de entradas ou assistente interativo).
# Campos com valor numérico neutro (0, 1 ou 100) são opcionais ou fatores de incerteza cujo valor
# determinístico é neutro. O registro completo das entradas, com marcadores [E-xx], está em entradas.py.
N = Optional[float]


@dataclass
class TecnicoConfig:
    especie: Optional[str] = None
    sistema_produtivo: Optional[str] = None
    area_lamina_m2: N = None
    profundidade_media_m: N = None
    numero_tanques: Optional[int] = None
    tanques_ativos: Optional[int] = None
    capacidade_suporte_kg_m3: float = 0.0          # opcional: 0 = referência do sistema produtivo
    peso_inicial_g: N = None
    peso_final_g: N = None
    ciclo_dias: Optional[int] = None
    ciclos_ano: N = None
    produtividade_esperada_kg_m2_ciclo: float = 0.0  # opcional: 0 = sem restrição por área
    alevinos_por_ciclo: float = 0.0                # opcional: 0 = estocagem dimensionada pelo modelo
    fcr: N = None
    mortalidade_pct: N = None
    desempenho_crescimento_pct: float = 100.0      # fator de incerteza (crescimento realizado / esperado, %)


@dataclass
class ItemCapex:
    descricao: str
    valor: float
    vida_util_anos: float = 10.0


@dataclass
class Produto:
    nome: str
    participacao: float   # fração da produção (soma = 1)
    fator_preco: float    # preço relativo ao preço-base de venda


@dataclass
class Tributo:
    nome: str
    aliquota_pct: float
    base: str = "faturamento"   # 'faturamento', 'lucro' ou 'folha'


@dataclass
class EconomicoConfig:
    tipo_organizacao: Optional[str] = None          # 'cooperativa', 'associação' ou 'empresa'
    capex: List[ItemCapex] = field(default_factory=list)
    capex_fator: float = 1.0                        # fator de incerteza do CAPEX (AACE Classe 5)
    capital_giro: N = None
    valor_residual_pct: N = None                    # do CAPEX, recuperado no fim do horizonte
    fomento_nao_reembolsavel_pct: N = None          # do CAPEX (perspectiva do beneficiário)
    preco_venda_kg: N = None
    mix_produtos: List[Produto] = field(default_factory=list)   # opcional: vazio = produto único
    receita_servicos_mes: float = 0.0               # opcional
    custo_racao_kg: N = None
    custo_alevino_milheiro: N = None
    tarifa_energia_kwh: N = None
    outros_variaveis_kg: N = None                   # embalagem, gelo, transporte e comercialização (R$/kg)
    mao_obra_mes: N = None                          # retiradas/salários (R$/mês)
    encargos_pct: N = None
    cooperados_trabalhadores: Optional[int] = None
    perc_sobras_trabalhadores: N = None             # % das sobras distribuídas aos trabalhadores
    manutencao_pct_capex_ano: N = None
    administrativo_mes: N = None
    seguros_outros_fixos_mes: N = None
    custos_fixos_fator: float = 1.0                 # fator de incerteza dos custos fixos
    tributos: List[Tributo] = field(default_factory=list)
    tma_aa_pct: N = None
    horizonte_anos: Optional[int] = None
    rampa_inicial_pct: N = None
    rampa_meses: Optional[int] = None
    ano_regime: Optional[int] = None


CHAVES_CHECKLIST_AMBIENTAL = ["licenca_ambiental", "outorga_agua", "app_respeitada", "tratamento_efluentes",
                              "monitoramento_agua", "car", "plano_residuos", "prevencao_escape", "energia_renovavel"]
CHAVES_GOVERNANCA = ["estatuto_registrado", "assembleias_regulares", "conselho_fiscal", "prestacao_contas_publica",
                     "plano_negocios", "registro_premissas_auditavel", "canal_denuncia"]


@dataclass
class AmbientalConfig:
    renovacao_agua_pct_dia: N = None
    enchimentos_ano: N = None                       # drenagens/enchimentos completos por ano
    evaporacao_mm_dia: N = None
    infiltracao_mm_dia: N = None
    metodo_tratamento: Optional[str] = None
    remocao_n_tratamento_pct: N = None
    remocao_p_tratamento_pct: N = None
    proteina_racao_pct: N = None
    fosforo_racao_pct: N = None
    nitrogenio_peixe_pct: N = None
    fosforo_peixe_pct: N = None
    classe_corpo_receptor: Optional[str] = None
    n_max_mg_l: N = None                            # limite da classe do corpo receptor (CONAMA 357/2005)
    n_natural_mg_l: N = None
    p_max_mg_l: N = None
    p_natural_mg_l: N = None
    fonte_energia: Optional[str] = None             # 'rede', 'solar' ou 'biomassa'
    fracao_renovavel_pct: N = None                  # parcela da energia de fonte própria renovável
    consumo_kwh_kg: N = None                        # aeração e bombeamento por kg produzido
    fator_emissao_rede_kg_kwh: N = None
    fator_emissao_racao_kgco2e_kg: N = None
    diesel_l_ano: N = None
    fator_emissao_diesel_kg_l: N = None
    distancia_app_m: N = None
    distancia_minima_app_m: N = None
    checklist: Dict[str, Optional[bool]] = field(default_factory=lambda: {k: None for k in CHAVES_CHECKLIST_AMBIENTAL})


@dataclass
class SocialConfig:
    empregos_diretos: Optional[int] = None
    empregos_indiretos: Optional[int] = None
    mao_obra_local_pct: N = None
    compras_locais_pct: N = None
    salario_minimo: N = None
    aderencia_nr: Optional[bool] = None             # Normas Regulamentadoras (segurança do trabalho)
    programa_capacitacao: Optional[bool] = None
    participacao_mulheres_pct: N = None
    relacao_comunidade: Optional[str] = None        # 'inexistente', 'parcial' ou 'consolidada'
    canais_formais_comunicacao: Optional[bool] = None
    reunioes_comunidade_ano: Optional[int] = None
    conflitos_registrados_ano: Optional[int] = None


@dataclass
class GovernancaConfig:
    itens: Dict[str, Optional[bool]] = field(default_factory=lambda: {k: None for k in CHAVES_GOVERNANCA})


@dataclass
class LeanGreenConfig:
    """Referências declaradas para quantificar desperdícios evitáveis (Lawrence et al., 2023)."""
    fcr_referencia: N = None
    mortalidade_referencia_pct: N = None
    consumo_kwh_kg_referencia: N = None


# Parâmetros do método (Capítulo 3): têm valores documentados e podem ser alterados pelo usuário.
@dataclass
class PesosConfig:
    metodo: str = "preset"                           # 'preset', 'likert' ou 'ahp'
    preset: Dict[str, float] = field(default_factory=lambda: {
        "tecnica": 0.25, "economica": 0.35, "ambiental": 0.20, "social": 0.20})
    # Escores Likert (1 a 5) por avaliador e por KPI (Quadro 7); preenchidos após os questionários
    likert: Dict[str, Dict[str, int]] = field(default_factory=dict)
    # Matriz AHP 4 x 4 (ordem: técnica, econômica, ambiental, social), escala de Saaty (1/9 a 9)
    ahp: List[List[float]] = field(default_factory=list)


@dataclass
class DecisaoConfig:
    aplicar_vetos: bool = True
    lso_minimo: float = 40.0
    limiar_indice_viavel: float = 0.60
    limiar_indice_ressalvas: float = 0.50
    prob_vpl_positivo_minima: float = 0.70


@dataclass
class MonteCarloConfig:
    iteracoes: int = 10_000
    seed: int = 20260612
    nivel_confianca: float = 0.95


@dataclass
class EVTEASConfig:
    projeto: Optional[str] = None
    localizacao: str = ""
    responsavel: str = ""
    versao: str = VERSAO
    tecnico: TecnicoConfig = field(default_factory=TecnicoConfig)
    economico: EconomicoConfig = field(default_factory=EconomicoConfig)
    ambiental: AmbientalConfig = field(default_factory=AmbientalConfig)
    social: SocialConfig = field(default_factory=SocialConfig)
    governanca: GovernancaConfig = field(default_factory=GovernancaConfig)
    lean_green: LeanGreenConfig = field(default_factory=LeanGreenConfig)
    pesos: PesosConfig = field(default_factory=PesosConfig)
    decisao: DecisaoConfig = field(default_factory=DecisaoConfig)
    monte_carlo: MonteCarloConfig = field(default_factory=MonteCarloConfig)
    distribuicoes: Dict[str, Distribuicao] = field(default_factory=dict)  # "bloco.campo" -> Distribuicao
    fontes: Dict[str, Fonte] = field(default_factory=dict)                # "bloco.campo" -> Fonte


# ---------------------------------------------------------------------------------------------
# Acesso por caminho e serialização
# ---------------------------------------------------------------------------------------------
def obter_por_caminho(cfg: EVTEASConfig, caminho: str):
    """Lê 'bloco.campo' da configuração (ex.: 'economico.preco_venda_kg')."""
    obj = cfg
    for parte in caminho.split("."):
        if isinstance(obj, dict):
            obj = obj[parte]
        else:
            if not hasattr(obj, parte):
                raise KeyError(f"Caminho inexistente na configuração: {caminho!r}")
            obj = getattr(obj, parte)
    return obj


def definir_por_caminho(cfg: EVTEASConfig, caminho: str, valor) -> None:
    """Altera 'bloco.campo' da configuração, preservando o tipo numérico original."""
    partes = caminho.split(".")
    obj = cfg
    for parte in partes[:-1]:
        obj = obj[parte] if isinstance(obj, dict) else getattr(obj, parte)
    final = partes[-1]
    atual = obj[final] if isinstance(obj, dict) else getattr(obj, final)
    if atual is None:
        from .entradas import tipo_do_caminho
        tipo = tipo_do_caminho(caminho)
        atual = {"bool": False, "int": 0, "num": 0.0}.get(tipo, valor)
    if isinstance(atual, bool):
        valor = bool(valor)
    elif isinstance(atual, int) and not isinstance(atual, bool):
        valor = int(round(float(valor)))
    elif isinstance(atual, float):
        valor = float(valor)
    if isinstance(obj, dict):
        obj[final] = valor
    else:
        setattr(obj, final, valor)


def copiar(cfg: EVTEASConfig) -> EVTEASConfig:
    return copy.deepcopy(cfg)


def config_para_dict(cfg: EVTEASConfig) -> Dict[str, Any]:
    return asdict(cfg)


def _construir(tipo, dados):
    """Reconstrói dataclasses aninhadas a partir de dicionários (JSON)."""
    if is_dataclass(tipo):
        kwargs = {}
        for f in fields(tipo):
            if f.name not in dados:
                continue
            kwargs[f.name] = _construir_campo(f.type, dados[f.name], f.name)
        return tipo(**kwargs)
    return dados


_LISTAS = {"capex": ItemCapex, "mix_produtos": Produto, "tributos": Tributo}
_BLOCOS = {"tecnico": TecnicoConfig, "economico": EconomicoConfig, "ambiental": AmbientalConfig,
           "social": SocialConfig, "governanca": GovernancaConfig, "lean_green": LeanGreenConfig, "pesos": PesosConfig,
           "decisao": DecisaoConfig, "monte_carlo": MonteCarloConfig}


def _construir_campo(tipo, valor, nome):
    if nome in _BLOCOS:
        return _construir(_BLOCOS[nome], valor)
    if nome in _LISTAS:
        return [_LISTAS[nome](**v) for v in valor]
    if nome == "distribuicoes":
        return {k: Distribuicao(**v) for k, v in valor.items()}
    if nome == "fontes":
        return {k: Fonte(**v) for k, v in valor.items()}
    return valor


def config_de_dict(dados: Dict[str, Any]) -> EVTEASConfig:
    return _construir(EVTEASConfig, dados)


def salvar_config(cfg: EVTEASConfig, caminho: str) -> Path:
    """Grava as premissas em JSON (versionamento das premissas de cada estudo)."""
    p = Path(caminho)
    p.write_text(json.dumps(config_para_dict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
    return p


def carregar_config(caminho: str) -> EVTEASConfig:
    return config_de_dict(json.loads(Path(caminho).read_text(encoding="utf-8")))


def capacidade_suporte(cfg: EVTEASConfig) -> float:
    """Capacidade de suporte informada ou, se não informada (0), a referência do sistema produtivo."""
    t = cfg.tecnico
    if t.capacidade_suporte_kg_m3 and t.capacidade_suporte_kg_m3 > 0:
        return float(t.capacidade_suporte_kg_m3)
    if t.sistema_produtivo not in CAPACIDADE_SUPORTE_PADRAO:
        raise ValueError(f"Sistema produtivo inválido: {t.sistema_produtivo!r}")
    return CAPACIDADE_SUPORTE_PADRAO[t.sistema_produtivo]


def validar_config(cfg: EVTEASConfig) -> List[str]:
    """Checagens das entradas; devolve a lista de problemas (vazia = ok).

    Primeiro verifica se todas as entradas obrigatórias foram informadas (nenhum resultado é
    calculado com entradas em branco); depois, a consistência entre elas.
    """
    from .entradas import entradas_faltantes
    faltam = entradas_faltantes(cfg)
    if faltam:
        return [f"Entrada obrigatória não preenchida: {f}" for f in faltam]
    t, e, a, s = cfg.tecnico, cfg.economico, cfg.ambiental, cfg.social
    p = []
    if t.area_lamina_m2 <= 0:
        p.append("Área de lâmina d'água deve ser positiva")
    if t.profundidade_media_m <= 0:
        p.append("Profundidade média deve ser positiva")
    if not (0 <= t.tanques_ativos <= t.numero_tanques) or t.numero_tanques <= 0:
        p.append("Viveiros ativos devem estar entre 0 e o número total de viveiros")
    if t.peso_final_g <= t.peso_inicial_g:
        p.append("Peso de abate deve superar o peso inicial")
    if t.ciclo_dias <= 0 or t.ciclos_ano <= 0 or t.ciclos_ano > 365 / max(t.ciclo_dias, 1) + 1e-9:
        p.append("Ciclos por ano incompatíveis com a duração do ciclo (máximo = 365 / dias do ciclo)")
    if not (0 <= t.mortalidade_pct < 100):
        p.append("Mortalidade deve estar em [0, 100)")
    if t.fcr <= 0:
        p.append("FCR deve ser positivo")
    if t.sistema_produtivo not in CAPACIDADE_SUPORTE_PADRAO:
        p.append(f"Sistema produtivo deve ser um de {list(CAPACIDADE_SUPORTE_PADRAO)}")
    if not e.capex or sum(i.valor for i in e.capex) <= 0:
        p.append("Informe ao menos um item de CAPEX com valor positivo")
    for item in e.capex:
        if item.valor < 0 or item.vida_util_anos <= 0:
            p.append(f"Item de CAPEX inválido (valor negativo ou vida útil não positiva): {item.descricao}")
    if e.mix_produtos and abs(sum(x.participacao for x in e.mix_produtos) - 1.0) > 1e-6:
        p.append("As participações do mix de produtos devem somar 1")
    if e.tipo_organizacao not in ("cooperativa", "associação", "empresa"):
        p.append("Tipo de organização deve ser 'cooperativa', 'associação' ou 'empresa'")
    if e.horizonte_anos <= 0 or e.horizonte_anos > 40:
        p.append("Horizonte deve estar entre 1 e 40 anos")
    if not (1 <= e.ano_regime <= e.horizonte_anos):
        p.append("Ano de regime deve estar entre 1 e o horizonte de análise")
    if a.n_max_mg_l <= a.n_natural_mg_l or a.p_max_mg_l <= a.p_natural_mg_l:
        p.append("Concentração máxima deve superar a natural (pegada hídrica cinza indefinida)")
    if a.fonte_energia not in ("rede", "solar", "biomassa"):
        p.append("Fonte de energia deve ser 'rede', 'solar' ou 'biomassa'")
    if s.relacao_comunidade not in ("inexistente", "parcial", "consolidada"):
        p.append("Relação com a comunidade deve ser 'inexistente', 'parcial' ou 'consolidada'")
    if cfg.pesos.metodo not in ("preset", "likert", "ahp"):
        p.append("Método de pesos deve ser 'preset', 'likert' ou 'ahp'")
    for caminho, d in cfg.distribuicoes.items():
        try:
            valor = float(obter_por_caminho(cfg, caminho))
        except (KeyError, AttributeError, TypeError):
            p.append(f"Distribuição aponta para caminho inexistente ou não informado: {caminho}")
            continue
        if not (d.minimo - 1e-9 <= valor <= d.maximo + 1e-9):
            p.append(f"O valor determinístico de {caminho} ({valor}) está fora da faixa da distribuição "
                     f"[{d.minimo}; {d.maximo}]")
        elif d.tipo == "triangular" and abs(d.mais_provavel - valor) > 1e-9 * max(1.0, abs(valor)):
            p.append(f"A moda da distribuição de {caminho} ({d.mais_provavel}) deve ser igual ao valor "
                     f"determinístico informado ({valor})")
    return p

#### Módulo `financeiro` — VPL, TIR com diagnóstico e payback

In [ ]:
%%writefile evteas_py/financeiro.py
"""EVTEAS-Py — módulo financeiro.

Indicadores de análise de investimentos vetorizados (cada linha é uma realização das premissas):
VPL, TIR com diagnóstico explícito e payback simples e descontado
(Casarotto Filho; Kopittke, 2007; Assaf Neto, 2014).
"""
from __future__ import annotations

from typing import Tuple

import numpy as np


def _2d(fluxos) -> np.ndarray:
    f = np.asarray(fluxos, dtype=float)
    return f[None, :] if f.ndim == 1 else f


def vpl(taxa, fluxos) -> np.ndarray:
    """Valor Presente Líquido por período: sum(F_t / (1 + i)^t), t = 0..T.

    ``taxa`` é escalar ou vetor (uma taxa por linha); ``fluxos`` tem forma (T+1,) ou (n, T+1).
    """
    f = _2d(fluxos)
    n, m = f.shape
    i = np.broadcast_to(np.asarray(taxa, dtype=float), (n,))[:, None]
    t = np.arange(m)[None, :]
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        desconto = (1.0 + i) ** (-t)
    return np.sum(f * desconto, axis=1)


def trocas_de_sinal(fluxos) -> np.ndarray:
    """Número de trocas de sinal do fluxo (ignorando zeros) — regra de sinais de Descartes."""
    f = _2d(fluxos)
    s = np.sign(f)
    trocas = np.zeros(f.shape[0], dtype=int)
    ultimo = np.zeros(f.shape[0])
    for t in range(f.shape[1]):
        st = s[:, t]
        mudou = (st != 0) & (ultimo != 0) & (st != ultimo)
        trocas += mudou
        ultimo = np.where(st != 0, st, ultimo)
    return trocas


def tir(fluxos, lo: float = -0.90, hi: float = 1.0, pontos: int = 160,
        iteracoes: int = 100) -> Tuple[np.ndarray, np.ndarray]:
    """TIR por varredura de malha + bisseção vetorizada, com diagnóstico explícito.

    Retorna (taxa por período, diagnóstico). A taxa é ``nan`` quando não há raiz no intervalo:
    nunca é forçada ao limite numérico do algoritmo. Diagnósticos: ``convencional`` (uma raiz e
    uma troca de sinal no fluxo), ``multiplas`` (mais de uma raiz na malha), ``multiplas_possiveis``
    (uma raiz, mas mais de uma troca de sinal — regra de Descartes) e ``sem_solucao`` (o VPL não
    muda de sinal no intervalo pesquisado).
    """
    f = _2d(fluxos)
    n = f.shape[0]
    malha = np.unique(np.concatenate([np.linspace(lo, -0.5, 12), np.linspace(-0.5, hi, pontos)]))
    valores = np.vstack([vpl(np.full(n, r), f) for r in malha]).T  # (n, pontos)
    troca = (valores[:, :-1] * valores[:, 1:]) <= 0
    n_raizes = troca.sum(axis=1)
    existe = n_raizes > 0
    k = np.where(existe, troca.argmax(axis=1), 0)
    a, b = malha[k].astype(float), malha[k + 1].astype(float)
    fa = valores[np.arange(n), k]
    for _ in range(iteracoes):
        m = (a + b) / 2
        fm = vpl(m, f)
        esquerda = (fa * fm) <= 0
        b = np.where(esquerda, m, b)
        a = np.where(esquerda, a, m)
        fa = np.where(esquerda, fa, fm)
    raiz = np.where(existe, (a + b) / 2, np.nan)
    trocas = trocas_de_sinal(f)
    diag = np.where(~existe, "sem_solucao",
                    np.where(n_raizes > 1, "multiplas",
                             np.where(trocas > 1, "multiplas_possiveis", "convencional")))
    return raiz, diag


def payback(fluxos, taxa=None) -> np.ndarray:
    """Período (fracionário) em que o saldo acumulado se torna não negativo.

    Sem ``taxa``: payback simples; com ``taxa``: payback descontado. ``nan`` se não houver
    recuperação dentro do horizonte. A interpolação é linear dentro do período de virada.
    """
    f = _2d(fluxos)
    n, m = f.shape
    if taxa is not None:
        i = np.broadcast_to(np.asarray(taxa, dtype=float), (n,))[:, None]
        f = f * (1.0 + i) ** (-np.arange(m)[None, :])
    acum = np.cumsum(f, axis=1)
    pos = acum >= 0
    # primeiro período a partir do qual o saldo nunca mais volta a ser negativo
    ultimo_negativo = np.where((~pos).any(axis=1), m - 1 - np.argmax((~pos)[:, ::-1], axis=1), -1)
    t = ultimo_negativo + 1
    recupera = (t < m) & (t > 0)
    tt = np.clip(t, 1, m - 1)
    antes = acum[np.arange(n), tt - 1]
    ganho = f[np.arange(n), tt]
    frac = np.where(ganho != 0, -antes / np.where(ganho != 0, ganho, 1.0), 0.0)
    pb = (tt - 1) + np.clip(frac, 0.0, 1.0)
    pb = np.where(t == 0, 0.0, pb)
    return np.where(recupera | (t == 0), pb, np.nan)

#### Módulo `entradas` — registro das entradas [E-xx], formulário, planilha e alternativas

In [ ]:
%%writefile evteas_py/entradas.py
"""EVTEAS-Py — módulo entradas.

Registro único das entradas do estudo (Quadro de premissas), com marcadores [E-xx] para o texto.
Nenhuma entrada do projeto tem valor predefinido: o usuário as informa pelo formulário do Colab,
pela planilha de entradas (.xlsx) ou pelo assistente interativo, e o estudo só é executado quando
todas as entradas obrigatórias estão preenchidas. Apenas os parâmetros do método (Monte Carlo,
pesos do preset e regra de decisão, descritos no Capítulo 3) vêm com valores documentados, que
podem ser alterados.
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

from .config import (CATEGORIAS_FONTE, CHAVES_CHECKLIST_AMBIENTAL, CHAVES_GOVERNANCA, Distribuicao, EVTEASConfig,
                     Fonte, ItemCapex, Produto, Tributo, copiar, definir_por_caminho, fmt_num, obter_por_caminho)


@dataclass(frozen=True)
class Entrada:
    marcador: str            # identificador no texto: [E-T03]
    var: str                 # nome do campo no formulário e na planilha
    caminho: str             # caminho na configuração ("tecnico.area_lamina_m2")
    rotulo: str
    unidade: str
    tipo: str                # 'num', 'int', 'texto', 'opcao' ou 'bool'
    grupo: str
    obrigatorio: bool = True
    opcoes: Tuple[str, ...] = ()
    vazio: Any = None        # valor adotado quando a entrada opcional fica em branco
    minimo: Optional[float] = None
    maximo: Optional[float] = None
    ajuda: str = ""
    metodo: bool = False     # parâmetro do método (valor documentado no Capítulo 3)
    padrao: Any = None       # valor documentado do parâmetro do método


def _E(*a, **k) -> Entrada:
    return Entrada(*a, **k)


GRUPOS = ["Identificação", "Técnica", "Econômica", "Ambiental", "Social", "Governança", "Lean-Green",
          "Parâmetros do método"]

ROTULOS_CHECKLIST = {
    "licenca_ambiental": "Licença ambiental da aquicultura (eliminatório)",
    "outorga_agua": "Outorga de direito de uso da água (eliminatório)",
    "app_respeitada": "Área de Preservação Permanente respeitada (eliminatório)",
    "tratamento_efluentes": "Tratamento de efluentes",
    "monitoramento_agua": "Monitoramento da qualidade da água",
    "car": "Cadastro Ambiental Rural (CAR)",
    "plano_residuos": "Plano de gestão de resíduos",
    "prevencao_escape": "Prevenção de escape de peixes",
    "energia_renovavel": "Uso de energia renovável",
}
ROTULOS_GOVERNANCA = {
    "estatuto_registrado": "Estatuto social registrado",
    "assembleias_regulares": "Assembleias regulares",
    "conselho_fiscal": "Conselho fiscal atuante",
    "prestacao_contas_publica": "Prestação de contas pública",
    "plano_negocios": "Plano de negócios aprovado",
    "registro_premissas_auditavel": "Registro auditável das premissas",
    "canal_denuncia": "Canal de denúncia",
}

ENTRADAS: List[Entrada] = [
    # Identificação
    _E("E-I01", "projeto", "projeto", "Nome do projeto", "—", "texto", "Identificação"),
    _E("E-I02", "localizacao", "localizacao", "Localização (município/UF)", "—", "texto", "Identificação", False, vazio=""),
    _E("E-I03", "responsavel", "responsavel", "Responsável pelo estudo", "—", "texto", "Identificação", False, vazio=""),
    # Técnica
    _E("E-T01", "especie", "tecnico.especie", "Espécie cultivada", "—", "texto", "Técnica"),
    _E("E-T02", "sistema_produtivo", "tecnico.sistema_produtivo", "Sistema produtivo", "—", "opcao", "Técnica",
       opcoes=("extensivo", "semi-intensivo", "intensivo", "superintensivo")),
    _E("E-T03", "area_lamina_m2", "tecnico.area_lamina_m2", "Área de lâmina d'água", "m²", "num", "Técnica", minimo=1),
    _E("E-T04", "profundidade_media_m", "tecnico.profundidade_media_m", "Profundidade média", "m", "num", "Técnica",
       minimo=0.1, maximo=20),
    _E("E-T05", "numero_tanques", "tecnico.numero_tanques", "Número de viveiros/tanques", "un.", "int", "Técnica", minimo=1),
    _E("E-T06", "tanques_ativos", "tecnico.tanques_ativos", "Viveiros/tanques em operação simultânea", "un.", "int",
       "Técnica", minimo=0),
    _E("E-T07", "capacidade_suporte_kg_m3", "tecnico.capacidade_suporte_kg_m3", "Capacidade de suporte", "kg/m³", "num",
       "Técnica", False, vazio=0.0, minimo=0, ajuda="Em branco: valor de referência do sistema produtivo (Kubitza, 2017)"),
    _E("E-T08", "peso_inicial_g", "tecnico.peso_inicial_g", "Peso inicial dos juvenis", "g", "num", "Técnica", minimo=0.01),
    _E("E-T09", "peso_final_g", "tecnico.peso_final_g", "Peso de abate", "g", "num", "Técnica", minimo=0.01),
    _E("E-T10", "ciclo_dias", "tecnico.ciclo_dias", "Duração do ciclo de engorda", "dias", "int", "Técnica",
       minimo=1, maximo=730),
    _E("E-T11", "ciclos_ano", "tecnico.ciclos_ano", "Ciclos por ano", "ciclos/ano", "num", "Técnica", minimo=0.01),
    _E("E-T12", "produtividade_esperada_kg_m2_ciclo", "tecnico.produtividade_esperada_kg_m2_ciclo",
       "Produtividade esperada por área", "kg/m²/ciclo", "num", "Técnica", False, vazio=0.0, minimo=0,
       ajuda="Em branco: sem restrição por área (vale só a capacidade de suporte)"),
    _E("E-T13", "alevinos_por_ciclo", "tecnico.alevinos_por_ciclo", "Juvenis estocados por ciclo", "un./ciclo", "num",
       "Técnica", False, vazio=0.0, minimo=0, ajuda="Em branco: estocagem dimensionada pelo modelo"),
    _E("E-T14", "fcr", "tecnico.fcr", "Conversão alimentar aparente (FCR)", "kg ração/kg", "num", "Técnica",
       minimo=0.1, maximo=10),
    _E("E-T15", "mortalidade_pct", "tecnico.mortalidade_pct", "Mortalidade no ciclo", "%", "num", "Técnica",
       minimo=0, maximo=99.9),
    # Econômica
    _E("E-E01", "tipo_organizacao", "economico.tipo_organizacao", "Tipo de organização", "—", "opcao", "Econômica",
       opcoes=("cooperativa", "associação", "empresa")),
    _E("E-E02", "capital_giro", "economico.capital_giro", "Capital de giro", "R$", "num", "Econômica", minimo=0),
    _E("E-E03", "valor_residual_pct", "economico.valor_residual_pct", "Valor residual ao fim do horizonte",
       "% do CAPEX", "num", "Econômica", minimo=0, maximo=100),
    _E("E-E04", "fomento_nao_reembolsavel_pct", "economico.fomento_nao_reembolsavel_pct",
       "Fomento não reembolsável", "% do CAPEX", "num", "Econômica", minimo=0, maximo=100),
    _E("E-E05", "preco_venda_kg", "economico.preco_venda_kg", "Preço de venda", "R$/kg", "num", "Econômica",
       minimo=0.01),
    _E("E-E06", "receita_servicos_mes", "economico.receita_servicos_mes", "Outras receitas (serviços)", "R$/mês", "num",
       "Econômica", False, vazio=0.0, minimo=0),
    _E("E-E07", "custo_racao_kg", "economico.custo_racao_kg", "Custo da ração", "R$/kg", "num", "Econômica", minimo=0.01),
    _E("E-E08", "custo_alevino_milheiro", "economico.custo_alevino_milheiro", "Custo do milheiro de juvenis", "R$/mil",
       "num", "Econômica", minimo=0),
    _E("E-E09", "tarifa_energia_kwh", "economico.tarifa_energia_kwh", "Tarifa de energia elétrica", "R$/kWh", "num",
       "Econômica", minimo=0),
    _E("E-E10", "outros_variaveis_kg", "economico.outros_variaveis_kg",
       "Outros custos variáveis (embalagem, gelo, frete, comercialização)", "R$/kg", "num", "Econômica", minimo=0),
    _E("E-E11", "mao_obra_mes", "economico.mao_obra_mes", "Retiradas/salários", "R$/mês", "num", "Econômica", minimo=0),
    _E("E-E12", "encargos_pct", "economico.encargos_pct", "Encargos sobre retiradas/salários", "%", "num", "Econômica",
       minimo=0, maximo=200),
    _E("E-E13", "cooperados_trabalhadores", "economico.cooperados_trabalhadores", "Cooperados/trabalhadores", "pessoas",
       "int", "Econômica", minimo=0),
    _E("E-E14", "perc_sobras_trabalhadores", "economico.perc_sobras_trabalhadores",
       "Sobras distribuídas aos trabalhadores", "%", "num", "Econômica", minimo=0, maximo=100),
    _E("E-E15", "manutencao_pct_capex_ano", "economico.manutencao_pct_capex_ano", "Manutenção", "% do CAPEX/ano", "num",
       "Econômica", minimo=0, maximo=100),
    _E("E-E16", "administrativo_mes", "economico.administrativo_mes", "Despesas administrativas", "R$/mês", "num",
       "Econômica", minimo=0),
    _E("E-E17", "seguros_outros_fixos_mes", "economico.seguros_outros_fixos_mes", "Seguros e outros custos fixos",
       "R$/mês", "num", "Econômica", minimo=0),
    _E("E-E18", "tributos_faturamento_pct", "economico.tributos#faturamento", "Tributos sobre o faturamento", "%",
       "num", "Econômica", minimo=0, maximo=100),
    _E("E-E19", "tributos_resultado_pct", "economico.tributos#lucro", "Tributos sobre o resultado", "%", "num",
       "Econômica", minimo=0, maximo=100),
    _E("E-E20", "tma_aa_pct", "economico.tma_aa_pct", "Taxa mínima de atratividade (TMA)", "% a.a.", "num", "Econômica",
       minimo=0, maximo=100),
    _E("E-E21", "horizonte_anos", "economico.horizonte_anos", "Horizonte de análise", "anos", "int", "Econômica",
       minimo=1, maximo=40),
    _E("E-E22", "rampa_inicial_pct", "economico.rampa_inicial_pct", "Produção inicial da rampa", "% do regime", "num",
       "Econômica", minimo=0, maximo=100),
    _E("E-E23", "rampa_meses", "economico.rampa_meses", "Duração da rampa de produção", "meses", "int", "Econômica",
       minimo=0, maximo=120),
    _E("E-E24", "ano_regime", "economico.ano_regime", "Ano de regime (indicadores anuais)", "ano", "int", "Econômica",
       minimo=1, maximo=40),
    # Ambiental
    _E("E-A01", "renovacao_agua_pct_dia", "ambiental.renovacao_agua_pct_dia", "Renovação de água", "% do volume/dia",
       "num", "Ambiental", minimo=0, maximo=100),
    _E("E-A02", "enchimentos_ano", "ambiental.enchimentos_ano", "Enchimentos completos dos viveiros", "por ano", "num",
       "Ambiental", minimo=0),
    _E("E-A03", "evaporacao_mm_dia", "ambiental.evaporacao_mm_dia", "Evaporação", "mm/dia", "num", "Ambiental", minimo=0),
    _E("E-A04", "infiltracao_mm_dia", "ambiental.infiltracao_mm_dia", "Infiltração", "mm/dia", "num", "Ambiental",
       minimo=0),
    _E("E-A05", "metodo_tratamento", "ambiental.metodo_tratamento", "Método de tratamento de efluentes", "—", "texto",
       "Ambiental"),
    _E("E-A06", "remocao_n_tratamento_pct", "ambiental.remocao_n_tratamento_pct", "Remoção de nitrogênio no tratamento",
       "%", "num", "Ambiental", minimo=0, maximo=100),
    _E("E-A07", "remocao_p_tratamento_pct", "ambiental.remocao_p_tratamento_pct", "Remoção de fósforo no tratamento", "%",
       "num", "Ambiental", minimo=0, maximo=100),
    _E("E-A08", "proteina_racao_pct", "ambiental.proteina_racao_pct", "Proteína bruta da ração", "%", "num", "Ambiental",
       minimo=0, maximo=100),
    _E("E-A09", "fosforo_racao_pct", "ambiental.fosforo_racao_pct", "Fósforo da ração", "%", "num", "Ambiental",
       minimo=0, maximo=100),
    _E("E-A10", "nitrogenio_peixe_pct", "ambiental.nitrogenio_peixe_pct", "Nitrogênio retido na biomassa", "% do peso",
       "num", "Ambiental", minimo=0, maximo=100),
    _E("E-A11", "fosforo_peixe_pct", "ambiental.fosforo_peixe_pct", "Fósforo retido na biomassa", "% do peso", "num",
       "Ambiental", minimo=0, maximo=100),
    _E("E-A12", "classe_corpo_receptor", "ambiental.classe_corpo_receptor", "Classe e ambiente do corpo receptor", "—",
       "texto", "Ambiental", ajuda="Ex.: classe 2, ambiente lótico (Resolução CONAMA nº 357/2005)"),
    _E("E-A13", "n_max_mg_l", "ambiental.n_max_mg_l", "Concentração máxima de N total (padrão da classe)", "mg/L", "num",
       "Ambiental", minimo=0),
    _E("E-A14", "n_natural_mg_l", "ambiental.n_natural_mg_l", "Concentração natural de N total", "mg/L", "num",
       "Ambiental", minimo=0),
    _E("E-A15", "p_max_mg_l", "ambiental.p_max_mg_l", "Concentração máxima de P total (padrão da classe)", "mg/L", "num",
       "Ambiental", minimo=0),
    _E("E-A16", "p_natural_mg_l", "ambiental.p_natural_mg_l", "Concentração natural de P total", "mg/L", "num",
       "Ambiental", minimo=0),
    _E("E-A17", "fonte_energia", "ambiental.fonte_energia", "Fonte principal de energia", "—", "opcao", "Ambiental",
       opcoes=("rede", "solar", "biomassa")),
    _E("E-A18", "fracao_renovavel_pct", "ambiental.fracao_renovavel_pct", "Energia renovável de fonte própria", "%",
       "num", "Ambiental", minimo=0, maximo=100),
    _E("E-A19", "consumo_kwh_kg", "ambiental.consumo_kwh_kg", "Consumo de energia (aeração e bombeamento)", "kWh/kg",
       "num", "Ambiental", minimo=0),
    _E("E-A20", "fator_emissao_rede_kg_kwh", "ambiental.fator_emissao_rede_kg_kwh", "Fator de emissão da rede elétrica",
       "kgCO2e/kWh", "num", "Ambiental", minimo=0),
    _E("E-A21", "fator_emissao_racao_kgco2e_kg", "ambiental.fator_emissao_racao_kgco2e_kg", "Fator de emissão da ração",
       "kgCO2e/kg", "num", "Ambiental", minimo=0),
    _E("E-A22", "diesel_l_ano", "ambiental.diesel_l_ano", "Consumo de diesel", "L/ano", "num", "Ambiental", minimo=0),
    _E("E-A23", "fator_emissao_diesel_kg_l", "ambiental.fator_emissao_diesel_kg_l", "Fator de emissão do diesel",
       "kgCO2e/L", "num", "Ambiental", minimo=0),
    _E("E-A24", "distancia_app_m", "ambiental.distancia_app_m", "Distância dos viveiros ao curso d'água", "m", "num",
       "Ambiental", minimo=0),
    _E("E-A25", "distancia_minima_app_m", "ambiental.distancia_minima_app_m", "Largura mínima da APP aplicável", "m",
       "num", "Ambiental", minimo=0),
] + [
    _E(f"E-A{26 + i:02d}", f"amb_{k}", f"ambiental.checklist.{k}", ROTULOS_CHECKLIST[k], "sim/não", "bool", "Ambiental")
    for i, k in enumerate(CHAVES_CHECKLIST_AMBIENTAL)
] + [
    # Social
    _E("E-S01", "empregos_diretos", "social.empregos_diretos", "Empregos diretos", "postos", "int", "Social", minimo=0),
    _E("E-S02", "empregos_indiretos", "social.empregos_indiretos", "Empregos indiretos", "postos", "int", "Social",
       minimo=0),
    _E("E-S03", "mao_obra_local_pct", "social.mao_obra_local_pct", "Mão de obra local", "%", "num", "Social", minimo=0,
       maximo=100),
    _E("E-S04", "compras_locais_pct", "social.compras_locais_pct", "Compras de insumos no município/região", "%", "num",
       "Social", minimo=0, maximo=100),
    _E("E-S05", "salario_minimo", "social.salario_minimo", "Salário mínimo vigente", "R$/mês", "num", "Social",
       minimo=1),
    _E("E-S06", "aderencia_nr", "social.aderencia_nr", "Aderência às Normas Regulamentadoras", "sim/não", "bool",
       "Social"),
    _E("E-S07", "programa_capacitacao", "social.programa_capacitacao", "Programa de capacitação", "sim/não", "bool",
       "Social"),
    _E("E-S08", "participacao_mulheres_pct", "social.participacao_mulheres_pct", "Participação de mulheres", "%", "num",
       "Social", minimo=0, maximo=100),
    _E("E-S09", "relacao_comunidade", "social.relacao_comunidade", "Relação com a comunidade", "—", "opcao", "Social",
       opcoes=("inexistente", "parcial", "consolidada")),
    _E("E-S10", "canais_formais_comunicacao", "social.canais_formais_comunicacao", "Canais formais de comunicação",
       "sim/não", "bool", "Social"),
    _E("E-S11", "reunioes_comunidade_ano", "social.reunioes_comunidade_ano", "Reuniões com a comunidade", "por ano",
       "int", "Social", minimo=0),
    _E("E-S12", "conflitos_registrados_ano", "social.conflitos_registrados_ano", "Conflitos registrados", "por ano",
       "int", "Social", minimo=0),
] + [
    _E(f"E-G{1 + i:02d}", f"gov_{k}", f"governanca.itens.{k}", ROTULOS_GOVERNANCA[k], "sim/não", "bool", "Governança")
    for i, k in enumerate(CHAVES_GOVERNANCA)
] + [
    # Lean-Green
    _E("E-L01", "fcr_referencia", "lean_green.fcr_referencia", "FCR de referência (boa prática)", "kg/kg", "num",
       "Lean-Green", minimo=0.1),
    _E("E-L02", "mortalidade_referencia_pct", "lean_green.mortalidade_referencia_pct", "Mortalidade de referência", "%",
       "num", "Lean-Green", minimo=0, maximo=99),
    _E("E-L03", "consumo_kwh_kg_referencia", "lean_green.consumo_kwh_kg_referencia", "Consumo de energia de referência",
       "kWh/kg", "num", "Lean-Green", minimo=0),
    # Parâmetros do método (Capítulo 3) — valores documentados, alteráveis
    _E("E-M01", "mc_iteracoes", "monte_carlo.iteracoes", "Iterações do Monte Carlo", "iterações", "int",
       "Parâmetros do método", minimo=100, maximo=500_000, metodo=True, padrao=10_000),
    _E("E-M02", "mc_semente", "monte_carlo.seed", "Semente do gerador pseudoaleatório", "—", "int",
       "Parâmetros do método", minimo=0, metodo=True, padrao=20260612),
    _E("E-M03", "mc_nivel_confianca", "monte_carlo.nivel_confianca", "Nível de confiança", "fração", "num",
       "Parâmetros do método", minimo=0.5, maximo=0.999, metodo=True, padrao=0.95),
    _E("E-M04", "pesos_metodo", "pesos.metodo", "Método de ponderação", "—", "opcao", "Parâmetros do método",
       opcoes=("preset", "likert", "ahp"), metodo=True, padrao="preset"),
    _E("E-M05", "peso_tecnica", "pesos.preset.tecnica", "Peso da dimensão técnica (preset)", "fração", "num",
       "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.25),
    _E("E-M06", "peso_economica", "pesos.preset.economica", "Peso da dimensão econômica (preset)", "fração", "num",
       "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.35),
    _E("E-M07", "peso_ambiental", "pesos.preset.ambiental", "Peso da dimensão ambiental (preset)", "fração", "num",
       "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.20),
    _E("E-M08", "peso_social", "pesos.preset.social", "Peso da dimensão social (preset)", "fração", "num",
       "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.20),
    _E("E-M09", "aplicar_vetos", "decisao.aplicar_vetos", "Aplicar vetos não compensatórios", "sim/não", "bool",
       "Parâmetros do método", metodo=True, padrao=True),
    _E("E-M10", "lso_minimo", "decisao.lso_minimo", "LSO mínima", "0–100", "num", "Parâmetros do método",
       minimo=0, maximo=100, metodo=True, padrao=40.0),
    _E("E-M11", "limiar_indice_viavel", "decisao.limiar_indice_viavel", "Limiar do índice para 'viável'", "0–1", "num",
       "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.60),
    _E("E-M12", "limiar_indice_ressalvas", "decisao.limiar_indice_ressalvas", "Limiar do índice para 'com ressalvas'",
       "0–1", "num", "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.50),
    _E("E-M13", "prob_vpl_positivo_minima", "decisao.prob_vpl_positivo_minima", "Probabilidade mínima de VPL > 0",
       "fração", "num", "Parâmetros do método", minimo=0, maximo=1, metodo=True, padrao=0.70),
]

POR_VAR: Dict[str, Entrada] = {e.var: e for e in ENTRADAS}
POR_CAMINHO: Dict[str, Entrada] = {e.caminho: e for e in ENTRADAS}
POR_MARCADOR: Dict[str, Entrada] = {e.marcador: e for e in ENTRADAS}

# Premissas incertas (Quadro 13): informe "mín; máx" (a moda é o valor determinístico informado)
# ou "mín; moda; máx" (a moda deve coincidir com o valor informado). Em branco = determinística.
DISTRIBUICOES = [
    ("E-D01", "dist_preco_venda_kg", "economico.preco_venda_kg", "Preço de venda", "R$/kg"),
    ("E-D02", "dist_custo_racao_kg", "economico.custo_racao_kg", "Custo da ração", "R$/kg"),
    ("E-D03", "dist_custo_alevino_milheiro", "economico.custo_alevino_milheiro", "Custo do milheiro de juvenis", "R$/mil"),
    ("E-D04", "dist_tarifa_energia_kwh", "economico.tarifa_energia_kwh", "Tarifa de energia", "R$/kWh"),
    ("E-D05", "dist_custos_fixos_fator", "economico.custos_fixos_fator", "Fator de custos fixos (moda = 1)", "fator"),
    ("E-D06", "dist_capex_fator", "economico.capex_fator", "Fator de CAPEX (moda = 1)", "fator"),
    ("E-D07", "dist_fcr", "tecnico.fcr", "Conversão alimentar (FCR)", "kg/kg"),
    ("E-D08", "dist_mortalidade_pct", "tecnico.mortalidade_pct", "Mortalidade", "%"),
    ("E-D09", "dist_desempenho_crescimento_pct", "tecnico.desempenho_crescimento_pct",
     "Desempenho de crescimento (moda = 100)", "%"),
]
N_CAPEX = 12
CAMPOS_ALTERNATIVAS = ("B", "C")


class EntradasInvalidas(ValueError):
    """Entradas em branco ou inválidas; ``erros`` lista cada problema com o seu marcador."""

    def __init__(self, erros: List[str]):
        self.erros = list(erros)
        super().__init__("Entradas em branco ou inválidas:\n- " + "\n- ".join(self.erros))


# ---------------------------------------------------------------------------------------------
# Conversão de textos digitados (vírgula decimal, sim/não, opções sem acento)
# ---------------------------------------------------------------------------------------------
def _sem_acento(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", str(t)) if unicodedata.category(c) != "Mn").lower().strip()


def ler_numero(texto) -> float:
    """'1.234,56' -> 1234.56; '1,5' -> 1.5; '1500' -> 1500; aceita R$, %, espaços e sinal '−'."""
    if isinstance(texto, bool):
        raise ValueError("valor lógico não é número")
    if isinstance(texto, (int, float)):
        return float(texto)
    t = str(texto).strip().replace("R$", "").replace("%", "").replace(" ", "").replace(" ", "").replace("−", "-")
    if not t:
        raise ValueError("vazio")
    if "," in t and "." in t:
        t = t.replace(".", "").replace(",", ".")
    elif "," in t:
        if t.count(",") > 1:
            raise ValueError(f"número inválido: {texto!r}")
        t = t.replace(",", ".")
    elif re.fullmatch(r"-?\d{1,3}(\.\d{3})+", t):
        t = t.replace(".", "")
    try:
        return float(t)
    except ValueError:
        raise ValueError(f"número inválido: {texto!r}") from None


def ler_bool(texto) -> bool:
    if isinstance(texto, bool):
        return texto
    t = _sem_acento(texto)
    if t in ("sim", "s", "yes", "y", "true", "verdadeiro", "1", "x", "atendido"):
        return True
    if t in ("nao", "n", "no", "false", "falso", "0", "nao atendido"):
        return False
    raise ValueError(f"responda sim ou não: {texto!r}")


def _vazio(v) -> bool:
    return v is None or (isinstance(v, str) and not v.strip()) or (isinstance(v, float) and v != v)


def converter(entrada: Entrada, texto) -> Any:
    """Converte e valida o texto digitado para o tipo da entrada (ValueError com mensagem clara)."""
    if entrada.tipo == "texto":
        return str(texto).strip()
    if entrada.tipo == "bool":
        return ler_bool(texto)
    if entrada.tipo == "opcao":
        alvo = _sem_acento(texto)
        for op in entrada.opcoes:
            if _sem_acento(op) == alvo:
                return op
        raise ValueError(f"opção inválida {texto!r}; use uma de: {', '.join(entrada.opcoes)}")
    v = ler_numero(texto)
    if entrada.tipo == "int":
        if abs(v - round(v)) > 1e-9:
            raise ValueError(f"informe um número inteiro: {texto!r}")
        v = int(round(v))
    if entrada.minimo is not None and v < entrada.minimo:
        raise ValueError(f"valor {fmt_auto(v)} abaixo do mínimo admitido ({fmt_auto(entrada.minimo)})")
    if entrada.maximo is not None and v > entrada.maximo:
        raise ValueError(f"valor {fmt_auto(v)} acima do máximo admitido ({fmt_auto(entrada.maximo)})")
    return v


def fmt_auto(v) -> str:
    """Formata número em pt-BR com as casas decimais necessárias (até 4)."""
    if isinstance(v, bool):
        return "sim" if v else "não"
    if isinstance(v, (int, float)):
        if float(v).is_integer():
            return fmt_num(v, 0)
        s = fmt_num(v, 4).rstrip("0")
        return s[:-1] if s.endswith(",") else s
    return str(v)


def tipo_do_caminho(caminho: str) -> str:
    e = POR_CAMINHO.get(caminho)
    return e.tipo if e else "num"


# ---------------------------------------------------------------------------------------------
# Leitura e escrita de valores na configuração
# ---------------------------------------------------------------------------------------------
_NOME_TRIBUTO = {"faturamento": "Tributos sobre o faturamento", "lucro": "Tributos sobre o resultado"}


def obter_valor(cfg: EVTEASConfig, entrada: Entrada):
    if "#" in entrada.caminho:
        base = entrada.caminho.split("#")[1]
        trib = [t for t in cfg.economico.tributos if t.base == base]
        if trib:
            return sum(t.aliquota_pct for t in trib)
        return None if not cfg.economico.tributos else 0.0
    try:
        return obter_por_caminho(cfg, entrada.caminho)
    except KeyError:
        return None


def definir_valor(cfg: EVTEASConfig, entrada: Entrada, valor) -> None:
    if "#" in entrada.caminho:
        base = entrada.caminho.split("#")[1]
        cfg.economico.tributos = [t for t in cfg.economico.tributos if t.base != base]
        cfg.economico.tributos.append(Tributo(_NOME_TRIBUTO[base], float(valor), base))
        return
    if entrada.caminho in ("projeto", "localizacao", "responsavel"):
        setattr(cfg, entrada.caminho, valor)
        return
    if entrada.tipo in ("texto", "opcao"):
        bloco, campo = entrada.caminho.split(".", 1)
        setattr(getattr(cfg, bloco), campo, valor)
        return
    definir_por_caminho(cfg, entrada.caminho, valor)


def config_vazia() -> EVTEASConfig:
    """Configuração sem nenhuma entrada do projeto (apenas os parâmetros do método)."""
    return EVTEASConfig()


def entradas_faltantes(cfg: EVTEASConfig) -> List[str]:
    """Entradas obrigatórias em branco, no formato '[E-xx] Rótulo (unidade)'."""
    faltam = []
    for e in ENTRADAS:
        if e.obrigatorio and _vazio(obter_valor(cfg, e)):
            faltam.append(f"[{e.marcador}] {e.rotulo} ({e.unidade})")
    if not cfg.economico.capex:
        faltam.append("[E-C] Itens de investimento (CAPEX)")
    return faltam


# ---------------------------------------------------------------------------------------------
# Listas: CAPEX, mix de produtos, distribuições, alterações das alternativas e fontes
# ---------------------------------------------------------------------------------------------
def _partes(texto: str, sep: str = ";") -> List[str]:
    return [p.strip() for p in str(texto).split(sep) if p.strip()]


def ler_item_capex(texto: str) -> ItemCapex:
    """'Escavação dos viveiros; 380.000; 20' -> ItemCapex (descrição; valor em R$; vida útil em anos)."""
    p = _partes(texto)
    if len(p) != 3:
        raise ValueError(f"use 'descrição; valor; vida útil (anos)': {texto!r}")
    valor, vida = ler_numero(p[1]), ler_numero(p[2])
    if valor < 0 or vida <= 0:
        raise ValueError(f"valor negativo ou vida útil não positiva: {texto!r}")
    return ItemCapex(p[0], valor, vida)


def ler_mix(texto: str) -> List[Produto]:
    """'Inteira; 0,7; 1 | Filé; 0,3; 2,4' -> lista de produtos (nome; participação; fator de preço)."""
    itens = []
    for bloco in _partes(texto, "|"):
        p = _partes(bloco)
        if len(p) != 3:
            raise ValueError(f"use 'nome; participação; fator de preço': {bloco!r}")
        itens.append(Produto(p[0], ler_numero(p[1]), ler_numero(p[2])))
    return itens


def ler_distribuicao(texto: str, valor_deterministico: float) -> Distribuicao:
    """'mín; máx' (moda = valor determinístico) ou 'mín; moda; máx'."""
    p = [ler_numero(x) for x in _partes(texto)]
    if len(p) == 2:
        return Distribuicao("triangular", p[0], float(valor_deterministico), p[1])
    if len(p) == 3:
        return Distribuicao("triangular", p[0], p[1], p[2])
    raise ValueError(f"use 'mín; máx' ou 'mín; moda; máx': {texto!r}")


def ler_alteracoes(texto: str) -> Dict[str, Any]:
    """'fonte_energia = solar; fracao_renovavel_pct = 90' -> {var: valor convertido}."""
    alt = {}
    for par in _partes(texto):
        if "=" not in par:
            raise ValueError(f"use 'variável = valor': {par!r}")
        var, val = (x.strip() for x in par.split("=", 1))
        if var not in POR_VAR:
            raise ValueError(f"variável desconhecida {var!r} (use os nomes do formulário)")
        alt[var] = converter(POR_VAR[var], val)
    return alt


def ler_fontes(texto: str) -> Dict[str, Fonte]:
    """'preco_venda_kg = cotação (CEASA-RJ, jun. 2026); fcr = parâmetro bibliográfico (Kubitza, 2017)'."""
    fontes = {}
    for par in _partes(texto):
        if "=" not in par:
            raise ValueError(f"use 'variável = categoria (referência)': {par!r}")
        var, resto = (x.strip() for x in par.split("=", 1))
        caminho = POR_VAR[var].caminho if var in POR_VAR else None
        if caminho is None:
            raise ValueError(f"variável desconhecida em fontes: {var!r}")
        m = re.match(r"([^()]+)(\((.*)\))?", resto)
        cat = _sem_acento(m.group(1)) if m else ""
        categoria = next((c for c in CATEGORIAS_FONTE if _sem_acento(c).startswith(cat) or cat.startswith(_sem_acento(c))),
                         None)
        if categoria is None:
            raise ValueError(f"categoria de fonte inválida em {par!r}; use uma de {CATEGORIAS_FONTE}")
        fontes[caminho] = Fonte(categoria, (m.group(3) or "").strip() if m else "")
    return fontes


# ---------------------------------------------------------------------------------------------
# Construção da configuração a partir do formulário (dicionário var -> texto)
# ---------------------------------------------------------------------------------------------
def campos_formulario() -> List[str]:
    nomes = [e.var for e in ENTRADAS] + [f"capex_{i:02d}" for i in range(1, N_CAPEX + 1)] + ["mix_produtos"]
    nomes += [d[1] for d in DISTRIBUICOES] + ["fontes"]
    for a in CAMPOS_ALTERNATIVAS:
        nomes += [f"alt_{a}_nome", f"alt_{a}_alteracoes", f"alt_{a}_capex_adicional"]
    return nomes


def coletar_formulario(namespace: Dict[str, Any]) -> Dict[str, Any]:
    """Recolhe do namespace do notebook (globals()) os campos do formulário."""
    return {k: namespace[k] for k in campos_formulario() if k in namespace}


def ler_formulario(valores: Dict[str, Any], base: Optional[EVTEASConfig] = None) -> Tuple[EVTEASConfig, List[str]]:
    """Monta a configuração e devolve (cfg, erros). Campos em branco: obrigatórios geram erro;
    opcionais recebem o valor 'não se aplica'; parâmetros do método mantêm o valor documentado."""
    cfg = copiar(base) if base is not None else config_vazia()
    erros: List[str] = []
    for e in ENTRADAS:
        bruto = valores.get(e.var)
        if _vazio(bruto):
            if base is not None and not _vazio(obter_valor(cfg, e)):
                continue
            if e.metodo:
                continue
            if not e.obrigatorio:
                definir_valor(cfg, e, e.vazio)
                continue
            erros.append(f"[{e.marcador}] {e.rotulo} ({e.unidade}): em branco")
            continue
        try:
            definir_valor(cfg, e, converter(e, bruto))
        except ValueError as exc:
            erros.append(f"[{e.marcador}] {e.rotulo}: {exc}")
    capex = []
    for i in range(1, N_CAPEX + 1):
        t = valores.get(f"capex_{i:02d}")
        if not _vazio(t):
            try:
                capex.append(ler_item_capex(t))
            except ValueError as exc:
                erros.append(f"[E-C{i:02d}] Item de CAPEX {i}: {exc}")
    if capex:
        cfg.economico.capex = capex
    elif not cfg.economico.capex:
        erros.append("[E-C] Itens de investimento (CAPEX): informe ao menos um item ('descrição; valor; vida útil')")
    if not _vazio(valores.get("mix_produtos")):
        try:
            cfg.economico.mix_produtos = ler_mix(valores["mix_produtos"])
        except ValueError as exc:
            erros.append(f"[E-E25] Mix de produtos: {exc}")
    for marcador, var, caminho, rot, _ in DISTRIBUICOES:
        t = valores.get(var)
        if _vazio(t):
            continue
        try:
            atual = obter_por_caminho(cfg, caminho)
            if atual is None:
                raise ValueError("informe primeiro o valor determinístico desta premissa")
            cfg.distribuicoes[caminho] = ler_distribuicao(t, float(atual))
        except ValueError as exc:
            erros.append(f"[{marcador}] Distribuição de {rot}: {exc}")
    if not _vazio(valores.get("fontes")):
        try:
            cfg.fontes.update(ler_fontes(valores["fontes"]))
        except ValueError as exc:
            erros.append(f"Fontes das premissas: {exc}")
    if not erros:
        from .config import validar_config
        erros += validar_config(cfg)
    return cfg, erros


def config_de_formulario(valores: Dict[str, Any], base: Optional[EVTEASConfig] = None) -> EVTEASConfig:
    cfg, erros = ler_formulario(valores, base)
    if erros:
        raise EntradasInvalidas(erros)
    return cfg


# ---------------------------------------------------------------------------------------------
# Alternativas de projeto definidas pelo usuário (comparação por TOPSIS)
# ---------------------------------------------------------------------------------------------
def criar_alternativa(base: EVTEASConfig, nome: str, alteracoes: Dict[str, Any],
                      capex_adicional: Optional[List[ItemCapex]] = None) -> EVTEASConfig:
    cfg = copiar(base)
    cfg.projeto = nome
    for var, valor in alteracoes.items():
        definir_valor(cfg, POR_VAR[var], valor)
    if capex_adicional:
        cfg.economico.capex = list(cfg.economico.capex) + list(capex_adicional)
    # Distribuições acompanham o novo valor determinístico (mantida a amplitude relativa)
    for caminho, d in list(cfg.distribuicoes.items()):
        novo = float(obter_por_caminho(cfg, caminho))
        antigo = float(obter_por_caminho(base, caminho))
        if abs(novo - antigo) > 1e-12 and antigo:
            f = novo / antigo
            cfg.distribuicoes[caminho] = Distribuicao(d.tipo, d.minimo * f, novo, d.maximo * f)
    return cfg


def alternativas_de_formulario(base: EVTEASConfig, valores: Dict[str, Any]) -> Dict[str, EVTEASConfig]:
    """Alternativa A = entradas informadas; B e C = alterações declaradas no formulário (opcionais)."""
    alts = {"A": copiar(base)}
    erros = []
    for a in CAMPOS_ALTERNATIVAS:
        nome = valores.get(f"alt_{a}_nome")
        if _vazio(nome):
            continue
        try:
            alt = ler_alteracoes(valores.get(f"alt_{a}_alteracoes") or "")
            capex = [ler_item_capex(b) for b in _partes(valores.get(f"alt_{a}_capex_adicional") or "", "|")]
            if not alt and not capex:
                raise ValueError("declare ao menos uma alteração ou um item de CAPEX adicional")
            alts[a] = criar_alternativa(base, str(nome).strip(), alt, capex)
        except ValueError as exc:
            erros.append(f"[E-X{a}] Alternativa {a}: {exc}")
    if erros:
        raise EntradasInvalidas(erros)
    return alts


# ---------------------------------------------------------------------------------------------
# Quadro de premissas e marcadores [E-xx]
# ---------------------------------------------------------------------------------------------
def _origem(cfg: EVTEASConfig, caminho: str) -> str:
    f = cfg.fontes.get(caminho)
    if f is None:
        return "não informada"
    return f.categoria + (f" ({f.referencia})" if f.referencia else "")


def tabela_entradas(cfg: EVTEASConfig, incluir_metodo: bool = True) -> pd.DataFrame:
    linhas = []
    for e in ENTRADAS:
        if e.metodo and not incluir_metodo:
            continue
        v = obter_valor(cfg, e)
        if not e.obrigatorio and (v == e.vazio or _vazio(v)):
            txt = "não se aplica" if not e.ajuda else "em branco (" + e.ajuda.split(":", 1)[-1].strip() + ")"
        else:
            txt = "em branco" if _vazio(v) else fmt_auto(v)
        linhas.append({"Marcador": e.marcador, "Grupo": e.grupo, "Parâmetro": e.rotulo, "Valor": txt,
                       "Unidade": e.unidade, "Origem": _origem(cfg, e.caminho)})
    for i, item in enumerate(cfg.economico.capex, start=1):
        linhas.append({"Marcador": f"E-C{i:02d}", "Grupo": "CAPEX", "Parâmetro": item.descricao,
                       "Valor": fmt_num(item.valor, 2), "Unidade": f"R$ (vida útil {fmt_auto(item.vida_util_anos)} anos)",
                       "Origem": _origem(cfg, "economico.capex")})
    linhas.append({"Marcador": "E-C00", "Grupo": "CAPEX", "Parâmetro": "CAPEX total",
                   "Valor": fmt_num(sum(i.valor for i in cfg.economico.capex), 2), "Unidade": "R$",
                   "Origem": _origem(cfg, "economico.capex")})
    mix = cfg.economico.mix_produtos
    linhas.append({"Marcador": "E-E25", "Grupo": "Econômica", "Parâmetro": "Mix de produtos",
                   "Valor": " | ".join(f"{p.nome}: {fmt_auto(p.participacao * 100)}% (fator de preço {fmt_auto(p.fator_preco)})"
                                       for p in mix) if mix else "produto único",
                   "Unidade": "—", "Origem": _origem(cfg, "economico.mix_produtos")})
    for marcador, var, caminho, rot, unid in DISTRIBUICOES:
        d = cfg.distribuicoes.get(caminho)
        txt = (f"{fmt_auto(d.minimo)} / {fmt_auto(d.mais_provavel)} / {fmt_auto(d.maximo)}" if d is not None
               else "determinística (sem distribuição)")
        linhas.append({"Marcador": marcador, "Grupo": "Distribuições (mín./moda/máx.)", "Parâmetro": rot, "Valor": txt,
                       "Unidade": unid, "Origem": _origem(cfg, caminho)})
    return pd.DataFrame(linhas)


def marcadores_entradas(cfg: EVTEASConfig, alternativas: Optional[Dict[str, EVTEASConfig]] = None) -> List[tuple]:
    """(marcador, descrição, valor, seção) para as entradas; '-F' indica a origem declarada."""
    L = []
    for _, r in tabela_entradas(cfg).iterrows():
        L.append((r["Marcador"], f"{r['Parâmetro']} ({r['Unidade']})", r["Valor"], "4.2"))
        L.append((r["Marcador"] + "F", f"Origem: {r['Parâmetro']}", r["Origem"], "4.2"))
    if alternativas:
        for k, alt in alternativas.items():
            if k == "A":
                continue
            difs = []
            for e in ENTRADAS:
                va, vb = obter_valor(cfg, e), obter_valor(alt, e)
                if va != vb and e.var != "projeto":
                    difs.append(f"{e.rotulo}: {fmt_auto(va)} → {fmt_auto(vb)}")
            extra = alt.economico.capex[len(cfg.economico.capex):]
            difs += [f"CAPEX adicional: {i.descricao} (R$ {fmt_num(i.valor, 2)}; {fmt_auto(i.vida_util_anos)} anos)"
                     for i in extra]
            L.append((f"E-X{k}", f"Alternativa {k}: {alt.projeto}", "; ".join(difs) or "sem alterações", "4.9"))
    return L


# ---------------------------------------------------------------------------------------------
# Planilha de entradas (.xlsx): alternativa ao formulário, com coluna de origem por premissa
# ---------------------------------------------------------------------------------------------
def gerar_planilha_entradas(caminho: str, cfg: Optional[EVTEASConfig] = None) -> str:
    """Gera a planilha de entradas em branco (ou com os valores de ``cfg``, para revisão)."""
    linhas = []
    for e in ENTRADAS:
        v = obter_valor(cfg, e) if cfg is not None else (e.padrao if e.metodo else None)
        f = cfg.fontes.get(e.caminho) if cfg is not None else None
        linhas.append({"Marcador": e.marcador, "Grupo": e.grupo, "Variável": e.var, "Parâmetro": e.rotulo,
                       "Unidade": e.unidade, "Obrigatório": "sim" if e.obrigatorio and not e.metodo else "não",
                       "Opções": " / ".join(e.opcoes) if e.opcoes else ("sim / não" if e.tipo == "bool" else ""),
                       "Valor": "" if _vazio(v) else (("sim" if v else "não") if isinstance(v, bool) else v),
                       "Categoria da fonte": f.categoria if f else "", "Referência": f.referencia if f else "",
                       "Orientação": e.ajuda})
    capex = [{"Item": f"E-C{i:02d}", "Descrição": "", "Valor (R$)": "", "Vida útil (anos)": ""} for i in range(1, N_CAPEX + 1)]
    if cfg is not None:
        for i, it in enumerate(cfg.economico.capex[:N_CAPEX]):
            capex[i].update({"Descrição": it.descricao, "Valor (R$)": it.valor, "Vida útil (anos)": it.vida_util_anos})
    dist = []
    for marcador, var, cam, rot, unid in DISTRIBUICOES:
        d = cfg.distribuicoes.get(cam) if cfg is not None else None
        dist.append({"Marcador": marcador, "Variável": var, "Premissa": rot, "Unidade": unid,
                     "Mínimo": d.minimo if d else "", "Máximo": d.maximo if d else "",
                     "Orientação": "Em branco = determinística; a moda é o valor informado na aba Entradas"})
    alts = [{"Alternativa": a, "Nome": "", "Alterações (variável = valor; ...)": "",
             "CAPEX adicional (descrição; valor; vida | ...)": ""} for a in CAMPOS_ALTERNATIVAS]
    with pd.ExcelWriter(caminho, engine="openpyxl") as w:
        pd.DataFrame(linhas).to_excel(w, sheet_name="Entradas", index=False)
        pd.DataFrame(capex).to_excel(w, sheet_name="CAPEX", index=False)
        pd.DataFrame(dist).to_excel(w, sheet_name="Distribuições", index=False)
        pd.DataFrame(alts).to_excel(w, sheet_name="Alternativas", index=False)
        pd.DataFrame({"Categorias de fonte admitidas (Seção 3.6)": CATEGORIAS_FONTE}).to_excel(
            w, sheet_name="Categorias de fonte", index=False)
    return caminho


def valores_de_planilha(caminho: str) -> Tuple[Dict[str, Any], Dict[str, Fonte]]:
    """Lê a planilha preenchida e devolve (valores no formato do formulário, fontes)."""
    abas = pd.read_excel(caminho, sheet_name=None, dtype=object)
    valores: Dict[str, Any] = {}
    fontes: Dict[str, Fonte] = {}
    for _, r in abas["Entradas"].iterrows():
        var = str(r["Variável"]).strip()
        if var not in POR_VAR:
            continue
        v = r.get("Valor")
        valores[var] = "" if _vazio(v) else v
        cat = r.get("Categoria da fonte")
        if not _vazio(cat):
            alvo = _sem_acento(cat)
            categoria = next((c for c in CATEGORIAS_FONTE if _sem_acento(c) == alvo), None)
            if categoria is None:
                raise EntradasInvalidas([f"[{r['Marcador']}] Categoria de fonte inválida: {cat!r}"])
            ref = r.get("Referência")
            fontes[POR_VAR[var].caminho] = Fonte(categoria, "" if _vazio(ref) else str(ref))
    for i, (_, r) in enumerate(abas.get("CAPEX", pd.DataFrame()).iterrows(), start=1):
        if _vazio(r.get("Descrição")) and _vazio(r.get("Valor (R$)")):
            continue
        valores[f"capex_{i:02d}"] = f"{r.get('Descrição')}; {r.get('Valor (R$)')}; {r.get('Vida útil (anos)')}"
    for _, r in abas.get("Distribuições", pd.DataFrame()).iterrows():
        if _vazio(r.get("Mínimo")) and _vazio(r.get("Máximo")):
            continue
        valores[str(r["Variável"]).strip()] = f"{r.get('Mínimo')}; {r.get('Máximo')}"
    for _, r in abas.get("Alternativas", pd.DataFrame()).iterrows():
        a = str(r["Alternativa"]).strip()
        if a in CAMPOS_ALTERNATIVAS and not _vazio(r.get("Nome")):
            valores[f"alt_{a}_nome"] = r.get("Nome")
            valores[f"alt_{a}_alteracoes"] = r.get("Alterações (variável = valor; ...)") or ""
            valores[f"alt_{a}_capex_adicional"] = r.get("CAPEX adicional (descrição; valor; vida | ...)") or ""
    return valores, fontes


def config_de_planilha(caminho: str) -> EVTEASConfig:
    valores, fontes = valores_de_planilha(caminho)
    cfg, erros = ler_formulario(valores)
    if erros:
        raise EntradasInvalidas(erros)
    cfg.fontes.update(fontes)
    return cfg

#### Módulo `modelo` — motor único das dimensões técnica, econômica, ambiental e social, governança e Lean-Green

In [ ]:
%%writefile evteas_py/modelo.py
"""EVTEAS-Py — módulo modelo.

Motor único e vetorizado das quatro dimensões do EVTEAS (técnica, econômica, ambiental e social),
mais governança e desperdícios Lean-Green. A classe ``Contexto`` devolve qualquer parâmetro
numérico como um vetor de tamanho n: com n = 1 o modelo é determinístico; com n = 10.000, é a
própria simulação de Monte Carlo (mesma fórmula nas duas camadas).
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from .config import (EVTEASConfig, capacidade_suporte, obter_por_caminho, safe_div,
                     taxa_anual_para_mensal, taxa_mensal_para_anual)
from .financeiro import payback, tir, vpl

DIAS_POR_MES = 365.25 / 12
RELACAO_COMUNIDADE = {"inexistente": 0.0, "parcial": 0.5, "consolidada": 1.0}

# Checklist normativo ambiental: (rótulo, peso, eliminatório) — pesos somam 10 (Score 0 a 10)
CHECKLIST_AMBIENTAL = {
    "licenca_ambiental": ("Licença ambiental da aquicultura", 1.50, True),
    "outorga_agua": ("Outorga de direito de uso da água", 1.50, True),
    "app_respeitada": ("Área de Preservação Permanente respeitada", 1.50, True),
    "tratamento_efluentes": ("Tratamento de efluentes", 1.25, False),
    "monitoramento_agua": ("Monitoramento da qualidade da água", 1.00, False),
    "car": ("Cadastro Ambiental Rural (CAR)", 1.00, False),
    "plano_residuos": ("Plano de gestão de resíduos", 0.75, False),
    "prevencao_escape": ("Prevenção de escape de peixes", 1.00, False),
    "energia_renovavel": ("Uso de energia renovável", 0.50, False),
}

ITENS_GOVERNANCA = {
    "estatuto_registrado": "Estatuto social registrado",
    "assembleias_regulares": "Assembleias regulares",
    "conselho_fiscal": "Conselho fiscal atuante",
    "prestacao_contas_publica": "Prestação de contas pública",
    "plano_negocios": "Plano de negócios aprovado",
    "registro_premissas_auditavel": "Registro auditável das premissas",
    "canal_denuncia": "Canal de denúncia",
}


class Contexto:
    """Acesso vetorizado aos parâmetros, com substituições (overrides) por caminho."""

    def __init__(self, cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1):
        self.cfg = cfg
        self.overrides = overrides or {}
        self.n = int(n)
        for k, v in self.overrides.items():
            if np.ndim(v) and len(v) != self.n:
                raise ValueError(f"Override '{k}' com tamanho {len(v)} != n={self.n}")

    def v(self, caminho: str) -> np.ndarray:
        if caminho in self.overrides:
            return np.broadcast_to(np.asarray(self.overrides[caminho], dtype=float), (self.n,)).astype(float)
        valor = obter_por_caminho(self.cfg, caminho)
        return np.full(self.n, float(valor), dtype=float)


# ---------------------------------------------------------------------------------------------
# Dimensão técnica
# ---------------------------------------------------------------------------------------------
def calcular_tecnico(ctx: Contexto) -> Dict[str, Any]:
    """Infraestrutura, parâmetros biológicos e eficiência de insumos (Seção 2.8.6; Quadro 5)."""
    cfg = ctx.cfg
    t = cfg.tecnico
    vol = t.area_lamina_m2 * t.profundidade_media_m
    cap = capacidade_suporte(cfg)
    disponibilidade = t.tanques_ativos / t.numero_tanques
    # Restrição dupla da biomassa: capacidade de suporte (volume) e produtividade (área)
    limite_volume = vol * cap * disponibilidade
    limite_area = (t.area_lamina_m2 * t.produtividade_esperada_kg_m2_ciclo * disponibilidade
                   if t.produtividade_esperada_kg_m2_ciclo > 0 else np.inf)
    biomassa_alvo = min(limite_volume, limite_area)
    restricao = "capacidade de suporte (volume)" if limite_volume <= limite_area else "produtividade por área"

    peso_final_kg = t.peso_final_g / 1000
    sobrevivencia_planejada = 1 - t.mortalidade_pct / 100
    # A estocagem é decisão de projeto; a mortalidade e o crescimento são incertos.
    alevinos = (t.alevinos_por_ciclo if t.alevinos_por_ciclo > 0
                else safe_div(biomassa_alvo, peso_final_kg * sobrevivencia_planejada, 0.0))
    sobrevivencia = 1 - np.clip(ctx.v("tecnico.mortalidade_pct"), 0, 99.9) / 100
    desempenho = np.maximum(ctx.v("tecnico.desempenho_crescimento_pct"), 0) / 100
    fcr = ctx.v("tecnico.fcr")
    ciclos = ctx.v("tecnico.ciclos_ano")

    biomassa_despesca = np.minimum(alevinos * sobrevivencia * peso_final_kg * desempenho, limite_volume)
    biomassa_inicial = alevinos * t.peso_inicial_g / 1000
    ganho_ciclo = np.maximum(biomassa_despesca - biomassa_inicial, 0)
    racao_ciclo = ganho_ciclo * fcr

    producao_ano = biomassa_despesca * ciclos
    performance = np.minimum(desempenho, 1.0)
    qualidade = sobrevivencia                      # Quadro 5: Qualidade = taxa de sobrevivência
    oee = disponibilidade * performance * qualidade
    potencial_ano = vol * cap * ciclos             # capacidade plena de todos os viveiros
    return {
        "volume_util_m3": vol, "capacidade_suporte_kg_m3": cap, "disponibilidade": disponibilidade,
        "limite_volume_kg": limite_volume, "limite_area_kg": limite_area, "biomassa_alvo_kg": biomassa_alvo,
        "restricao_ativa": restricao, "alevinos_por_ciclo": alevinos, "alevinos_ano": alevinos * ciclos,
        "densidade_peixes_m3": safe_div(alevinos, vol * disponibilidade, 0.0),
        "sobrevivencia": sobrevivencia, "desempenho": desempenho, "fcr": fcr, "ciclos_ano": ciclos,
        "biomassa_despesca_ciclo_kg": biomassa_despesca, "biomassa_inicial_ciclo_kg": biomassa_inicial,
        "ganho_biomassa_ciclo_kg": ganho_ciclo, "racao_ciclo_kg": racao_ciclo,
        "producao_ano_kg": producao_ano, "producao_kg_mes": producao_ano / 12,
        "ganho_biomassa_ano_kg": ganho_ciclo * ciclos, "racao_ano_kg": racao_ciclo * ciclos,
        "produtividade_kg_m2_ano": producao_ano / t.area_lamina_m2,
        "produtividade_volumetrica_kg_m3_ano": producao_ano / vol,
        "produtividade_relativa": safe_div(producao_ano, potencial_ano, 0.0),
        "performance": performance, "qualidade": qualidade, "oee": oee,
    }


def curva_crescimento(cfg: EVTEASConfig, pontos: int = 0) -> pd.DataFrame:
    """Curva diária do ciclo: taxa de crescimento específico constante, mortalidade distribuída
    ao longo do ciclo e arraçoamento calibrado para reproduzir o FCR do ciclo."""
    tec = calcular_tecnico(Contexto(cfg))
    t = cfg.tecnico
    dias = np.arange(0, t.ciclo_dias + 1)
    peso_final = t.peso_final_g * t.desempenho_crescimento_pct / 100
    tce = math.log(peso_final / t.peso_inicial_g) / t.ciclo_dias      # taxa de crescimento específico
    peso = t.peso_inicial_g * np.exp(tce * dias)
    sobrev_final = 1 - t.mortalidade_pct / 100
    z = -math.log(sobrev_final) / t.ciclo_dias if sobrev_final < 1 else 0.0
    vivos = float(tec["alevinos_por_ciclo"][0] if np.ndim(tec["alevinos_por_ciclo"]) else tec["alevinos_por_ciclo"]) * np.exp(-z * dias)
    biomassa = vivos * peso / 1000
    ganho_diario = np.maximum(np.diff(biomassa, prepend=biomassa[0]), 0)
    ganho_total = max(ganho_diario.sum(), 1e-9)
    racao_total = float(np.atleast_1d(tec["racao_ciclo_kg"])[0])
    racao_diaria = ganho_diario / ganho_total * racao_total
    df = pd.DataFrame({"Dia": dias, "Peso médio (g)": peso, "Peixes vivos": vivos, "Biomassa (kg)": biomassa,
                       "Ração diária (kg)": racao_diaria, "Ração acumulada (kg)": np.cumsum(racao_diaria)})
    if pontos and pontos < len(df):
        df = df.iloc[np.linspace(0, len(df) - 1, pontos).astype(int)].reset_index(drop=True)
    return df


# ---------------------------------------------------------------------------------------------
# Dimensão econômica
# ---------------------------------------------------------------------------------------------
def aliquotas(cfg: EVTEASConfig) -> Dict[str, float]:
    a = {"faturamento": 0.0, "lucro": 0.0, "folha": 0.0}
    for tr in cfg.economico.tributos:
        if tr.base not in a:
            raise ValueError(f"Base de tributo inválida: {tr.base!r}")
        a[tr.base] += tr.aliquota_pct / 100
    return a


def fator_mix(cfg: EVTEASConfig) -> float:
    """Preço médio relativo do mix de produtos; sem mix informado, produto único (fator 1)."""
    mix = cfg.economico.mix_produtos
    return float(sum(p.participacao * p.fator_preco for p in mix)) if mix else 1.0


def calcular_economico(ctx: Contexto, tec: Dict[str, Any]) -> Dict[str, Any]:
    """Projeção mensal, DRE, fluxo de caixa e indicadores (Casarotto Filho; Kopittke, 2007)."""
    cfg, e, n = ctx.cfg, ctx.cfg.economico, ctx.n
    M = int(e.horizonte_anos * 12)
    meses = np.arange(1, M + 1)
    L = max(1, int(round(cfg.tecnico.ciclo_dias / DIAS_POR_MES)))     # meses até a 1ª despesca
    rampa = e.rampa_inicial_pct / 100 + (1 - e.rampa_inicial_pct / 100) * np.minimum(
        np.arange(M) / max(e.rampa_meses, 1), 1.0)
    aliq = aliquotas(cfg)

    prod_mes = tec["producao_kg_mes"][:, None]
    em_engorda = prod_mes * rampa[None, :]                    # kg/mês em crescimento
    vendida = np.zeros((n, M))
    vendida[:, L:] = em_engorda[:, : M - L]                   # despesca após o 1º ciclo

    preco = ctx.v("economico.preco_venda_kg")[:, None] * fator_mix(cfg)
    receita_produtos = vendida * preco
    receita_servicos = np.where(meses > L, e.receita_servicos_mes, 0.0)[None, :] * np.ones((n, 1))
    receita = receita_produtos + receita_servicos

    producao_ano = np.maximum(tec["producao_ano_kg"], 1e-9)
    racao_kg = (tec["racao_ano_kg"] / producao_ano)[:, None]
    alevinos_kg = (tec["alevinos_ano"] / producao_ano)[:, None]
    kwh_kg = ctx.v("ambiental.consumo_kwh_kg")[:, None]
    fracao_compra = 1 - ctx.v("ambiental.fracao_renovavel_pct")[:, None] / 100
    custo_racao = em_engorda * racao_kg * ctx.v("economico.custo_racao_kg")[:, None]
    custo_alevinos = em_engorda * alevinos_kg * ctx.v("economico.custo_alevino_milheiro")[:, None] / 1000
    custo_energia = em_engorda * kwh_kg * fracao_compra * ctx.v("economico.tarifa_energia_kwh")[:, None]
    custo_outros_var = vendida * e.outros_variaveis_kg
    custos_variaveis = custo_racao + custo_alevinos + custo_energia + custo_outros_var

    capex_nominal = sum(i.valor for i in e.capex)
    fator_capex = ctx.v("economico.capex_fator")
    capex = capex_nominal * fator_capex
    fator_fixos = ctx.v("economico.custos_fixos_fator")[:, None]
    pessoal = e.mao_obra_mes * (1 + e.encargos_pct / 100 + aliq["folha"]) * fator_fixos * np.ones((1, M))
    manutencao = (capex * e.manutencao_pct_capex_ano / 100 / 12)[:, None] * fator_fixos * np.ones((1, M))
    outros_fixos = (e.administrativo_mes + e.seguros_outros_fixos_mes) * fator_fixos * np.ones((1, M))
    custos_fixos = pessoal + manutencao + outros_fixos

    dep_mes = np.zeros(M)
    for item in e.capex:
        vida_m = max(int(round(item.vida_util_anos * 12)), 1)
        dep_mes[:min(vida_m, M)] += item.valor * (1 - e.valor_residual_pct / 100) / vida_m
    depreciacao = dep_mes[None, :] * fator_capex[:, None]

    impostos_fat = receita * aliq["faturamento"]
    receita_liquida = receita - impostos_fat
    lucro_bruto = receita_liquida - custos_variaveis
    ebitda = lucro_bruto - custos_fixos
    lair = ebitda - depreciacao
    impostos_lucro = np.maximum(lair, 0) * aliq["lucro"]
    lucro_liquido = lair - impostos_lucro
    fco = lucro_liquido + depreciacao

    investimento = capex + e.capital_giro
    residual = capex * e.valor_residual_pct / 100
    estoque_final = ((custo_racao + custo_alevinos + custo_energia)[:, M - L:].sum(axis=1)
                     if L > 0 else np.zeros(n))
    terminal = residual + e.capital_giro + estoque_final

    fluxo = np.zeros((n, M + 1))
    fluxo[:, 0] = -investimento
    fluxo[:, 1:] = fco
    fluxo[:, -1] += terminal

    tma_am = taxa_anual_para_mensal(ctx.v("economico.tma_aa_pct") / 100)
    vpl_v = vpl(tma_am, fluxo)
    tir_am, diag_tir = tir(fluxo)
    tir_aa = taxa_mensal_para_anual(tir_am)
    pb_s = payback(fluxo)
    pb_d = payback(fluxo, tma_am)
    il = safe_div(vpl_v + investimento, investimento, np.nan)

    # Perspectiva do beneficiário: parte do CAPEX aportada por fomento não reembolsável
    fomento = capex * e.fomento_nao_reembolsavel_pct / 100
    fluxo_b = fluxo.copy()
    fluxo_b[:, 0] += fomento
    vpl_b = vpl(tma_am, fluxo_b)
    tir_b_am, diag_b = tir(fluxo_b)
    pb_d_b = payback(fluxo_b, tma_am)

    # Ano de regime (indicadores anuais do Quadro 5)
    a0, a1 = (e.ano_regime - 1) * 12, e.ano_regime * 12
    soma = lambda x: x[:, a0:a1].sum(axis=1)
    receita_r = soma(receita)
    vendida_r = soma(vendida)
    cv_r, cf_r, dep_r = soma(custos_variaveis), soma(custos_fixos), soma(depreciacao)
    imp_fat_r, imp_luc_r, ll_r = soma(impostos_fat), soma(impostos_lucro), soma(lucro_liquido)
    custo_op_r = cv_r + cf_r
    preco_medio = (preco[:, 0] if preco.ndim == 2 else preco)
    cv_unit = safe_div(cv_r, vendida_r, 0.0) + preco_medio * aliq["faturamento"]
    margem_unit = preco_medio - cv_unit
    pe_kg = np.where(margem_unit > 0, safe_div(cf_r + dep_r, margem_unit, np.inf), np.inf)
    margem_seg = np.where(vendida_r > 0, 1 - safe_div(pe_kg, vendida_r, np.inf), -np.inf)
    insumos_externos_r = (soma(custo_racao) + soma(custo_alevinos) + soma(custo_energia)
                          + soma(custo_outros_var) + soma(manutencao) + soma(outros_fixos))
    return {
        "meses": M, "meses_ate_despesca": L, "rampa": rampa,
        "vendida_kg": vendida, "receita": receita, "receita_servicos": receita_servicos,
        "custo_racao": custo_racao, "custo_alevinos": custo_alevinos, "custo_energia": custo_energia,
        "custo_outros_variaveis": custo_outros_var, "custos_variaveis": custos_variaveis,
        "pessoal": pessoal, "manutencao": manutencao, "outros_fixos": outros_fixos, "custos_fixos": custos_fixos,
        "impostos_faturamento": impostos_fat, "receita_liquida": receita_liquida, "lucro_bruto": lucro_bruto,
        "ebitda": ebitda, "depreciacao": depreciacao, "lair": lair, "impostos_lucro": impostos_lucro,
        "lucro_liquido": lucro_liquido, "fco": fco, "fluxo_caixa": fluxo, "fluxo_caixa_beneficiario": fluxo_b,
        "capex": capex, "investimento": investimento, "residual": residual, "estoque_final": estoque_final,
        "terminal": terminal, "tma_am": tma_am,
        "vpl": vpl_v, "tir_am": tir_am, "tir_aa": tir_aa, "diagnostico_tir": diag_tir,
        "payback_simples_meses": pb_s, "payback_descontado_meses": pb_d, "indice_lucratividade": il,
        "fomento": fomento, "investimento_beneficiario": investimento - fomento,
        "vpl_beneficiario": vpl_b, "tir_aa_beneficiario": taxa_mensal_para_anual(tir_b_am),
        "diagnostico_tir_beneficiario": diag_b, "payback_descontado_beneficiario_meses": pb_d_b,
        "receita_regime": receita_r, "producao_vendida_regime_kg": vendida_r,
        "custos_variaveis_regime": cv_r, "custos_fixos_regime": cf_r, "depreciacao_regime": dep_r,
        "pessoal_regime": soma(pessoal), "impostos_regime": imp_fat_r + imp_luc_r,
        "lucro_liquido_regime": ll_r, "custo_operacional_regime": custo_op_r,
        "custo_racao_regime": soma(custo_racao),
        "roi_regime": safe_div(ll_r, investimento, np.nan),
        "ponto_equilibrio_kg_ano": pe_kg, "margem_seguranca": margem_seg,
        "custo_total_kg": safe_div(custo_op_r + dep_r, vendida_r, np.nan),
        "custo_operacional_kg": safe_div(custo_op_r, vendida_r, np.nan),
        "participacao_racao_custo_op": safe_div(soma(custo_racao), custo_op_r, np.nan),
        "valor_adicionado_regime": receita_r - insumos_externos_r,
        "preco_medio_kg": preco_medio,
    }


def dre_mensal(eco: Dict[str, Any], linha: int = 0) -> pd.DataFrame:
    """Demonstração do Resultado mensal da realização ``linha`` (0 = determinística)."""
    M = eco["meses"]
    col = lambda k: np.asarray(eco[k])[linha]
    df = pd.DataFrame({
        "Mês": np.arange(1, M + 1),
        "Produção vendida (kg)": col("vendida_kg"),
        "Receita Bruta": col("receita"),
        "Impostos s/ Faturamento": col("impostos_faturamento"),
        "Receita Líquida": col("receita_liquida"),
        "Custos Variáveis": col("custos_variaveis"),
        "Lucro Bruto": col("lucro_bruto"),
        "Custos Fixos": col("custos_fixos"),
        "EBITDA": col("ebitda"),
        "Depreciação": col("depreciacao"),
        "LAIR": col("lair"),
        "Impostos s/ Lucro": col("impostos_lucro"),
        "Resultado Líquido": col("lucro_liquido"),
        "Fluxo de Caixa Operacional": col("fco"),
    })
    return df


def dre_anual(df_mensal: pd.DataFrame) -> pd.DataFrame:
    df = df_mensal.copy()
    df.insert(0, "Ano", (df["Mês"] - 1) // 12 + 1)
    return df.drop(columns="Mês").groupby("Ano", as_index=False).sum()


def distribuicao_sobras(cfg: EVTEASConfig, eco: Dict[str, Any], linha: int = 0) -> Dict[str, float]:
    """Destinação das sobras do ano de regime (Lei nº 5.764/1971, art. 28: Fundo de Reserva
    mínimo de 10% e FATES mínimo de 5% das sobras líquidas)."""
    e = cfg.economico
    sobras = max(float(np.asarray(eco["lucro_liquido_regime"])[linha]), 0.0)
    if e.tipo_organizacao == "empresa":
        return {"Resultado líquido do ano de regime": sobras, "Fundo de Reserva": 0.0, "FATES": 0.0,
                "Distribuído aos trabalhadores": 0.0, "Retido/reinvestido": sobras}
    reserva, fates = 0.10 * sobras, 0.05 * sobras
    distribuivel = sobras - reserva - fates
    trab = distribuivel * e.perc_sobras_trabalhadores / 100
    return {"Sobras líquidas do ano de regime": sobras, "Fundo de Reserva (10%)": reserva, "FATES (5%)": fates,
            "Distribuído aos trabalhadores": trab, "Retido/reinvestido": distribuivel - trab}


def sobras_trabalhadores(cfg: EVTEASConfig, eco: Dict[str, Any]) -> np.ndarray:
    """Vetor das sobras distribuídas aos trabalhadores no ano de regime."""
    e = cfg.economico
    ll = np.maximum(np.asarray(eco["lucro_liquido_regime"], dtype=float), 0)
    if e.tipo_organizacao == "empresa":
        return np.zeros_like(ll)
    return ll * 0.85 * e.perc_sobras_trabalhadores / 100


# ---------------------------------------------------------------------------------------------
# Dimensão ambiental
# ---------------------------------------------------------------------------------------------
def calcular_ambiental(ctx: Contexto, tec: Dict[str, Any], eco: Dict[str, Any]) -> Dict[str, Any]:
    """Água (Hoekstra, 2017), nutrientes e pegada hídrica cinza (CONAMA 357/2005), energia,
    emissões, ecoeficiência (WBCSD, 2000) e conformidade normativa."""
    cfg = ctx.cfg
    a, t = cfg.ambiental, cfg.tecnico
    vol = tec["volume_util_m3"]
    producao_t = np.maximum(tec["producao_ano_kg"], 1e-9) / 1000

    # Água captada (enchimentos, renovação e reposição de perdas) e água consumida (evaporação)
    enchimento = vol * a.enchimentos_ano
    renovacao = vol * tec["disponibilidade"] * ctx.v("ambiental.renovacao_agua_pct_dia") / 100 * 365
    perdas = t.area_lamina_m2 * (a.evaporacao_mm_dia + a.infiltracao_mm_dia) / 1000 * 365
    captada = enchimento + renovacao + perdas
    evaporada = t.area_lamina_m2 * a.evaporacao_mm_dia / 1000 * 365
    ph_azul = evaporada / producao_t                                     # m³/t (água consumida)
    indice_uso_agua = safe_div(tec["producao_ano_kg"], captada, 0.0)    # kg de peixe por m³ captado

    # Balanço de nutrientes e pegada hídrica cinza (poluente crítico)
    racao = tec["racao_ano_kg"]
    ganho = tec["ganho_biomassa_ano_kg"]
    n_in = racao * a.proteina_racao_pct / 100 / 6.25
    p_in = racao * a.fosforo_racao_pct / 100
    n_ret = ganho * a.nitrogenio_peixe_pct / 100
    p_ret = ganho * a.fosforo_peixe_pct / 100
    remocao_n = ctx.v("ambiental.remocao_n_tratamento_pct") / 100
    remocao_p = ctx.v("ambiental.remocao_p_tratamento_pct") / 100
    n_carga = np.maximum(n_in - n_ret, 0) * (1 - remocao_n)
    p_carga = np.maximum(p_in - p_ret, 0) * (1 - remocao_p)
    dn = a.n_max_mg_l - a.n_natural_mg_l
    dp = a.p_max_mg_l - a.p_natural_mg_l
    if dn <= 0 or dp <= 0:
        raise ValueError("Concentração máxima deve superar a natural (pegada hídrica cinza indefinida)")
    cinza_n = n_carga * 1000 / dn          # kg -> g; mg/L = g/m³
    cinza_p = p_carga * 1000 / dp
    cinza = np.maximum(cinza_n, cinza_p)
    critico = np.where(cinza_p >= cinza_n, "fósforo total", "nitrogênio total")

    # Energia e emissões (proxy da pegada de carbono)
    kwh = tec["producao_ano_kg"] * ctx.v("ambiental.consumo_kwh_kg")
    fracao_rede = 1 - ctx.v("ambiental.fracao_renovavel_pct") / 100
    fe_rede = ctx.v("ambiental.fator_emissao_rede_kg_kwh")
    co2_racao = racao * a.fator_emissao_racao_kgco2e_kg
    co2_energia = kwh * fracao_rede * fe_rede
    co2_diesel = np.full(ctx.n, a.diesel_l_ano * a.fator_emissao_diesel_kg_l)
    co2_total = co2_racao + co2_energia + co2_diesel
    intensidade = safe_div(co2_total, tec["producao_ano_kg"], np.nan)

    va = eco["valor_adicionado_regime"]
    # Ecoeficiência (WBCSD): valor adicionado / influência ambiental. Indefinida se VA <= 0.
    eco_carbono = np.where(va > 0, safe_div(va, co2_total, np.nan), np.nan)
    eco_agua = np.where(va > 0, safe_div(va, captada, np.nan), np.nan)
    status_eco = np.where(va > 0, "definida", "não definida (valor adicionado <= 0)")

    # Conformidade normativa: checklist ponderado (0 a 10) com itens eliminatórios
    itens = dict(a.checklist)
    if a.distancia_app_m < a.distancia_minima_app_m:
        itens["app_respeitada"] = False
    score = 0.0
    vetos: List[str] = []
    tabela = []
    for chave, (rotulo, peso, elim) in CHECKLIST_AMBIENTAL.items():
        ok = bool(itens.get(chave, False))
        score += peso if ok else 0.0
        if elim and not ok:
            vetos.append(rotulo)
        tabela.append({"Item": rotulo, "Peso": peso, "Eliminatório": "sim" if elim else "não",
                       "Atendido": "sim" if ok else "não"})
    return {
        "agua_captada_m3_ano": captada, "agua_enchimento_m3": enchimento, "agua_renovacao_m3": renovacao,
        "agua_reposicao_perdas_m3": perdas, "agua_evaporada_m3": evaporada, "indice_uso_agua_kg_m3": indice_uso_agua,
        "ph_azul_m3_t": ph_azul, "n_aportado_kg": n_in, "p_aportado_kg": p_in, "n_retido_kg": n_ret,
        "p_retido_kg": p_ret, "n_lancado_kg": n_carga, "p_lancado_kg": p_carga,
        "ph_cinza_n_m3_t": cinza_n / 1 / producao_t, "ph_cinza_p_m3_t": cinza_p / producao_t,
        "ph_cinza_m3_t": cinza / producao_t, "poluente_critico": critico,
        "energia_kwh_ano": kwh, "co2_racao_kg": co2_racao, "co2_energia_kg": co2_energia,
        "co2_diesel_kg": co2_diesel, "co2_total_kg": co2_total, "intensidade_carbono_kg_kg": intensidade,
        "ecoeficiencia_rs_kgco2e": eco_carbono, "ecoeficiencia_rs_m3": eco_agua, "status_ecoeficiencia": status_eco,
        "score_conformidade_0_10": score, "vetos_ambientais": vetos, "checklist": pd.DataFrame(tabela),
    }


# ---------------------------------------------------------------------------------------------
# Dimensão social
# ---------------------------------------------------------------------------------------------
def calcular_social(ctx: Contexto, eco: Dict[str, Any]) -> Dict[str, Any]:
    """Emprego, renda, Retenção de Valor Local (Quadro 5; GRI 201-1 e 204-1), Índice de Impacto
    Local, Licença Social para Operar (Hurst; Ihlen, 2018), risco social e Radar Social."""
    cfg = ctx.cfg
    s = cfg.social
    massa_anual = cfg.economico.mao_obra_mes * 12
    massa_local = massa_anual * s.mao_obra_local_pct / 100
    compras = (eco["custos_variaveis_regime"] + eco["custos_fixos_regime"] - eco["pessoal_regime"])
    compras_locais = np.maximum(compras, 0) * s.compras_locais_pct / 100
    custo_operacional = eco["custos_variaveis_regime"] + eco["custos_fixos_regime"]
    rvl = 100 * safe_div(massa_local + compras_locais, custo_operacional, 0.0)

    sobras = sobras_trabalhadores(cfg, eco)
    renda_local = massa_local + sobras * s.mao_obra_local_pct / 100
    iil = safe_div(renda_local, eco["receita_regime"], 0.0)
    n_trab = max(cfg.economico.cooperados_trabalhadores if cfg.economico.tipo_organizacao != "empresa"
                 else s.empregos_diretos, 1)
    renda_mensal = (massa_anual + sobras) / n_trab / 12
    renda_sm = renda_mensal / s.salario_minimo

    relacao = RELACAO_COMUNIDADE.get(s.relacao_comunidade)
    if relacao is None:
        raise ValueError(f"relacao_comunidade deve ser uma de {list(RELACAO_COMUNIDADE)}")
    participacao = 1.0 if cfg.governanca.itens.get("assembleias_regulares") else 0.0
    componentes_lso = {
        "Relação com a comunidade": relacao,
        "Canais formais de comunicação": 1.0 if s.canais_formais_comunicacao else 0.0,
        "Frequência de engajamento": min(s.reunioes_comunidade_ano / 12, 1.0),
        "Ausência de conflitos": math.exp(-0.5 * max(s.conflitos_registrados_ano, 0)),
        "Participação na governança": participacao,
    }
    lso = 100 * float(np.mean(list(componentes_lso.values())))
    risco = 1 / (1 + math.exp(8 * (lso / 100 - 0.5)))      # heurística logística (a calibrar)

    area_ha = cfg.tecnico.area_lamina_m2 / 10_000
    radar = {
        "Emprego": min(s.empregos_diretos / max(area_ha, 1e-9), 1.0),        # referência: 1 emprego/ha
        "Renda": np.minimum(renda_sm / 2.0, 1.0),                              # referência: 2 salários mínimos
        "Segurança do trabalho": 1.0 if s.aderencia_nr else 0.0,
        "Capacitação": 1.0 if s.programa_capacitacao else 0.0,
        "Relação com a comunidade": relacao,
        "Inclusão": min(s.participacao_mulheres_pct / 50.0, 1.0),             # referência: paridade
    }
    radar_vals = [np.broadcast_to(np.asarray(v, dtype=float), (ctx.n,)) for v in radar.values()]
    radar_media = np.mean(np.vstack(radar_vals), axis=0)
    return {
        "massa_salarial_anual": massa_anual, "massa_salarial_local": massa_local,
        "compras_locais": compras_locais, "rvl_pct": rvl, "renda_local": renda_local,
        "indice_impacto_local": iil, "sobras_trabalhadores": sobras, "renda_mensal_trabalhador": renda_mensal,
        "renda_salarios_minimos": renda_sm, "componentes_lso": componentes_lso, "lso_0_100": lso,
        "risco_social": risco, "radar": radar, "radar_social_media": radar_media,
        "empregos_diretos": s.empregos_diretos, "empregos_indiretos": s.empregos_indiretos,
    }


def calcular_governanca(cfg: EVTEASConfig) -> Dict[str, Any]:
    itens = cfg.governanca.itens
    atendidos = sum(1 for k in ITENS_GOVERNANCA if itens.get(k))
    tabela = pd.DataFrame([{"Item": rot, "Atendido": "sim" if itens.get(k) else "não"}
                           for k, rot in ITENS_GOVERNANCA.items()])
    return {"itens_atendidos": atendidos, "itens_total": len(ITENS_GOVERNANCA),
            "score_governanca": atendidos / len(ITENS_GOVERNANCA), "checklist": tabela}


def calcular_lean_green(ctx: Contexto, tec: Dict[str, Any], eco: Dict[str, Any]) -> Dict[str, Any]:
    """Desperdícios evitáveis em R$/ano frente às referências declaradas (Lawrence et al., 2023)."""
    cfg = ctx.cfg
    lg, t = cfg.lean_green, cfg.tecnico
    preco_racao = ctx.v("economico.custo_racao_kg")
    racao_excedente_kg = np.maximum(tec["fcr"] - lg.fcr_referencia, 0) * tec["ganho_biomassa_ano_kg"]
    racao_rs = racao_excedente_kg * preco_racao
    mort = 100 * (1 - tec["sobrevivencia"])
    mortos_evitaveis = np.maximum(mort - lg.mortalidade_referencia_pct, 0) / 100 * tec["alevinos_ano"]
    racao_por_peixe_morto = 0.5 * (t.peso_final_g - t.peso_inicial_g) / 1000 * tec["fcr"]
    custo_peixe_morto = ctx.v("economico.custo_alevino_milheiro") / 1000 + racao_por_peixe_morto * preco_racao
    mortalidade_rs = mortos_evitaveis * custo_peixe_morto
    kwh_kg = ctx.v("ambiental.consumo_kwh_kg")
    energia_excedente = np.maximum(kwh_kg - lg.consumo_kwh_kg_referencia, 0) * tec["producao_ano_kg"]
    energia_rs = energia_excedente * (1 - ctx.v("ambiental.fracao_renovavel_pct") / 100) * ctx.v("economico.tarifa_energia_kwh")
    total = racao_rs + mortalidade_rs + energia_rs
    return {"racao_excedente_kg": racao_excedente_kg, "racao_excedente_rs": racao_rs,
            "mortalidade_evitavel_peixes": mortos_evitaveis, "mortalidade_evitavel_rs": mortalidade_rs,
            "energia_excedente_kwh": energia_excedente, "energia_excedente_rs": energia_rs,
            "desperdicio_total_rs": total,
            "desperdicio_pct_custo_operacional": safe_div(total, eco["custo_operacional_regime"], np.nan)}


def alinhamento_ods(cfg: EVTEASConfig, tec, eco, amb, soc, gov, linha: int = 0) -> pd.DataFrame:
    """Alinhamento aos ODS por regras explícitas e auditáveis."""
    g = lambda d, k: float(np.atleast_1d(np.asarray(d[k], dtype=float))[linha])
    s, a = cfg.social, cfg.ambiental
    regras = [
        ("ODS 1", "Erradicação da pobreza", g(soc, "renda_salarios_minimos") >= 1.0, "renda por trabalhador ≥ 1 salário mínimo"),
        ("ODS 2", "Fome zero e agricultura sustentável", g(tec, "producao_ano_kg") > 0, "produção de alimento proteico"),
        ("ODS 5", "Igualdade de gênero", s.participacao_mulheres_pct >= 30, "participação de mulheres ≥ 30%"),
        ("ODS 6", "Água potável e saneamento", bool(a.checklist.get("tratamento_efluentes")), "tratamento de efluentes"),
        ("ODS 7", "Energia limpa e acessível", a.fracao_renovavel_pct >= 50 or a.fonte_energia != "rede", "≥ 50% de energia renovável própria"),
        ("ODS 8", "Trabalho decente e crescimento econômico", s.aderencia_nr and g(soc, "renda_salarios_minimos") >= 1.0, "NR atendidas e renda ≥ 1 SM"),
        ("ODS 12", "Consumo e produção responsáveis", g(tec, "fcr") <= 1.8, "FCR ≤ 1,8"),
        ("ODS 13", "Ação contra a mudança do clima", g(amb, "intensidade_carbono_kg_kg") <= 3.0, "intensidade ≤ 3 kgCO2e/kg"),
        ("ODS 14", "Vida na água", bool(a.checklist.get("prevencao_escape")) and bool(a.checklist.get("tratamento_efluentes")), "prevenção de escape e tratamento"),
        ("ODS 15", "Vida terrestre", bool(a.checklist.get("app_respeitada")) and a.distancia_app_m >= a.distancia_minima_app_m, "APP respeitada"),
        ("ODS 16", "Paz, justiça e instituições eficazes", gov["score_governanca"] >= 0.5, "governança ≥ 50% dos itens"),
    ]
    return pd.DataFrame([{"ODS": o, "Tema": tema, "Atendido": "sim" if ok else "não", "Regra": regra}
                         for o, tema, ok, regra in regras])


# ---------------------------------------------------------------------------------------------
# Motor único
# ---------------------------------------------------------------------------------------------
def motor(cfg: EVTEASConfig, overrides: Optional[Dict[str, np.ndarray]] = None, n: int = 1,
          pesos: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Executa as quatro dimensões e o índice para n realizações das premissas."""
    from .ponderacao import calcular_indice
    ctx = Contexto(cfg, overrides, n)
    tec = calcular_tecnico(ctx)
    eco = calcular_economico(ctx, tec)
    amb = calcular_ambiental(ctx, tec, eco)
    soc = calcular_social(ctx, eco)
    gov = calcular_governanca(cfg)
    lean = calcular_lean_green(ctx, tec, eco)
    idx = calcular_indice(cfg, tec, eco, amb, soc, gov, pesos)
    return {"tecnico": tec, "economico": eco, "ambiental": amb, "social": soc,
            "governanca": gov, "lean_green": lean, "indice": idx}

#### Módulo `ponderacao` — normalização dos KPIs, pesos (preset, Likert, AHP), índice, decisão e TOPSIS

In [ ]:
%%writefile evteas_py/ponderacao.py
"""EVTEAS-Py — módulo ponderacao.

Normalização dos KPIs (sentido e faixa de referência declarados), scores dimensionais, índice
integrado EVTEAS, pesos (preset, escala Likert do Quadro 7 ou AHP de Saaty, 1980), regra de
decisão em dois estágios com vetos não compensatórios e ordenação de alternativas por TOPSIS
(Hwang; Yoon, 1981).
"""
from __future__ import annotations

from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from .config import EVTEASConfig

DIMENSOES = ["tecnica", "economica", "ambiental", "social"]
NOMES_DIMENSOES = {"tecnica": "Técnica", "economica": "Econômica", "ambiental": "Ambiental", "social": "Social"}

# KPI: (dimensão, rótulo, sentido, mínimo de referência, máximo de referência, extrator)
KPIS: Dict[str, tuple] = {
    "oee": ("tecnica", "OEE", "benefício", 0.0, 1.0, lambda r: r["tecnico"]["oee"]),
    "fcr": ("tecnica", "FCR", "custo", 1.0, 2.5, lambda r: r["tecnico"]["fcr"]),
    "produtividade_relativa": ("tecnica", "Produtividade relativa", "benefício", 0.0, 1.0,
                               lambda r: r["tecnico"]["produtividade_relativa"]),
    "vpl_investimento": ("economica", "VPL/investimento", "benefício", -1.0, 1.0,
                         lambda r: r["economico"]["vpl"] / r["economico"]["investimento"]),
    "roi": ("economica", "ROI anual", "benefício", -0.10, 0.30, lambda r: r["economico"]["roi_regime"]),
    "margem_seguranca": ("economica", "Margem de segurança", "benefício", 0.0, 0.50,
                         lambda r: r["economico"]["margem_seguranca"]),
    "ph_azul": ("ambiental", "PH azul (m³/t)", "custo", 0.0, 3000.0, lambda r: r["ambiental"]["ph_azul_m3_t"]),
    "ph_cinza": ("ambiental", "PH cinza (m³/t)", "custo", 0.0, 100000.0, lambda r: r["ambiental"]["ph_cinza_m3_t"]),
    "ecoeficiencia": ("ambiental", "Ecoeficiência (R$/kgCO2e)", "benefício", 0.0, 10.0,
                      lambda r: np.nan_to_num(r["ambiental"]["ecoeficiencia_rs_kgco2e"], nan=0.0)),
    "intensidade_carbono": ("ambiental", "Intensidade de carbono (kgCO2e/kg)", "custo", 0.0, 4.0,
                            lambda r: r["ambiental"]["intensidade_carbono_kg_kg"]),
    "conformidade": ("ambiental", "Conformidade ambiental (0–10)", "benefício", 0.0, 10.0,
                     lambda r: r["ambiental"]["score_conformidade_0_10"]),
    "lso": ("social", "LSO (0–100)", "benefício", 0.0, 100.0, lambda r: r["social"]["lso_0_100"]),
    "rvl": ("social", "RVL (%)", "benefício", 0.0, 100.0, lambda r: r["social"]["rvl_pct"]),
    "radar_social": ("social", "Radar Social", "benefício", 0.0, 1.0, lambda r: r["social"]["radar_social_media"]),
}


def normalizar(valor, sentido: str, minimo: float, maximo: float) -> np.ndarray:
    """Normalização min–máx com saturação em [0, 1]; custos são invertidos."""
    v = np.asarray(valor, dtype=float)
    x = (v - minimo) / (maximo - minimo)
    x = np.clip(np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)
    return 1.0 - x if sentido == "custo" else x


# ---------------------------------------------------------------------------------------------
# Pesos
# ---------------------------------------------------------------------------------------------
def pesos_likert(escores: Dict[str, Dict[str, int]]) -> Dict[str, Any]:
    """Pesos a partir da escala Likert (Quadro 7): média dos escores (1 a 5) de cada KPI entre os
    avaliadores, dividida pela soma total, de modo que os pesos somem 1. Também calcula a
    concordância média entre avaliadores (1 = escores idênticos; 0 = amplitude máxima de 4 pontos)."""
    if not escores:
        raise ValueError("Informe os escores Likert de pelo menos um avaliador")
    avaliadores = list(escores)
    for av, d in escores.items():
        faltam = [k for k in KPIS if k not in d]
        if faltam:
            raise ValueError(f"Avaliador {av!r} sem escore para: {faltam}")
        for k, v in d.items():
            if k in KPIS and not (1 <= int(v) <= 5):
                raise ValueError(f"Escore Likert fora de 1..5 ({av}, {k}: {v})")
    media = {k: float(np.mean([escores[a][k] for a in avaliadores])) for k in KPIS}
    total = sum(media.values())
    peso_kpi = {k: v / total for k, v in media.items()}
    peso_dim = {d: sum(p for k, p in peso_kpi.items() if KPIS[k][0] == d) for d in DIMENSOES}
    amplitude = {k: max(escores[a][k] for a in avaliadores) - min(escores[a][k] for a in avaliadores) for k in KPIS}
    concordancia = {k: 1 - amp / 4 for k, amp in amplitude.items()}
    return {"metodo": "likert", "dimensoes": peso_dim, "kpis": peso_kpi, "media_escores": media,
            "concordancia_por_kpi": concordancia, "concordancia_media": float(np.mean(list(concordancia.values()))),
            "avaliadores": avaliadores}


RI_SAATY = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}


def pesos_ahp(matriz) -> Dict[str, Any]:
    """Pesos pelo autovetor principal da matriz de comparação par a par (Saaty, 1980), com índice
    e razão de consistência (RC <= 0,10 indica julgamentos consistentes)."""
    A = np.asarray(matriz, dtype=float)
    k = A.shape[0]
    if A.shape != (k, k) or k != len(DIMENSOES):
        raise ValueError("A matriz AHP deve ser 4 x 4 (técnica, econômica, ambiental, social)")
    if np.any(A <= 0) or not np.allclose(A * A.T, 1.0, atol=1e-6):
        raise ValueError("A matriz AHP deve ser positiva e recíproca (a_ji = 1/a_ij)")
    autovalores, autovetores = np.linalg.eig(A)
    i = int(np.argmax(autovalores.real))
    w = np.abs(autovetores[:, i].real)
    w = w / w.sum()
    lam = float(autovalores[i].real)
    ic = (lam - k) / (k - 1)
    rc = ic / RI_SAATY[k] if RI_SAATY[k] else 0.0
    return {"metodo": "ahp", "dimensoes": dict(zip(DIMENSOES, w.tolist())), "kpis": None,
            "lambda_max": lam, "indice_consistencia": ic, "razao_consistencia": rc, "consistente": rc <= 0.10}


def resolver_pesos(cfg: EVTEASConfig) -> Dict[str, Any]:
    p = cfg.pesos
    if p.metodo == "likert":
        return pesos_likert(p.likert)
    if p.metodo == "ahp":
        return pesos_ahp(p.ahp)
    if p.metodo != "preset":
        raise ValueError("Método de pesos deve ser 'preset', 'likert' ou 'ahp'")
    soma = sum(p.preset[d] for d in DIMENSOES)
    return {"metodo": "preset", "dimensoes": {d: p.preset[d] / soma for d in DIMENSOES}, "kpis": None}


# ---------------------------------------------------------------------------------------------
# Índice integrado
# ---------------------------------------------------------------------------------------------
def calcular_indice(cfg: EVTEASConfig, tec, eco, amb, soc, gov, pesos: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Scores dimensionais (médias ponderadas dos KPIs normalizados) e índice EVTEAS (média
    ponderada das quatro dimensões)."""
    pesos = pesos or resolver_pesos(cfg)
    r = {"tecnico": tec, "economico": eco, "ambiental": amb, "social": soc}
    n = np.size(eco["vpl"])
    brutos, normalizados = {}, {}
    for k, (dim, rotulo, sentido, mn, mx, f) in KPIS.items():
        v = np.broadcast_to(np.asarray(f(r), dtype=float), (n,))
        brutos[k] = v
        normalizados[k] = normalizar(v, sentido, mn, mx)
    peso_kpi = pesos.get("kpis")
    scores = {}
    for d in DIMENSOES:
        ks = [k for k in KPIS if KPIS[k][0] == d]
        w = np.array([peso_kpi[k] for k in ks]) if peso_kpi else np.ones(len(ks))
        w = w / w.sum()
        scores[d] = np.sum(np.vstack([normalizados[k] for k in ks]) * w[:, None], axis=0)
    wd = pesos["dimensoes"]
    indice = sum(wd[d] * scores[d] for d in DIMENSOES)
    esg = {"ambiental": scores["ambiental"], "social": scores["social"],
           "governanca": np.full(n, gov["score_governanca"])}
    esg_score = (esg["ambiental"] + esg["social"] + esg["governanca"]) / 3
    return {"kpis_brutos": brutos, "kpis_normalizados": normalizados, "scores": scores,
            "pesos": pesos, "indice_evteas": indice, "pilares_esg": esg, "score_esg": esg_score}


def tabela_normalizacao(indice: Dict[str, Any], linha: int = 0) -> pd.DataFrame:
    linhas = []
    for k, (dim, rotulo, sentido, mn, mx, _) in KPIS.items():
        linhas.append({"Dimensão": NOMES_DIMENSOES[dim], "KPI": rotulo, "Sentido": sentido,
                       "Faixa de referência": f"{mn:g} a {mx:g}",
                       "Valor bruto": float(np.asarray(indice["kpis_brutos"][k])[linha]),
                       "Normalizado": float(np.asarray(indice["kpis_normalizados"][k])[linha])})
    return pd.DataFrame(linhas)


# ---------------------------------------------------------------------------------------------
# Regra de decisão
# ---------------------------------------------------------------------------------------------
def classificar(cfg: EVTEASConfig, vpl: float, indice: float, lso: float, vetos_ambientais: List[str],
                prob_vpl_positivo: Optional[float] = None) -> Dict[str, Any]:
    """Regra de decisão em dois estágios: vetos não compensatórios + índice compensatório.

    O índice agregado sozinho pode ocultar fragilidades; por isso VPL negativo, item ambiental
    eliminatório não atendido ou LSO abaixo do mínimo impedem a classificação "VIÁVEL",
    qualquer que seja o índice (Garnett; Röös; Little, 2015).
    """
    d = cfg.decisao
    motivos = []
    if d.aplicar_vetos:
        if vpl < 0:
            motivos.append("VPL negativo (veto econômico)")
        motivos += [f"Item eliminatório não atendido: {v}" for v in vetos_ambientais]
        if lso < d.lso_minimo:
            motivos.append(f"LSO abaixo do mínimo ({lso:.0f} < {d.lso_minimo:.0f})")
    ressalvas = []
    if prob_vpl_positivo is not None and prob_vpl_positivo < d.prob_vpl_positivo_minima:
        ressalvas.append(f"P(VPL > 0) = {prob_vpl_positivo:.0%} abaixo de {d.prob_vpl_positivo_minima:.0%}")
    if motivos:
        classe = "NÃO VIÁVEL"
    elif indice >= d.limiar_indice_viavel and not ressalvas:
        classe = "VIÁVEL"
    elif indice >= d.limiar_indice_ressalvas:
        classe = "VIÁVEL COM RESSALVAS"
        if indice < d.limiar_indice_viavel:
            ressalvas.append(f"Índice EVTEAS {indice:.3f} abaixo do limiar {d.limiar_indice_viavel:.2f}")
    else:
        classe = "NÃO VIÁVEL"
        motivos.append(f"Índice EVTEAS {indice:.3f} abaixo de {d.limiar_indice_ressalvas:.2f}")
    return {"classificacao": classe, "vetos": motivos, "ressalvas": ressalvas}


# ---------------------------------------------------------------------------------------------
# TOPSIS
# ---------------------------------------------------------------------------------------------
def topsis(matriz, pesos, sentidos: List[str]) -> np.ndarray:
    """Proximidade relativa à solução ideal (0 a 1) de cada alternativa (linhas da matriz).

    ``sentidos``: 'benefício' ou 'custo' por critério (colunas). Normalização vetorial.
    """
    X = np.asarray(matriz, dtype=float)
    w = np.asarray(pesos, dtype=float)
    w = w / w.sum()
    norma = np.sqrt((X ** 2).sum(axis=0))
    R = X / np.where(norma == 0, 1.0, norma)
    V = R * w
    beneficio = np.array([s == "benefício" for s in sentidos])
    ideal = np.where(beneficio, V.max(axis=0), V.min(axis=0))
    anti = np.where(beneficio, V.min(axis=0), V.max(axis=0))
    d_mais = np.sqrt(((V - ideal) ** 2).sum(axis=1))
    d_menos = np.sqrt(((V - anti) ** 2).sum(axis=1))
    return np.where(d_mais + d_menos > 0, d_menos / (d_mais + d_menos), 0.0)

#### Módulo `incerteza` — Monte Carlo, importância, sensibilidade, valores críticos e cenários

In [ ]:
%%writefile evteas_py/incerteza.py
"""EVTEAS-Py — módulo incerteza.

Simulação de Monte Carlo sobre o motor único (Rubinstein; Kroese, 2017), estatísticas de risco,
convergência, importância das variáveis (correlação de postos de Spearman), sensibilidade de um
fator por vez, valores críticos (VPL = 0) e cenários pessimista e otimista.
"""
from __future__ import annotations

from statistics import NormalDist
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from .config import EVTEASConfig, copiar, definir_por_caminho, obter_por_caminho
from .modelo import motor
from .ponderacao import resolver_pesos

VARIAVEIS_SENSIBILIDADE = [
    "economico.preco_venda_kg", "economico.custo_racao_kg", "tecnico.fcr",
    "tecnico.desempenho_crescimento_pct", "economico.custos_fixos_fator", "economico.capex_fator",
    "economico.tma_aa_pct", "tecnico.mortalidade_pct", "economico.tarifa_energia_kwh",
    "economico.custo_alevino_milheiro",
]

ROTULOS = {
    "economico.preco_venda_kg": "Preço de venda (R$/kg)",
    "economico.custo_racao_kg": "Custo da ração (R$/kg)",
    "economico.custo_alevino_milheiro": "Milheiro de alevinos (R$)",
    "economico.tarifa_energia_kwh": "Tarifa de energia (R$/kWh)",
    "economico.custos_fixos_fator": "Fator de custos fixos",
    "economico.capex_fator": "Fator de CAPEX (AACE Classe 5)",
    "economico.tma_aa_pct": "TMA (% a.a.)",
    "tecnico.fcr": "FCR",
    "tecnico.mortalidade_pct": "Mortalidade (%)",
    "tecnico.desempenho_crescimento_pct": "Desempenho de crescimento (%)",
}


def rotulo(caminho: str) -> str:
    return ROTULOS.get(caminho, caminho)


def amostrar(cfg: EVTEASConfig, n: int, seed: int) -> Dict[str, np.ndarray]:
    """Amostras das premissas incertas, em ordem fixa (alfabética) de variáveis, o que garante a
    reprodutibilidade com a mesma semente (np.random.default_rng)."""
    rng = np.random.default_rng(seed)
    entradas = {}
    for caminho in sorted(cfg.distribuicoes):
        d = cfg.distribuicoes[caminho]
        if d.tipo == "triangular":
            if d.minimo == d.maximo:
                entradas[caminho] = np.full(n, float(d.minimo))
            else:
                entradas[caminho] = rng.triangular(d.minimo, d.mais_provavel, d.maximo, size=n)
        else:
            entradas[caminho] = rng.uniform(d.minimo, d.maximo, size=n)
    return entradas


def _estatisticas(x: np.ndarray, nivel: float = 0.95) -> Dict[str, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    media, dp = float(np.mean(x)), float(np.std(x, ddof=1)) if n > 1 else 0.0
    ep = dp / np.sqrt(n) if n else np.nan
    z = NormalDist().inv_cdf((1 + nivel) / 2)
    p5, p50, p95 = np.percentile(x, [5, 50, 95])
    corte = np.percentile(x, 5)
    cvar = float(x[x <= corte].mean()) if np.any(x <= corte) else float(corte)
    return {"n": n, "media": media, "ic_inf": media - z * ep, "ic_sup": media + z * ep, "mediana": float(p50),
            "desvio_padrao": dp, "coef_variacao": abs(dp / media) if media else np.nan, "p5": float(p5),
            "p50": float(p50), "p95": float(p95), "var_5": float(corte), "cvar_5": cvar, "erro_padrao": ep,
            "minimo": float(np.min(x)), "maximo": float(np.max(x))}


def monte_carlo(cfg: EVTEASConfig, n: Optional[int] = None, seed: Optional[int] = None) -> Dict[str, Any]:
    """Simulação de Monte Carlo estática: cada iteração amostra um vetor de premissas e recalcula
    os fluxos de todo o horizonte com o mesmo motor da análise determinística."""
    n = int(n or cfg.monte_carlo.iteracoes)
    seed = int(cfg.monte_carlo.seed if seed is None else seed)
    if not cfg.distribuicoes:
        raise ValueError("Nenhuma distribuição de incerteza definida (cfg.distribuicoes vazio)")
    entradas = amostrar(cfg, n, seed)
    pesos = resolver_pesos(cfg)
    r = motor(cfg, entradas, n, pesos)
    eco, tec, amb, soc, idx = r["economico"], r["tecnico"], r["ambiental"], r["social"], r["indice"]
    tma = cfg.economico.tma_aa_pct / 100

    amostra = pd.DataFrame({k: v for k, v in entradas.items()})
    amostra["VPL"] = eco["vpl"]
    amostra["TIR a.a."] = eco["tir_aa"]
    amostra["VPL beneficiário"] = eco["vpl_beneficiario"]
    amostra["Índice EVTEAS"] = idx["indice_evteas"]
    amostra["OEE"] = tec["oee"]
    amostra["Intensidade de carbono"] = amb["intensidade_carbono_kg_kg"]
    amostra["RVL (%)"] = soc["rvl_pct"]

    est = _estatisticas(eco["vpl"], cfg.monte_carlo.nivel_confianca)
    tir_aa = eco["tir_aa"]
    tir_fin = tir_aa[np.isfinite(tir_aa)]
    est.update({
        "prob_vpl_positivo": float(np.mean(eco["vpl"] > 0)),
        "prob_tir_maior_tma": float(np.mean(np.nan_to_num(tir_aa, nan=-np.inf) > tma)),
        "tir_p5": float(np.percentile(tir_fin, 5)) if len(tir_fin) else np.nan,
        "tir_p50": float(np.percentile(tir_fin, 50)) if len(tir_fin) else np.nan,
        "tir_p95": float(np.percentile(tir_fin, 95)) if len(tir_fin) else np.nan,
        "tir_sem_solucao": float(np.mean(~np.isfinite(tir_aa))),
        "indice_media": float(np.mean(idx["indice_evteas"])),
        "prob_indice_viavel": float(np.mean(idx["indice_evteas"] >= cfg.decisao.limiar_indice_viavel)),
        "vpl_beneficiario_media": float(np.mean(eco["vpl_beneficiario"])),
        "prob_vpl_beneficiario_positivo": float(np.mean(eco["vpl_beneficiario"] > 0)),
        "payback_descontado_mediana": float(np.nanmedian(eco["payback_descontado_meses"]))
        if np.any(np.isfinite(eco["payback_descontado_meses"])) else np.nan,
    })

    # Convergência: média acumulada e intervalo de confiança
    v = eco["vpl"]
    k = np.arange(1, n + 1)
    media_acum = np.cumsum(v) / k
    var_acum = np.maximum(np.cumsum(v ** 2) / k - media_acum ** 2, 0) * k / np.maximum(k - 1, 1)
    ep_acum = np.sqrt(var_acum / k)
    passos = np.unique(np.geomspace(10, n, num=min(200, n)).astype(int)) if n >= 10 else np.arange(1, n + 1)
    convergencia = pd.DataFrame({"Iterações": passos, "Média do VPL": media_acum[passos - 1],
                                 "IC inferior": media_acum[passos - 1] - 1.96 * ep_acum[passos - 1],
                                 "IC superior": media_acum[passos - 1] + 1.96 * ep_acum[passos - 1]})

    # Importância das variáveis: correlação de postos de Spearman entre entradas e VPL
    imp = []
    for c in entradas:
        if np.std(entradas[c]) > 0:
            rho = pd.Series(entradas[c]).rank().corr(pd.Series(v).rank())
            imp.append({"Variável": c, "Rótulo": rotulo(c), "Spearman (VPL)": float(rho)})
    importancia = pd.DataFrame(imp, columns=["Variável", "Rótulo", "Spearman (VPL)"])
    importancia = importancia.sort_values("Spearman (VPL)", key=lambda s: -s.abs()).reset_index(drop=True)
    return {"n": n, "seed": seed, "entradas": entradas, "amostra": amostra, "estatisticas": est,
            "convergencia": convergencia, "importancia": importancia, "vpl": v,
            "indice": idx["indice_evteas"], "vpl_beneficiario": eco["vpl_beneficiario"]}


def _vpl_com(cfg: EVTEASConfig, caminho: str, valor: float) -> float:
    r = motor(cfg, {caminho: np.array([float(valor)])}, 1)
    return float(r["economico"]["vpl"][0])


def sensibilidade(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None,
                  variacoes=(-0.20, -0.10, 0.10, 0.20)) -> pd.DataFrame:
    """Sensibilidade de um fator por vez: cada premissa varia ±10% e ±20%, as demais no caso-base."""
    variaveis = variaveis or VARIAVEIS_SENSIBILIDADE
    base = float(motor(cfg, None, 1)["economico"]["vpl"][0])
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        linha = {"Variável": var, "Rótulo": rotulo(var), "Valor base": x}
        for d in variacoes:
            linha[f"ΔVPL ({d:+.0%})"] = _vpl_com(cfg, var, x * (1 + d)) - base
        linhas.append(linha)
    df = pd.DataFrame(linhas)
    col = [c for c in df.columns if c.startswith("ΔVPL (+10")]
    if col:
        df = df.reindex(df[col[0]].abs().sort_values(ascending=False).index).reset_index(drop=True)
    return df


def valores_criticos(cfg: EVTEASConfig, variaveis: Optional[List[str]] = None) -> pd.DataFrame:
    """Valor de cada variável que zera o VPL (demais premissas no caso-base), por bisseção."""
    variaveis = variaveis or [v for v in VARIAVEIS_SENSIBILIDADE if v != "economico.tma_aa_pct"]
    linhas = []
    for var in variaveis:
        x = float(obter_por_caminho(cfg, var))
        a, b = 0.0, max(x * 5, 1.0)
        if var == "tecnico.mortalidade_pct":
            b = 99.0
        fa, fb = _vpl_com(cfg, var, a), _vpl_com(cfg, var, b)
        critico = float("nan")
        if fa * fb <= 0:
            for _ in range(80):
                m = (a + b) / 2
                fm = _vpl_com(cfg, var, m)
                if fa * fm <= 0:
                    b, fb = m, fm
                else:
                    a, fa = m, fm
            critico = (a + b) / 2
        folga = (critico - x) / x if (x and np.isfinite(critico)) else float("nan")
        linhas.append({"Variável": var, "Rótulo": rotulo(var), "Valor base": x,
                       "Valor crítico (VPL = 0)": critico, "Folga relativa": folga})
    return pd.DataFrame(linhas)


def cenarios(cfg: EVTEASConfig) -> pd.DataFrame:
    """Cenários pessimista e otimista: cada premissa incerta no extremo desfavorável (ou
    favorável) da sua distribuição; o sentido é dado pelo efeito marginal sobre o VPL."""
    from .ponderacao import classificar
    base = motor(cfg, None, 1)
    vpl_base = float(base["economico"]["vpl"][0])
    pess, otim = {}, {}
    for caminho, d in cfg.distribuicoes.items():
        x = float(obter_por_caminho(cfg, caminho))
        delta = _vpl_com(cfg, caminho, x * 1.01 if x else 0.01) - vpl_base
        favoravel_max = delta >= 0
        pess[caminho] = np.array([d.minimo if favoravel_max else d.maximo])
        otim[caminho] = np.array([d.maximo if favoravel_max else d.minimo])
    linhas = []
    for nome, ov in (("Pessimista", pess), ("Caso-base", None), ("Otimista", otim)):
        r = motor(cfg, ov, 1)
        e, s, a, i = r["economico"], r["social"], r["ambiental"], r["indice"]
        dec = classificar(cfg, float(e["vpl"][0]), float(i["indice_evteas"][0]), float(s["lso_0_100"]),
                          a["vetos_ambientais"])
        linhas.append({"Cenário": nome, "VPL": float(e["vpl"][0]), "TIR a.a.": float(e["tir_aa"][0]),
                       "Índice EVTEAS": float(i["indice_evteas"][0]), "Classificação": dec["classificacao"]})
    return pd.DataFrame(linhas)

#### Módulo `pipeline` — execução integrada do estudo e comparação de alternativas

In [ ]:
%%writefile evteas_py/pipeline.py
"""EVTEAS-Py — módulo pipeline.

Ponto único de entrada para execução reprodutível (Etapas 6 e 7 da DSR): valida a configuração,
resolve os pesos, executa o motor das quatro dimensões, monta DRE, curva de crescimento, ODS e
tabela de premissas, habilita Monte Carlo, sensibilidade, valores críticos e cenários, aplica a
regra de decisão e registra a verificação por invariantes.
"""
from __future__ import annotations

import time
from dataclasses import asdict
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd

from .config import VERSAO, EVTEASConfig, obter_por_caminho, validar_config
from .modelo import (alinhamento_ods, curva_crescimento, distribuicao_sobras, dre_anual, dre_mensal, motor)
from .ponderacao import KPIS, classificar, resolver_pesos, tabela_normalizacao, topsis


def _escalar(d):
    """Converte vetores de tamanho 1 em escalares (resultado determinístico)."""
    if isinstance(d, dict):
        return {k: _escalar(v) for k, v in d.items()}
    if isinstance(d, np.ndarray):
        if d.ndim == 1 and d.size == 1:
            v = d[0]
            return v.item() if hasattr(v, "item") else v
        if d.ndim == 2 and d.shape[0] == 1:
            return d[0]
    return d


def tabela_premissas(cfg: EVTEASConfig) -> pd.DataFrame:
    """Entradas informadas, com marcador, unidade e origem declarada (rastreabilidade, Seção 3.6)."""
    from .entradas import tabela_entradas
    return tabela_entradas(cfg)


def executar_evteas(cfg: EVTEASConfig, executar_mc: bool = True, executar_sens: bool = True,
                    n_mc: Optional[int] = None) -> Dict[str, Any]:
    if not isinstance(cfg, EVTEASConfig):
        raise TypeError("cfg deve ser uma instância de EVTEASConfig")
    from .entradas import EntradasInvalidas
    problemas = validar_config(cfg)
    if executar_mc and not cfg.distribuicoes:
        problemas.append("[E-D] Nenhuma distribuição de incerteza informada: preencha ao menos uma premissa "
                         "do Quadro de distribuições para executar o Monte Carlo")
    if problemas:
        raise EntradasInvalidas(problemas)
    from .incerteza import cenarios, monte_carlo, sensibilidade, valores_criticos
    from .vv import validar_invariantes

    inicio = time.perf_counter()
    pesos = resolver_pesos(cfg)
    r = motor(cfg, None, 1, pesos)
    df_m = dre_mensal(r["economico"])
    det = {k: _escalar(v) for k, v in r.items()}
    resultado: Dict[str, Any] = {
        "versao": VERSAO,
        "config": asdict(cfg),
        **det,
        "curva_crescimento": curva_crescimento(cfg),
        "dre_mensal": df_m,
        "dre_anual": dre_anual(df_m),
        "fluxo_caixa": r["economico"]["fluxo_caixa"][0],
        "sobras": distribuicao_sobras(cfg, r["economico"]),
        "ods": alinhamento_ods(cfg, r["tecnico"], r["economico"], r["ambiental"], r["social"], r["governanca"]),
        "premissas": tabela_premissas(cfg),
        "normalizacao": tabela_normalizacao(r["indice"]),
    }
    prob = None
    if executar_mc:
        resultado["monte_carlo"] = monte_carlo(cfg, n=n_mc)
        prob = resultado["monte_carlo"]["estatisticas"]["prob_vpl_positivo"]
    if executar_sens:
        resultado["sensibilidade"] = sensibilidade(cfg)
        resultado["valores_criticos"] = valores_criticos(cfg)
        resultado["cenarios"] = cenarios(cfg)
    resultado["decisao"] = classificar(cfg, det["economico"]["vpl"], det["indice"]["indice_evteas"],
                                       det["social"]["lso_0_100"], det["ambiental"]["vetos_ambientais"], prob)
    resultado["vv"] = validar_invariantes(cfg, resultado, r)
    resultado["tempo_execucao_s"] = time.perf_counter() - inicio
    return resultado


CRITERIOS_TOPSIS = [
    ("VPL", "benefício"), ("P(VPL>0)", "benefício"), ("OEE", "benefício"), ("PH cinza (m³/t)", "custo"),
    ("kgCO2e/kg", "custo"), ("LSO", "benefício"), ("RVL (%)", "benefício"),
]


def comparar_alternativas(alternativas: Dict[str, EVTEASConfig], n_mc: Optional[int] = None,
                          pesos_criterios=None) -> pd.DataFrame:
    """Compara alternativas de projeto por TOPSIS (sete critérios), cada uma com Monte Carlo."""
    from .incerteza import monte_carlo
    linhas = []
    for nome, cfg in alternativas.items():
        r = motor(cfg, None, 1)
        mc = monte_carlo(cfg, n=n_mc)
        e, t, a, s, i = r["economico"], r["tecnico"], r["ambiental"], r["social"], r["indice"]
        linhas.append({"Alternativa": nome, "Descrição": cfg.projeto, "VPL": float(e["vpl"][0]),
                       "P(VPL>0)": mc["estatisticas"]["prob_vpl_positivo"], "OEE": float(t["oee"][0]),
                       "PH cinza (m³/t)": float(a["ph_cinza_m3_t"][0]), "kgCO2e/kg": float(a["intensidade_carbono_kg_kg"][0]),
                       "LSO": float(s["lso_0_100"]), "RVL (%)": float(s["rvl_pct"][0]),
                       "Índice": float(i["indice_evteas"][0])})
    df = pd.DataFrame(linhas)
    crit = [c for c, _ in CRITERIOS_TOPSIS]
    sentidos = [s for _, s in CRITERIOS_TOPSIS]
    w = pesos_criterios if pesos_criterios is not None else np.ones(len(crit))
    df["TOPSIS"] = topsis(df[crit].to_numpy(), w, sentidos)
    df = df.sort_values("TOPSIS", ascending=False).reset_index(drop=True)
    df["Posição"] = [f"{k}º" for k in range(1, len(df) + 1)]
    return df

#### Módulo `relatorios` — marcadores [R-xx], tabelas, figuras, exportações e preenchimento do .docx

In [ ]:
%%writefile evteas_py/relatorios.py
"""EVTEAS-Py — módulo relatorios.

Resumo executivo, tabelas no formato da dissertação, marcadores de resultados ([R-xx]) para
inserção no texto, figuras, workbook Excel, JSON, relatório Markdown e preenchimento automático
de um arquivo .docx que contenha os marcadores (Etapa 12 da DSR — comunicação dos resultados).
"""
from __future__ import annotations

import json
import math
import os
import re
import zipfile
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from .config import EVTEASConfig, VERSAO, fmt_num, fmt_pct, fmt_rs
from .entradas import marcadores_entradas, tabela_entradas
from .incerteza import rotulo
from .ponderacao import KPIS, NOMES_DIMENSOES

DIAG_TIR = {"convencional": "convencional", "multiplas": "múltiplas", "multiplas_possiveis": "múltiplas possíveis",
            "sem_solucao": "sem solução"}


def _f(x) -> float:
    try:
        v = np.atleast_1d(np.asarray(x, dtype=float))[0]
        return float(v)
    except (TypeError, ValueError):
        return float("nan")


def _mes(v) -> str:
    return "não recupera no horizonte" if not math.isfinite(_f(v)) else fmt_num(_f(v), 1)


# ---------------------------------------------------------------------------------------------
# Marcadores de resultados para o texto da dissertação
# ---------------------------------------------------------------------------------------------
def marcadores(resultado: Dict[str, Any], cfg: EVTEASConfig, alternativas: Optional[pd.DataFrame] = None,
               testes: Optional[pd.DataFrame] = None,
               configs_alternativas: Optional[Dict[str, EVTEASConfig]] = None) -> pd.DataFrame:
    """Tabela Marcador | Descrição | Valor | Seção.

    No texto da dissertação, [E-xx] são as entradas informadas pelo usuário e [R-xx] os resultados
    da simulação; ambos são preenchidos depois da execução, com os valores desta tabela.
    """
    T, E, A, S = resultado["tecnico"], resultado["economico"], resultado["ambiental"], resultado["social"]
    G, LG, I, D = resultado["governanca"], resultado["lean_green"], resultado["indice"], resultado["decisao"]
    e, t = cfg.economico, cfg.tecnico
    L: List[tuple] = []
    add = lambda m, desc, val, sec: L.append((m, desc, val, sec))

    # Entradas informadas pelo usuário (Quadro de premissas, Seção 4.2) — marcadores [E-xx]
    for linha in marcadores_entradas(cfg, configs_alternativas):
        L.append(linha)

    # Técnica (4.3)
    add("R-T01", "Restrição ativa da biomassa", T["restricao_ativa"], "4.3")
    add("R-T02", "Juvenis estocados por ciclo", fmt_num(_f(T["alevinos_por_ciclo"]), 0), "4.3")
    add("R-T03", "Densidade de estocagem (peixes/m³)", fmt_num(_f(T["densidade_peixes_m3"]), 2), "4.3")
    add("R-T04", "Despesca por ciclo (kg)", fmt_num(_f(T["biomassa_despesca_ciclo_kg"]), 0), "4.3")
    add("R-T05", "Produção anual (kg/ano)", fmt_num(_f(T["producao_ano_kg"]), 0), "4.3")
    add("R-T06", "Produtividade por área (kg/m²/ano)", fmt_num(_f(T["produtividade_kg_m2_ano"]), 2), "4.3")
    add("R-T07", "Produtividade relativa ao potencial do sistema", fmt_pct(_f(T["produtividade_relativa"])), "4.3")
    add("R-T08", "OEE aquícola", fmt_num(_f(T["oee"]), 3), "4.3")
    add("R-T09", "Disponibilidade; performance; sobrevivência",
        f"{fmt_num(_f(T['disponibilidade']), 4)}; {fmt_num(_f(T['performance']), 2)}; {fmt_num(_f(T['sobrevivencia']), 2)}", "4.3")
    add("R-T10", "Consumo de ração (kg/ano)", fmt_num(_f(T["racao_ano_kg"]), 0), "4.3")
    add("R-T11", "Desperdício: ração excedente (R$/ano)", fmt_rs(_f(LG["racao_excedente_rs"])), "4.3")
    add("R-T12", "Desperdício: mortalidade evitável (R$/ano)", fmt_rs(_f(LG["mortalidade_evitavel_rs"])), "4.3")
    add("R-T13", "Desperdício: energia excedente (R$/ano)", fmt_rs(_f(LG["energia_excedente_rs"])), "4.3")
    add("R-T14", "Desperdícios Lean-Green totais (R$/ano)", fmt_rs(_f(LG["desperdicio_total_rs"])), "4.3")
    add("R-T15", "Desperdícios em % do custo operacional", fmt_pct(_f(LG["desperdicio_pct_custo_operacional"])), "4.3")

    # Econômica (4.4)
    add("R-E01", "Meses até a primeira despesca", str(E["meses_ate_despesca"]), "4.4")
    add("R-E02", "Investimento inicial do projeto (CAPEX + capital de giro)", fmt_rs(_f(E["investimento"])), "4.4")
    add("R-E03", "Investimento do beneficiário (com fomento)", fmt_rs(_f(E["investimento_beneficiario"])), "4.4")
    add("R-E04", "VPL do projeto", fmt_rs(_f(E["vpl"])), "4.4")
    add("R-E05", "VPL do beneficiário", fmt_rs(_f(E["vpl_beneficiario"])), "4.4")
    add("R-E06", "TIR anual do projeto (diagnóstico)", f"{fmt_pct(_f(E['tir_aa']))} ({DIAG_TIR.get(str(E['diagnostico_tir']), E['diagnostico_tir'])})", "4.4")
    add("R-E07", "TIR anual do beneficiário", fmt_pct(_f(E["tir_aa_beneficiario"])), "4.4")
    add("R-E08", "Payback descontado do projeto (meses)", _mes(E["payback_descontado_meses"]), "4.4")
    add("R-E09", "Payback descontado do beneficiário (meses)", _mes(E["payback_descontado_beneficiario_meses"]), "4.4")
    add("R-E10", "Payback simples do projeto (meses)", _mes(E["payback_simples_meses"]), "4.4")
    add("R-E11", "Índice de lucratividade", fmt_num(_f(E["indice_lucratividade"]), 3), "4.4")
    add("R-E12", f"Receita bruta no ano de regime (ano {e.ano_regime})", fmt_rs(_f(E["receita_regime"])), "4.4")
    add("R-E13", "Resultado (sobras) líquido no ano de regime", fmt_rs(_f(E["lucro_liquido_regime"])), "4.4")
    add("R-E14", "ROI anual em regime", fmt_pct(_f(E["roi_regime"])), "4.4")
    add("R-E15", "Ponto de equilíbrio (kg/ano)", fmt_num(_f(E["ponto_equilibrio_kg_ano"]), 0), "4.4")
    add("R-E16", "Margem de segurança", fmt_pct(_f(E["margem_seguranca"])), "4.4")
    add("R-E17", "Custo total de produção (R$/kg)", fmt_num(_f(E["custo_total_kg"]), 2), "4.4")
    add("R-E18", "Custo operacional efetivo (R$/kg)", fmt_num(_f(E["custo_operacional_kg"]), 2), "4.4")
    add("R-E19", "Participação da ração no custo operacional", fmt_pct(_f(E["participacao_racao_custo_op"])), "4.4")
    add("R-E20", "TMA (% a.a.)", fmt_num(e.tma_aa_pct, 1) + "%", "4.4")

    # Ambiental (4.5)
    add("R-A01", "Água captada (m³/ano)", fmt_num(_f(A["agua_captada_m3_ano"]), 0), "4.5")
    add("R-A02", "Índice de uso da água (kg/m³)", fmt_num(_f(A["indice_uso_agua_kg_m3"]), 3), "4.5")
    add("R-A03", "Pegada hídrica azul (m³/t)", fmt_num(_f(A["ph_azul_m3_t"]), 0), "4.5")
    add("R-A04", "Pegada hídrica cinza (m³/t)", fmt_num(_f(A["ph_cinza_m3_t"]), 0), "4.5")
    add("R-A05", "Poluente crítico da pegada cinza", str(np.atleast_1d(A["poluente_critico"])[0]), "4.5")
    add("R-A06", "Fósforo aportado pela ração (kg/ano)", fmt_num(_f(A["p_aportado_kg"]), 0), "4.5")
    add("R-A07", "Fósforo retido na biomassa (kg/ano)", fmt_num(_f(A["p_retido_kg"]), 0), "4.5")
    add("R-A08", "Fósforo lançado após tratamento (kg/ano)", fmt_num(_f(A["p_lancado_kg"]), 0), "4.5")
    add("R-A09", "Nitrogênio lançado após tratamento (kg/ano)", fmt_num(_f(A["n_lancado_kg"]), 0), "4.5")
    add("R-A10", "Emissões totais (tCO2e/ano)", fmt_num(_f(A["co2_total_kg"]) / 1000, 1), "4.5")
    add("R-A11", "Emissões: ração / energia / diesel (tCO2e)",
        f"{fmt_num(_f(A['co2_racao_kg']) / 1000, 1)} / {fmt_num(_f(A['co2_energia_kg']) / 1000, 1)} / {fmt_num(_f(A['co2_diesel_kg']) / 1000, 1)}", "4.5")
    add("R-A12", "Participação da ração nas emissões", fmt_pct(_f(A["co2_racao_kg"]) / _f(A["co2_total_kg"])), "4.5")
    add("R-A13", "Intensidade de carbono (kgCO2e/kg)", fmt_num(_f(A["intensidade_carbono_kg_kg"]), 2), "4.5")
    add("R-A14", "Ecoeficiência (R$ de valor adicionado/kgCO2e)", fmt_num(_f(A["ecoeficiencia_rs_kgco2e"]), 2), "4.5")
    add("R-A15", "Score de conformidade ambiental (0–10)", fmt_num(_f(A["score_conformidade_0_10"]), 2), "4.5")
    nao = A["checklist"][A["checklist"]["Atendido"] == "não"]["Item"].tolist()
    add("R-A16", "Itens do checklist ambiental não atendidos", "; ".join(nao) if nao else "nenhum", "4.5")
    add("R-A17", "Itens eliminatórios violados", "; ".join(A["vetos_ambientais"]) if A["vetos_ambientais"] else "nenhum", "4.5")

    # Social e ESG (4.6)
    add("R-S01", "Retenção de Valor Local (RVL)", fmt_pct(_f(S["rvl_pct"]) / 100), "4.6")
    add("R-S02", "Índice de Impacto Local (renda local/receita)", fmt_pct(_f(S["indice_impacto_local"])), "4.6")
    add("R-S03", "Renda mensal média por trabalhador", fmt_rs(_f(S["renda_mensal_trabalhador"])), "4.6")
    add("R-S04", "Renda em salários mínimos", fmt_num(_f(S["renda_salarios_minimos"]), 2), "4.6")
    add("R-S05", "Licença Social para Operar (0–100)", fmt_num(S["lso_0_100"], 1), "4.6")
    add("R-S06", "Score de risco social (0–1)", fmt_num(S["risco_social"], 3), "4.6")
    add("R-S07", "Radar Social (média dos seis eixos)", fmt_num(_f(S["radar_social_media"]), 3), "4.6")
    radar = {k: _f(v) for k, v in S["radar"].items()}
    menores = sorted(radar, key=radar.get)[:2]
    add("R-S08", "Eixos de menor valor no Radar Social", " e ".join(m.lower() for m in menores), "4.6")
    ods = resultado["ods"]
    atend = ods[ods["Atendido"] == "sim"]["ODS"].tolist()
    nao_ods = ods[ods["Atendido"] == "não"]["ODS"].tolist()
    add("R-S09", "ODS atendidos", f"{len(atend)} de {len(ods)} ({', '.join(atend)})", "4.6")
    add("R-S10", "ODS não atendidos", ", ".join(nao_ods) if nao_ods else "nenhum", "4.6")
    add("R-S11", "Governança: itens atendidos e score", f"{G['itens_atendidos']} de {G['itens_total']} ({fmt_num(G['score_governanca'], 3)})", "4.6")
    pil = I["pilares_esg"]
    add("R-S12", "Pilares ESG: ambiental / social / governança",
        f"{fmt_num(_f(pil['ambiental']), 3)} / {fmt_num(_f(pil['social']), 3)} / {fmt_num(_f(pil['governanca']), 3)}", "4.6")
    add("R-S13", "Score ESG", fmt_num(_f(I["score_esg"]), 3), "4.6")

    # Índice e decisão (4.7)
    sc = I["scores"]
    pw = I["pesos"]["dimensoes"]
    add("R-I01", "Método e pesos dimensionais (T/E/A/S)",
        f"{I['pesos']['metodo']}: {fmt_num(pw['tecnica'], 3)} / {fmt_num(pw['economica'], 3)} / {fmt_num(pw['ambiental'], 3)} / {fmt_num(pw['social'], 3)}", "4.7")
    add("R-I02", "Score técnico", fmt_num(_f(sc["tecnica"]), 3), "4.7")
    add("R-I03", "Score econômico", fmt_num(_f(sc["economica"]), 3), "4.7")
    add("R-I04", "Score ambiental", fmt_num(_f(sc["ambiental"]), 3), "4.7")
    add("R-I05", "Score social", fmt_num(_f(sc["social"]), 3), "4.7")
    add("R-I06", "Índice EVTEAS", fmt_num(_f(I["indice_evteas"]), 3), "4.7")
    norm = resultado["normalizacao"].sort_values("Normalizado")
    add("R-I07", "KPIs de pior desempenho relativo", " e ".join(norm["KPI"].head(2).tolist()), "4.7")
    add("R-I08", "Classificação final", D["classificacao"], "4.7")
    add("R-I09", "Vetos aplicados", "; ".join(D["vetos"]) if D["vetos"] else "nenhum", "4.7")
    add("R-I10", "Ressalvas", "; ".join(D["ressalvas"]) if D["ressalvas"] else "nenhuma", "4.7")
    if I["pesos"]["metodo"] == "likert":
        add("R-I11", "Concordância média entre avaliadores (Likert)", fmt_num(I["pesos"]["concordancia_media"], 3), "4.7")
    if I["pesos"]["metodo"] == "ahp":
        add("R-I11", "Razão de consistência AHP", fmt_num(I["pesos"]["razao_consistencia"], 3), "4.7")

    # Monte Carlo (4.8)
    if "monte_carlo" in resultado:
        mc = resultado["monte_carlo"]
        st = mc["estatisticas"]
        add("R-M01", "Iterações e semente", f"{fmt_num(mc['n'], 0)} (seed {mc['seed']})", "4.8")
        add("R-M02", "Média do VPL (IC 95% da média)", f"{fmt_rs(st['media'])} ({fmt_rs(st['ic_inf'])} a {fmt_rs(st['ic_sup'])})", "4.8")
        add("R-M03", "Mediana; desvio-padrão; coeficiente de variação",
            f"{fmt_rs(st['mediana'])}; {fmt_rs(st['desvio_padrao'])}; {fmt_num(st['coef_variacao'], 2)}", "4.8")
        add("R-M04", "Percentis P5 / P50 / P95 do VPL", f"{fmt_rs(st['p5'])} / {fmt_rs(st['p50'])} / {fmt_rs(st['p95'])}", "4.8")
        add("R-M05", "VaR 5% e CVaR 5% do VPL", f"{fmt_rs(st['var_5'])} e {fmt_rs(st['cvar_5'])}", "4.8")
        add("R-M06", "P(VPL > 0) = P(TIR > TMA)", fmt_pct(st["prob_vpl_positivo"]), "4.8")
        add("R-M07", "TIR anual: P5 / P50 / P95", f"{fmt_pct(st['tir_p5'])} / {fmt_pct(st['tir_p50'])} / {fmt_pct(st['tir_p95'])}", "4.8")
        add("R-M08", "Índice EVTEAS: média; P(índice ≥ limiar)", f"{fmt_num(st['indice_media'], 3)}; {fmt_pct(st['prob_indice_viavel'])}", "4.8")
        add("R-M09", "VPL do beneficiário: média; P(VPL > 0)", f"{fmt_rs(st['vpl_beneficiario_media'])}; {fmt_pct(st['prob_vpl_beneficiario_positivo'])}", "4.8")
        add("R-M10", "Erro-padrão da média do VPL", f"{fmt_rs(st['erro_padrao'])} ({fmt_pct(st['erro_padrao'] / st['desvio_padrao'] if st['desvio_padrao'] else float('nan'))} do desvio-padrão)", "4.8")
        imp = mc["importancia"]
        add("R-M11", "Três variáveis mais importantes (Spearman)",
            "; ".join(f"{r['Rótulo']} ({fmt_num(r['Spearman (VPL)'], 3)})" for _, r in imp.head(3).iterrows()), "4.9")

    # Sensibilidade, valores críticos e cenários (4.9)
    if "sensibilidade" in resultado:
        sens = resultado["sensibilidade"]
        top = sens.iloc[0]
        add("R-V01", "Variável de maior impacto (±10%)", top["Rótulo"], "4.9")
        add("R-V02", "Variação do VPL para ±10% na variável de maior impacto", fmt_rs(abs(top["ΔVPL (+10%)"])), "4.9")
        vc = resultado["valores_criticos"].set_index("Variável")
        if "economico.preco_venda_kg" in vc.index:
            p = vc.loc["economico.preco_venda_kg"]
            add("R-V03", "Preço crítico (VPL = 0) e folga", f"{fmt_rs(p['Valor crítico (VPL = 0)'], 2)}/kg ({fmt_pct(p['Folga relativa'])})", "4.9")
        if "tecnico.fcr" in vc.index:
            p = vc.loc["tecnico.fcr"]
            add("R-V04", "FCR crítico e folga", f"{fmt_num(p['Valor crítico (VPL = 0)'], 3)} ({fmt_pct(p['Folga relativa'])})", "4.9")
        if "tecnico.mortalidade_pct" in vc.index:
            p = vc.loc["tecnico.mortalidade_pct"]
            add("R-V05", "Mortalidade crítica", f"{fmt_num(p['Valor crítico (VPL = 0)'], 1)}%", "4.9")
        cen = resultado["cenarios"].set_index("Cenário")
        add("R-V06", "Cenário pessimista: VPL; índice", f"{fmt_rs(cen.loc['Pessimista', 'VPL'])}; {fmt_num(cen.loc['Pessimista', 'Índice EVTEAS'], 3)}", "4.9")
        add("R-V07", "Cenário otimista: VPL; índice", f"{fmt_rs(cen.loc['Otimista', 'VPL'])}; {fmt_num(cen.loc['Otimista', 'Índice EVTEAS'], 3)}", "4.9")

    # Alternativas (4.9)
    if alternativas is not None and len(alternativas):
        alt = alternativas
        best = alt.iloc[0]
        add("R-X01", "Ordem TOPSIS das alternativas", " > ".join(alt["Alternativa"].tolist()), "4.9")
        add("R-X02", "Alternativa preferida e proximidade à solução ideal", f"{best['Alternativa']} ({fmt_num(best['TOPSIS'], 3)})", "4.9")
        ix = alt.set_index("Alternativa")
        for nome in ix.index:
            add(f"R-X{nome}", f"Alternativa {nome}: VPL; P(VPL>0); PH cinza; kgCO2e/kg",
                f"{fmt_rs(ix.loc[nome, 'VPL'])}; {fmt_pct(ix.loc[nome, 'P(VPL>0)'])}; {fmt_num(ix.loc[nome, 'PH cinza (m³/t)'], 0)} m³/t; {fmt_num(ix.loc[nome, 'kgCO2e/kg'], 2)}", "4.9")
        add("R-X03", "Maior probabilidade de VPL positivo entre as alternativas",
            f"{fmt_pct(alt['P(VPL>0)'].max())} (alternativa {alt.loc[alt['P(VPL>0)'].idxmax(), 'Alternativa']})", "4.9")

    # Verificação (4.10) e execução (4.11)
    vv = resultado["vv"]
    add("R-W01", "Invariantes aprovados", f"{vv['aprovadas']} de {vv['total']}", "4.10")
    if testes is not None and len(testes):
        add("R-W02", "Testes automatizados aprovados", f"{int(testes['Aprovado'].sum())} de {len(testes)}", "4.10")
    add("R-W03", "Tempo de execução do estudo (s)", fmt_num(resultado.get("tempo_execucao_s", float("nan")), 1), "4.11")
    add("R-W04", "Versão do EVTEAS-Py", VERSAO, "4.11")

    for chave, texto in textos_interpretativos(resultado, cfg, alternativas).items():
        add(chave, "Texto interpretativo sugerido (revise antes de usar)", texto, "4.x")
    return pd.DataFrame(L, columns=["Marcador", "Descrição", "Valor", "Seção"])


def textos_interpretativos(resultado: Dict[str, Any], cfg: EVTEASConfig,
                           alternativas: Optional[pd.DataFrame] = None) -> Dict[str, str]:
    """Frases geradas por regras explícitas, para apoiar (não substituir) a interpretação."""
    E, D = resultado["economico"], resultado["decisao"]
    tma = cfg.economico.tma_aa_pct / 100
    vpl, tir_aa = _f(E["vpl"]), _f(E["tir_aa"])
    txt = {}
    if vpl >= 0:
        txt["R-TXT01"] = (f"Na perspectiva do projeto, o VPL de {fmt_rs(vpl)} e a TIR de {fmt_pct(tir_aa)} a.a., superior "
                          f"à TMA de {fmt_pct(tma)} a.a., indicam viabilidade econômica no cenário determinístico.")
    else:
        txt["R-TXT01"] = (f"Na perspectiva do projeto, o VPL de {fmt_rs(vpl)} e a TIR de {fmt_pct(tir_aa)} a.a., inferior "
                          f"à TMA de {fmt_pct(tma)} a.a., indicam que o projeto não remunera o capital à taxa mínima de atratividade no cenário determinístico.")
    if "monte_carlo" in resultado:
        st = resultado["monte_carlo"]["estatisticas"]
        rel = "abaixo" if st["media"] < vpl else "acima"
        txt["R-TXT02"] = (f"A média do VPL simulado ({fmt_rs(st['media'])}) ficou {rel} do VPL calculado na moda das premissas "
                          f"({fmt_rs(vpl)}), e a probabilidade de VPL positivo foi de {fmt_pct(st['prob_vpl_positivo'])}"
                          + (", o que evidencia o efeito das distribuições assimétricas desfavoráveis." if rel == "abaixo"
                             else ", o que indica distribuições com caudas favoráveis."))
        txt["R-TXT03"] = (f"Na perspectiva do beneficiário, com fomento de {fmt_num(cfg.economico.fomento_nao_reembolsavel_pct, 0)}% do CAPEX, "
                          f"a probabilidade de VPL positivo passa a {fmt_pct(st['prob_vpl_beneficiario_positivo'])}.")
    if "sensibilidade" in resultado:
        s = resultado["sensibilidade"]
        rot = [r[0].lower() + r[1:] for r in s["Rótulo"].head(3)]
        txt["R-TXT04"] = ("As variáveis de maior impacto sobre o VPL foram " + ", ".join(rot[:-1]) + " e " + rot[-1]
                          + ", que concentram as prioridades de coleta de dados e de gestão de risco.")
    txt["R-TXT05"] = (f"O estudo foi classificado como \"{D['classificacao']}\""
                      + (f", em razão de: {'; '.join(D['vetos'])}" if D["vetos"] else "")
                      + (f"; ressalvas: {'; '.join(D['ressalvas'])}" if D["ressalvas"] else "") + ".")
    if alternativas is not None and len(alternativas):
        b = alternativas.iloc[0]
        txt["R-TXT06"] = (f"A alternativa {b['Alternativa']} obteve a maior proximidade à solução ideal ({fmt_num(b['TOPSIS'], 3)}), "
                          f"com VPL de {fmt_rs(b['VPL'])} e probabilidade de VPL positivo de {fmt_pct(b['P(VPL>0)'])}.")
    return txt


# ---------------------------------------------------------------------------------------------
# Tabelas no formato da dissertação
# ---------------------------------------------------------------------------------------------
def tabelas_dissertacao(resultado: Dict[str, Any], cfg: EVTEASConfig,
                        alternativas: Optional[pd.DataFrame] = None) -> Dict[str, pd.DataFrame]:
    m = marcadores(resultado, cfg, alternativas).set_index("Marcador")["Valor"]
    v = lambda k: m.get(k, "n/d")
    tab = {}
    tab["Tabela - Indicadores econômicos"] = pd.DataFrame([
        ["Investimento inicial (CAPEX + capital de giro)", v("R-E02"), v("R-E03")],
        [f"VPL (TMA de {v('R-E20')} a.a.)", v("R-E04"), v("R-E05")],
        ["TIR anual (diagnóstico)", v("R-E06"), v("R-E07")],
        ["Payback descontado (meses)", v("R-E08"), v("R-E09")],
        ["Payback simples (meses)", v("R-E10"), "—"],
        ["Índice de lucratividade", v("R-E11"), "—"],
        [f"Receita bruta no ano de regime (ano {cfg.economico.ano_regime})", v("R-E12"), "—"],
        ["Resultado (sobras) líquido no ano de regime", v("R-E13"), "—"],
        ["ROI anual em regime (Quadro 5)", v("R-E14"), "—"],
        ["Ponto de equilíbrio (kg/ano) e margem de segurança", f"{v('R-E15')} kg; {v('R-E16')}", "—"],
        ["Custo total e custo operacional efetivo (R$/kg)", f"{v('R-E17')} e {v('R-E18')}", "—"],
    ], columns=["Indicador", "Projeto", "Beneficiário (com fomento)"])
    tab["Tabela - Indicadores ambientais, sociais e de governança"] = pd.DataFrame([
        ["Ambiental", "Água captada / índice de uso da água", f"{v('R-A01')} m³/ano; {v('R-A02')} kg/m³"],
        ["Ambiental", "Pegada hídrica azul", f"{v('R-A03')} m³/t"],
        ["Ambiental", "Pegada hídrica cinza (poluente crítico)", f"{v('R-A04')} m³/t ({v('R-A05')})"],
        ["Ambiental", "N e P lançados após tratamento", f"{v('R-A09')} kg N; {v('R-A08')} kg P"],
        ["Ambiental", "Emissões (ração / energia / diesel)", f"{v('R-A10')} tCO2e/ano ({v('R-A11')})"],
        ["Ambiental", "Intensidade de carbono; ecoeficiência", f"{v('R-A13')} kgCO2e/kg; R$ {v('R-A14')}/kgCO2e"],
        ["Ambiental", "Score de conformidade ambiental", f"{v('R-A15')} de 10"],
        ["Social", "Retenção de Valor Local (RVL)", f"{v('R-S01')} do custo operacional"],
        ["Social", "Índice de Impacto Local", f"{v('R-S02')} da receita retorna como renda local"],
        ["Social", "Renda mensal por trabalhador", f"{v('R-S03')} ({v('R-S04')} salários mínimos)"],
        ["Social", "LSO; score de risco social", f"{v('R-S05')} de 100; {v('R-S06')}"],
        ["Social", "Radar Social (média dos seis eixos)", v("R-S07")],
        ["Governança", "Checklist do pilar G", v("R-S11")],
    ], columns=["Dimensão", "Indicador", "Resultado"])
    norm = resultado["normalizacao"].copy()
    norm["Valor bruto"] = norm["Valor bruto"].map(lambda x: fmt_num(x, 3))
    norm["Normalizado"] = norm["Normalizado"].map(lambda x: fmt_num(x, 3))
    tab["Quadro - Normalização dos KPIs"] = norm
    if "monte_carlo" in resultado:
        tab["Tabela - Estatísticas da simulação de Monte Carlo"] = pd.DataFrame([
            ["Média do VPL (IC 95% da média)", v("R-M02")], ["Mediana; desvio-padrão; coeficiente de variação", v("R-M03")],
            ["Percentis P5 / P50 / P95", v("R-M04")], ["VaR 5% e CVaR 5% do VPL", v("R-M05")],
            ["P(VPL > 0) = P(TIR > TMA)", v("R-M06")], ["TIR anual: P5 / P50 / P95", v("R-M07")],
            ["Índice EVTEAS: média; P(índice ≥ 0,60)", v("R-M08")], ["VPL do beneficiário: média; P(VPL > 0)", v("R-M09")],
            ["Erro-padrão da média", v("R-M10")],
        ], columns=["Estatística", "Valor"])
    if "sensibilidade" in resultado:
        s = resultado["sensibilidade"].copy()
        vc = resultado["valores_criticos"].set_index("Variável")
        imp = resultado.get("monte_carlo", {}).get("importancia")
        rho = imp.set_index("Variável")["Spearman (VPL)"] if imp is not None else pd.Series(dtype=float)
        linhas = []
        for _, r in s.iterrows():
            var = r["Variável"]
            crit = "—"
            if var in vc.index and np.isfinite(vc.loc[var, "Valor crítico (VPL = 0)"]):
                crit = f"{fmt_num(vc.loc[var, 'Valor crítico (VPL = 0)'], 3)} ({fmt_pct(vc.loc[var, 'Folga relativa'])})"
            linhas.append([r["Rótulo"], fmt_rs(r["ΔVPL (-10%)"]), fmt_rs(r["ΔVPL (+10%)"]), crit,
                           fmt_num(rho[var], 3) if var in rho.index else "—"])
        tab["Tabela - Sensibilidade, valores críticos e importância"] = pd.DataFrame(
            linhas, columns=["Variável", "ΔVPL (−10%)", "ΔVPL (+10%)", "Valor crítico (VPL = 0)", "Spearman (MC)"])
    if alternativas is not None and len(alternativas):
        a = alternativas.copy()
        tab["Tabela - Comparação de alternativas por TOPSIS"] = pd.DataFrame({
            "Alternativa": a["Alternativa"], "VPL": a["VPL"].map(fmt_rs), "P(VPL>0)": a["P(VPL>0)"].map(fmt_pct),
            "PH cinza (m³/t)": a["PH cinza (m³/t)"].map(lambda x: fmt_num(x, 0)),
            "kgCO2e/kg": a["kgCO2e/kg"].map(lambda x: fmt_num(x, 2)), "RVL (%)": a["RVL (%)"].map(lambda x: fmt_num(x, 1)),
            "Índice": a["Índice"].map(lambda x: fmt_num(x, 3)), "TOPSIS": a["TOPSIS"].map(lambda x: fmt_num(x, 3)),
            "Posição": a["Posição"]})
    return tab


# ---------------------------------------------------------------------------------------------
# Figuras
# ---------------------------------------------------------------------------------------------
FIGURAS = {
    "FIG-CRESCIMENTO": ("fig_curva_crescimento.png", "Curva de crescimento, biomassa e ração acumulada no ciclo"),
    "FIG-ECONOMICO": ("fig_receita_custos_fluxo.png", "Receita, custos operacionais e fluxo de caixa acumulado"),
    "FIG-RADAR": ("fig_radar_social.png", "Radar Social"),
    "FIG-INDICE": ("fig_scores_indice.png", "Scores dimensionais e índice EVTEAS"),
    "FIG-VPL": ("fig_distribuicao_vpl.png", "Distribuição e função de probabilidade acumulada do VPL"),
    "FIG-CONVERGENCIA": ("fig_convergencia.png", "Convergência da média do VPL com intervalo de confiança de 95%"),
    "FIG-TORNADO": ("fig_tornado.png", "Diagrama de tornado da sensibilidade do VPL a ±10%"),
    "FIG-TOPSIS": ("fig_topsis.png", "Proximidade relativa à solução ideal das alternativas de projeto"),
    "FIG-LEANGREEN": ("fig_lean_green.png", "Desperdícios Lean-Green evitáveis"),
}


def _mpl():
    import matplotlib
    if "inline" not in matplotlib.get_backend().lower():
        try:
            matplotlib.use("Agg")
        except Exception:  # noqa: BLE001
            pass
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 100,
                         "axes.spines.top": False, "axes.spines.right": False})
    return plt


def gerar_figuras(resultado: Dict[str, Any], cfg: EVTEASConfig, pasta: str,
                  alternativas: Optional[pd.DataFrame] = None, mostrar: bool = False) -> Dict[str, str]:
    """Gera as figuras em PNG (300 dpi) e devolve {marcador: caminho}."""
    plt = _mpl()
    from matplotlib.ticker import FuncFormatter
    Path(pasta).mkdir(parents=True, exist_ok=True)
    from .entradas import fmt_auto
    reais = FuncFormatter(lambda x, _: fmt_num(x / 1000, 0) + " mil")
    virg = FuncFormatter(lambda x, _: fmt_auto(round(float(x), 6)))   # vírgula decimal (pt-BR)
    saida = {}

    def salvar(fig, chave):
        caminho = os.path.join(pasta, FIGURAS[chave][0])
        for ax_ in fig.axes:
            if getattr(ax_, "name", "") == "polar":
                continue
            for eixo in (ax_.xaxis, ax_.yaxis):
                if not isinstance(eixo.get_major_formatter(), FuncFormatter) and eixo.get_scale() == "linear":
                    eixo.set_major_formatter(virg)
        fig.tight_layout()
        fig.savefig(caminho, dpi=300, bbox_inches="tight")
        if mostrar:
            plt.show()
        plt.close(fig)
        saida[chave] = caminho

    c = resultado["curva_crescimento"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(c["Dia"], c["Peso médio (g)"], color="#1f77b4", label="Peso médio (g)")
    ax.set_xlabel("Dia do ciclo")
    ax.set_ylabel("Peso médio (g)")
    ax2 = ax.twinx()
    ax2.plot(c["Dia"], c["Biomassa (kg)"], color="#2ca02c", label="Biomassa (kg)")
    ax2.plot(c["Dia"], c["Ração acumulada (kg)"], color="#ff7f0e", ls="--", label="Ração acumulada (kg)")
    ax2.set_ylabel("kg")
    ax2.grid(False)
    linhas = ax.get_lines() + ax2.get_lines()
    ax.legend(linhas, [l.get_label() for l in linhas], loc="upper left")
    salvar(fig, "FIG-CRESCIMENTO")

    dre = resultado["dre_anual"]
    fluxo = np.asarray(resultado["fluxo_caixa"])
    acum = np.cumsum(fluxo)
    fig, ax = plt.subplots(figsize=(8, 4))
    anos = dre["Ano"].to_numpy()
    ax.bar(anos - 0.7, dre["Receita Bruta"], width=0.4, label="Receita bruta (ano)", color="#4c9be8")
    ax.bar(anos - 0.3, dre["Custos Variáveis"] + dre["Custos Fixos"], width=0.4, label="Custos operacionais (ano)",
           color="#f2a65a")
    ax.plot(np.arange(len(acum)) / 12, acum, color="#333333", label="Fluxo de caixa acumulado")
    ax.axhline(0, color="black", lw=0.8)
    ax.yaxis.set_major_formatter(reais)
    ax.set_xlabel("Ano")
    ax.set_ylabel("R$")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
    salvar(fig, "FIG-ECONOMICO")

    radar = {k: _f(v) for k, v in resultado["social"]["radar"].items()}
    nomes = list(radar)
    vals = list(radar.values()) + [list(radar.values())[0]]
    ang = np.linspace(0, 2 * np.pi, len(nomes), endpoint=False).tolist()
    ang += ang[:1]
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, polar=True)
    ax.plot(ang, vals, color="#2ca02c")
    ax.fill(ang, vals, color="#2ca02c", alpha=0.25)
    ax.set_xticks(ang[:-1])
    ax.set_xticklabels(nomes)
    ax.set_ylim(0, 1)
    salvar(fig, "FIG-RADAR")

    sc = resultado["indice"]["scores"]
    fig, ax = plt.subplots(figsize=(8, 4))
    nomes = [NOMES_DIMENSOES[d] for d in sc] + ["Índice EVTEAS"]
    vals = [_f(v) for v in sc.values()] + [_f(resultado["indice"]["indice_evteas"])]
    cores = ["#4c9be8"] * len(sc) + ["#2ca02c"]
    ax.bar(nomes, vals, color=cores)
    ax.axhline(cfg.decisao.limiar_indice_viavel, color="#2ca02c", ls="--", lw=1, label="Limiar de viabilidade")
    ax.axhline(cfg.decisao.limiar_indice_ressalvas, color="#d62728", ls=":", lw=1, label="Limiar com ressalvas")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, fmt_num(v, 3), ha="center")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="upper center", ncol=2, frameon=False)
    salvar(fig, "FIG-INDICE")

    if "monte_carlo" in resultado:
        mc = resultado["monte_carlo"]
        v = np.sort(mc["vpl"])
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
        a1.hist(v, bins=60, color="#4c9be8", alpha=0.85)
        a1.axvline(0, color="#d62728", lw=1)
        a1.axvline(np.mean(v), color="black", ls="--", lw=1, label="Média")
        a1.xaxis.set_major_formatter(reais)
        a1.set_xlabel("VPL (R$)")
        a1.set_ylabel("Frequência")
        a1.legend()
        a2.plot(v, np.arange(1, len(v) + 1) / len(v), color="#333333")
        a2.axvline(0, color="#d62728", lw=1)
        a2.xaxis.set_major_formatter(reais)
        a2.set_xlabel("VPL (R$)")
        a2.set_ylabel("Probabilidade acumulada")
        salvar(fig, "FIG-VPL")

        cv = mc["convergencia"]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(cv["Iterações"], cv["Média do VPL"], color="#1f77b4", label="Média acumulada")
        ax.fill_between(cv["Iterações"], cv["IC inferior"], cv["IC superior"], color="#1f77b4", alpha=0.2, label="IC 95%")
        ax.set_xscale("log")
        ax.yaxis.set_major_formatter(reais)
        ax.set_xlabel("Iterações (escala logarítmica)")
        ax.set_ylabel("VPL médio (R$)")
        ax.legend()
        salvar(fig, "FIG-CONVERGENCIA")

    if "sensibilidade" in resultado:
        s = resultado["sensibilidade"].iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, 4.2))
        y = np.arange(len(s))
        ax.barh(y, s["ΔVPL (-10%)"], color="#f2a65a", label="−10%")
        ax.barh(y, s["ΔVPL (+10%)"], color="#4c9be8", label="+10%")
        ax.set_yticks(y)
        ax.set_yticklabels(s["Rótulo"])
        ax.axvline(0, color="black", lw=0.8)
        ax.xaxis.set_major_formatter(reais)
        ax.set_xlabel("Variação do VPL (R$)")
        ax.legend()
        salvar(fig, "FIG-TORNADO")

    if alternativas is not None and len(alternativas):
        a = alternativas.sort_values("TOPSIS")
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.barh(a["Alternativa"], a["TOPSIS"], color="#2ca02c")
        for i, v in enumerate(a["TOPSIS"]):
            ax.text(v + 0.01, i, fmt_num(v, 3), va="center")
        ax.set_xlim(0, 1.1)
        ax.set_xlabel("Proximidade relativa à solução ideal")
        salvar(fig, "FIG-TOPSIS")

    lg = resultado["lean_green"]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    itens = {"Ração excedente": _f(lg["racao_excedente_rs"]), "Mortalidade evitável": _f(lg["mortalidade_evitavel_rs"]),
             "Energia excedente": _f(lg["energia_excedente_rs"])}
    ax.barh(list(itens), list(itens.values()), color="#f2a65a")
    for i, v in enumerate(itens.values()):
        ax.text(v, i, " " + fmt_rs(v), va="center")
    ax.set_xlabel("R$/ano")
    salvar(fig, "FIG-LEANGREEN")
    return saida


# ---------------------------------------------------------------------------------------------
# Exportações
# ---------------------------------------------------------------------------------------------
def resumo_executivo(resultado: Dict[str, Any], cfg: EVTEASConfig) -> str:
    m = marcadores(resultado, cfg).set_index("Marcador")["Valor"]
    g = lambda k: m.get(k, "n/d")
    linhas = [
        f"EVTEAS-Py {VERSAO} — {cfg.projeto}",
        f"Classificação: {g('R-I08')} | Índice EVTEAS: {g('R-I06')} | Vetos: {g('R-I09')} | Ressalvas: {g('R-I10')}",
        f"Técnica: produção {g('R-T05')} kg/ano; OEE {g('R-T08')}; FCR {fmt_num(cfg.tecnico.fcr, 2)}",
        f"Econômica: investimento {g('R-E02')}; VPL {g('R-E04')}; TIR {g('R-E06')}; payback descontado {g('R-E08')} meses",
        f"Beneficiário: VPL {g('R-E05')}; TIR {g('R-E07')}",
        f"Ambiental: PH azul {g('R-A03')} m³/t; PH cinza {g('R-A04')} m³/t; {g('R-A13')} kgCO2e/kg; conformidade {g('R-A15')}/10",
        f"Social: RVL {g('R-S01')}; LSO {g('R-S05')}; renda {g('R-S03')} ({g('R-S04')} SM)",
    ]
    if "monte_carlo" in resultado:
        linhas.append(f"Monte Carlo: VPL médio {g('R-M02')}; P(VPL>0) {g('R-M06')}")
    linhas.append(f"Verificação: {g('R-W01')} invariantes aprovados")
    return "\n".join(linhas)


def _jsonavel(x):
    if isinstance(x, dict):
        return {str(k): _jsonavel(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonavel(v) for v in x]
    if isinstance(x, pd.DataFrame):
        return x.to_dict(orient="records")
    if isinstance(x, np.ndarray):
        return x.tolist() if x.size <= 500 else {"tamanho": int(x.size), "media": float(np.nanmean(x))}
    if isinstance(x, (np.floating, float)):
        return float(x) if math.isfinite(float(x)) else None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def exportar_excel(resultado: Dict[str, Any], cfg: EVTEASConfig, caminho: str,
                   alternativas: Optional[pd.DataFrame] = None, testes: Optional[pd.DataFrame] = None,
                   configs_alternativas: Optional[Dict[str, EVTEASConfig]] = None) -> str:
    """Workbook com premissas, resultados e registros da verificação."""
    abas: Dict[str, pd.DataFrame] = {}
    abas["Resumo"] = pd.DataFrame({"Resumo executivo": resumo_executivo(resultado, cfg).split("\n")})
    abas["Marcadores"] = marcadores(resultado, cfg, alternativas, testes, configs_alternativas)
    abas["Entradas"] = tabela_entradas(cfg)
    abas["Curva crescimento"] = resultado["curva_crescimento"]
    abas["DRE mensal"] = resultado["dre_mensal"]
    abas["DRE anual"] = resultado["dre_anual"]
    abas["Fluxo de caixa"] = pd.DataFrame({"Mês": np.arange(len(resultado["fluxo_caixa"])),
                                           "Fluxo": resultado["fluxo_caixa"], "Acumulado": np.cumsum(resultado["fluxo_caixa"])})
    abas["Sobras"] = pd.DataFrame(list(resultado["sobras"].items()), columns=["Destinação", "R$"])
    A = resultado["ambiental"]
    abas["Ambiental"] = pd.DataFrame([(k, _f(v)) for k, v in A.items() if not isinstance(v, (pd.DataFrame, list, str))
                                      and np.size(v) == 1 and not isinstance(np.atleast_1d(v)[0], str)], columns=["Indicador", "Valor"])
    abas["Conformidade"] = A["checklist"]
    S = resultado["social"]
    abas["Social"] = pd.DataFrame([(k, _f(v)) for k, v in S.items() if not isinstance(v, dict) and np.size(v) == 1],
                                  columns=["Indicador", "Valor"])
    abas["Radar social"] = pd.DataFrame([(k, _f(v)) for k, v in S["radar"].items()], columns=["Eixo", "Valor"])
    abas["LSO componentes"] = pd.DataFrame(list(S["componentes_lso"].items()), columns=["Componente", "Valor"])
    abas["Governança"] = resultado["governanca"]["checklist"]
    abas["ODS"] = resultado["ods"]
    abas["Lean-Green"] = pd.DataFrame([(k, _f(v)) for k, v in resultado["lean_green"].items()], columns=["Indicador", "Valor"])
    abas["KPIs normalizados"] = resultado["normalizacao"]
    if "monte_carlo" in resultado:
        mc = resultado["monte_carlo"]
        abas["MC estatísticas"] = pd.DataFrame([(k, v) for k, v in mc["estatisticas"].items()], columns=["Estatística", "Valor"])
        abas["MC amostra"] = mc["amostra"].head(10000)
        abas["MC convergência"] = mc["convergencia"]
        abas["Importância"] = mc["importancia"]
    if "sensibilidade" in resultado:
        abas["Sensibilidade"] = resultado["sensibilidade"]
        abas["Valores críticos"] = resultado["valores_criticos"]
        abas["Cenários"] = resultado["cenarios"]
    if alternativas is not None:
        abas["Alternativas TOPSIS"] = alternativas
    abas["Verificação"] = resultado["vv"]["tabela"]
    if testes is not None:
        abas["Testes"] = testes
    with pd.ExcelWriter(caminho, engine="openpyxl") as w:
        for nome, df in abas.items():
            df.to_excel(w, sheet_name=nome[:31], index=False)
    return caminho


def exportar_json(resultado: Dict[str, Any], caminho: str) -> str:
    dados = {k: v for k, v in resultado.items() if k not in ("dre_mensal",)}
    if "monte_carlo" in dados:
        dados["monte_carlo"] = {k: v for k, v in dados["monte_carlo"].items() if k not in ("amostra", "entradas", "vpl", "indice", "vpl_beneficiario")}
    Path(caminho).write_text(json.dumps(_jsonavel(dados), ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho


def exportar_markdown(resultado: Dict[str, Any], cfg: EVTEASConfig, caminho: str,
                      alternativas: Optional[pd.DataFrame] = None, testes: Optional[pd.DataFrame] = None,
                      configs_alternativas: Optional[Dict[str, EVTEASConfig]] = None) -> str:
    partes = [f"# Resultados do EVTEAS-Py {VERSAO}", "", f"**Projeto:** {cfg.projeto}", "", "```",
              resumo_executivo(resultado, cfg), "```", "", "## Marcadores para o texto da dissertação", ""]
    mk = marcadores(resultado, cfg, alternativas, testes, configs_alternativas)
    partes.append("| Marcador | Descrição | Valor | Seção |")
    partes.append("|---|---|---|---|")
    for _, r in mk.iterrows():
        partes.append(f"| [{r['Marcador']}] | {r['Descrição']} | {str(r['Valor']).replace('|', '/')} | {r['Seção']} |")
    for nome, df in tabelas_dissertacao(resultado, cfg, alternativas).items():
        partes += ["", f"## {nome}", "", df.to_markdown(index=False) if hasattr(df, "to_markdown") and _tem_tabulate() else df.to_string(index=False)]
    Path(caminho).write_text("\n".join(partes), encoding="utf-8")
    return caminho


def _tem_tabulate() -> bool:
    try:
        import tabulate  # noqa: F401
        return True
    except ImportError:
        return False


MARCADOR_DOCX = re.compile(r"\[((?:R|E)-[A-Z0-9]+|FIG-[A-Z]+)\]")


def _substituir_no_paragrafo(p, padrao, funcao) -> None:
    """Substitui ocorrências do padrão no parágrafo mesmo quando o marcador está dividido entre
    vários trechos (runs) do Word; o valor herda a formatação do trecho onde o marcador começa."""
    texto = "".join(r.text for r in p.runs)
    for m in reversed(list(padrao.finditer(texto))):
        novo = funcao(m)
        if novo is None:
            continue
        ini, fim = m.span()
        pos, afetados = 0, []
        for r in p.runs:
            a, b = pos, pos + len(r.text)
            if b > ini and a < fim:
                afetados.append((r, a))
            pos = b
        if not afetados:
            continue
        r0, a0 = afetados[0]
        if len(afetados) == 1:
            r0.text = r0.text[:ini - a0] + novo + r0.text[fim - a0:]
        else:
            r0.text = r0.text[:ini - a0] + novo
            for r, _ in afetados[1:-1]:
                r.text = ""
            rl, al = afetados[-1]
            rl.text = rl.text[fim - al:]
        r0.font.highlight_color = None


def preencher_docx(entrada: str, saida: str, marcadores_df: pd.DataFrame, figuras: Optional[Dict[str, str]] = None,
                   largura_cm: float = 15.0) -> Dict[str, Any]:
    """Substitui os marcadores [E-xx] e [R-xx] de um .docx pelos valores da simulação (mantendo a
    formatação do trecho e removendo o realce) e insere as figuras nos parágrafos com [FIG-...].
    Marcadores sem valor nesta execução permanecem no texto e são listados. Requer python-docx."""
    from docx import Document
    from docx.shared import Cm
    valores = dict(zip(marcadores_df["Marcador"], marcadores_df["Valor"].astype(str)))
    doc = Document(entrada)
    cont = {"trocados": 0, "imagens": 0}
    ausentes = set()

    def paragrafos(container):
        for p in container.paragraphs:
            yield p
        for t in container.tables:
            for row in t.rows:
                for cell in row.cells:
                    yield from paragrafos(cell)

    def valor(m):
        chave = m.group(1)
        if chave.startswith("FIG-"):
            return None
        if chave in valores:
            cont["trocados"] += 1
            return valores[chave]
        ausentes.add(chave)
        return None

    for p in paragrafos(doc):
        if "[" not in p.text:
            continue
        _substituir_no_paragrafo(p, MARCADOR_DOCX, valor)
        figs = re.findall(r"\[(FIG-[A-Z]+)\]", p.text)
        for chave in figs:
            if figuras and chave in figuras and os.path.exists(figuras[chave]):
                _substituir_no_paragrafo(p, re.compile(re.escape(f"[{chave}]")), lambda m: "")
                p.add_run().add_picture(figuras[chave], width=Cm(largura_cm))
                cont["imagens"] += 1
            else:
                ausentes.add(chave)
    doc.save(saida)
    return {"marcadores_substituidos": cont["trocados"], "marcadores_sem_valor": sorted(ausentes),
            "figuras_inseridas": cont["imagens"], "arquivo": saida}


def exportar_tudo(resultado: Dict[str, Any], cfg: EVTEASConfig, pasta: str, figuras: bool = True,
                  alternativas: Optional[pd.DataFrame] = None, testes: Optional[pd.DataFrame] = None,
                  configs_alternativas: Optional[Dict[str, EVTEASConfig]] = None) -> Dict[str, str]:
    """Gera todas as saídas na pasta e um .zip com tudo."""
    from .config import salvar_config
    Path(pasta).mkdir(parents=True, exist_ok=True)
    arq = {
        "excel": exportar_excel(resultado, cfg, os.path.join(pasta, "EVTEAS_resultados.xlsx"), alternativas, testes,
                                configs_alternativas),
        "json": exportar_json(resultado, os.path.join(pasta, "EVTEAS_resultados.json")),
        "markdown": exportar_markdown(resultado, cfg, os.path.join(pasta, "EVTEAS_relatorio.md"), alternativas, testes,
                                      configs_alternativas),
        "config": str(salvar_config(cfg, os.path.join(pasta, "EVTEAS_premissas.json"))),
    }
    mk = marcadores(resultado, cfg, alternativas, testes, configs_alternativas)
    arq["marcadores"] = os.path.join(pasta, "EVTEAS_marcadores.csv")
    mk.to_csv(arq["marcadores"], index=False, encoding="utf-8-sig", sep=";")
    Path(os.path.join(pasta, "EVTEAS_resumo.txt")).write_text(resumo_executivo(resultado, cfg), encoding="utf-8")
    arq["resumo"] = os.path.join(pasta, "EVTEAS_resumo.txt")
    if figuras:
        for k, v in gerar_figuras(resultado, cfg, os.path.join(pasta, "figuras"), alternativas).items():
            arq[k] = v
    zipp = os.path.join(pasta, "EVTEAS_saidas.zip")
    with zipfile.ZipFile(zipp, "w", zipfile.ZIP_DEFLATED) as z:
        for k, v in arq.items():
            z.write(v, arcname=os.path.relpath(v, pasta))
    arq["zip"] = zipp
    return arq


def comparar_pesos(r_preset: Dict[str, Any], r_likert: Dict[str, Any]) -> pd.DataFrame:
    """Marcadores [R-Lxx]: efeito dos pesos Likert dos especialistas (obtidos nos questionários,
    após a simulação) sobre o índice e a classificação, em comparação com o preset."""
    ip, il = r_preset["indice"], r_likert["indice"]
    pl = il["pesos"]
    w = pl["dimensoes"]
    linhas = [
        ("R-L01", "Pesos dimensionais Likert (T/E/A/S)",
         f"{fmt_num(w['tecnica'], 3)} / {fmt_num(w['economica'], 3)} / {fmt_num(w['ambiental'], 3)} / {fmt_num(w['social'], 3)}"),
        ("R-L02", "Concordância média entre os especialistas", fmt_num(pl.get("concordancia_media"), 3)),
        ("R-L03", "Índice EVTEAS: preset → Likert",
         f"{fmt_num(_f(ip['indice_evteas']), 3)} → {fmt_num(_f(il['indice_evteas']), 3)}"),
        ("R-L04", "Classificação: preset → Likert",
         f"{r_preset['decisao']['classificacao']} → {r_likert['decisao']['classificacao']}"),
    ]
    conc = pl.get("concordancia_por_kpi", {})
    if conc:
        baixos = [KPIS[k][1] for k, v in sorted(conc.items(), key=lambda kv: kv[1])[:3]]
        linhas.append(("R-L05", "KPIs de menor concordância entre os especialistas", "; ".join(baixos)))
    return pd.DataFrame([(m, d, v, "4.7") for m, d, v in linhas], columns=["Marcador", "Descrição", "Valor", "Seção"])

#### Módulo `interface` — assistente interativo de entrada e análise de preços

In [ ]:
%%writefile evteas_py/interface.py
"""EVTEAS-Py — módulo interface.

Assistente interativo de entrada de dados (alternativa ao formulário do Colab e à planilha),
com validação a cada resposta, e análise de preços de mercado para apoiar a definição do preço
de venda e da sua distribuição de incerteza.
"""
from __future__ import annotations

from typing import Any, Callable, Dict, Optional

import numpy as np
import pandas as pd

from .config import EVTEASConfig, copiar, fmt_num, obter_por_caminho
from .entradas import (DISTRIBUICOES, ENTRADAS, Entrada, EntradasInvalidas, config_vazia, converter, definir_valor,
                       fmt_auto, ler_distribuicao, ler_item_capex, obter_valor)

MAX_TENTATIVAS = 20


def _vazio(t) -> bool:
    return t is None or not str(t).strip()


def perguntar(entrada: Entrada, ler: Callable[[str], str], atual: Any = None, mostrar: Callable[[str], None] = print):
    """Pergunta até obter um valor válido. Enter mantém o valor atual (se houver); numa entrada
    opcional sem valor atual, Enter adota 'não se aplica'; numa obrigatória, repete a pergunta."""
    sufixo = f" ({entrada.unidade})" if entrada.unidade not in ("—", "") else ""
    if entrada.opcoes:
        sufixo += f" [{' / '.join(entrada.opcoes)}]"
    elif entrada.tipo == "bool":
        sufixo += " [sim / não]"
    if atual is not None and atual != "":
        sufixo += f" (Enter = {fmt_auto(atual)})"
    elif not entrada.obrigatorio:
        sufixo += " (opcional: Enter = não se aplica)"
    texto = f"[{entrada.marcador}] {entrada.rotulo}{sufixo}: "
    for _ in range(MAX_TENTATIVAS):
        resp = ler(texto)
        if _vazio(resp):
            if atual is not None and atual != "":
                return atual
            if not entrada.obrigatorio:
                return entrada.vazio
            mostrar("  Entrada obrigatória: informe um valor.")
            continue
        try:
            return converter(entrada, resp)
        except ValueError as exc:
            mostrar(f"  Valor não aceito: {exc}")
    raise EntradasInvalidas([f"[{entrada.marcador}] {entrada.rotulo}: número máximo de tentativas excedido"])


def wizard_evteas(entrada: Callable[[str], str] = input, base: Optional[EVTEASConfig] = None,
                  silencioso: bool = False, perguntar_metodo: bool = False) -> EVTEASConfig:
    """Assistente de entrada de dados, bloco a bloco (identificação, técnica, econômica, ambiental,
    social, governança, Lean-Green, CAPEX e distribuições de incerteza).

    ``base``: configuração existente (por exemplo, carregada de JSON) cujos valores são mantidos com
    Enter. Sem ``base``, todas as entradas obrigatórias precisam ser digitadas.
    """
    mostrar = (lambda *_: None) if silencioso else print
    cfg = copiar(base) if base is not None else config_vazia()
    grupo = None
    for e in ENTRADAS:
        if e.metodo and not perguntar_metodo:
            continue
        if e.grupo != grupo:
            grupo = e.grupo
            mostrar(f"\n=== {grupo} ===")
        atual = obter_valor(cfg, e)
        if atual is None or (base is None and not e.metodo):
            atual = None if not e.metodo else atual
        valor = perguntar(e, entrada, atual, mostrar)
        definir_valor(cfg, e, valor)

    mostrar("\n=== Investimento (CAPEX) ===")
    mostrar("Informe um item por linha: 'descrição; valor (R$); vida útil (anos)'. Linha vazia encerra.")
    if base is not None and base.economico.capex:
        mostrar(f"  (Enter na primeira linha mantém os {len(base.economico.capex)} itens atuais)")
    itens = []
    for k in range(1, 61):
        resp = entrada(f"[E-C{k:02d}] Item {k}: ")
        if _vazio(resp):
            break
        try:
            itens.append(ler_item_capex(resp))
        except ValueError as exc:
            mostrar(f"  Valor não aceito: {exc}")
    if itens:
        cfg.economico.capex = itens
    elif not cfg.economico.capex:
        raise EntradasInvalidas(["[E-C] Informe ao menos um item de CAPEX"])

    mostrar("\n=== Distribuições de incerteza (Monte Carlo) ===")
    mostrar("Informe 'mín; máx' (a moda é o valor informado acima). Enter = premissa determinística"
            + (" ou mantém a distribuição atual." if base is not None else "."))
    for marcador, var, caminho, rot, unid in DISTRIBUICOES:
        valor = float(obter_por_caminho(cfg, caminho))
        atual = cfg.distribuicoes.get(caminho)
        dica = f" (Enter = {fmt_auto(atual.minimo)}; {fmt_auto(atual.maximo)})" if atual else ""
        for _ in range(MAX_TENTATIVAS):
            resp = entrada(f"[{marcador}] {rot} — valor informado {fmt_auto(valor)} {unid}{dica}: ")
            if _vazio(resp):
                break
            try:
                cfg.distribuicoes[caminho] = ler_distribuicao(resp, valor)
                break
            except ValueError as exc:
                mostrar(f"  Valor não aceito: {exc}")
    from .config import validar_config
    problemas = validar_config(cfg)
    if problemas:
        raise EntradasInvalidas(problemas)
    mostrar("\nEntradas registradas. Execute executar_evteas(cfg) para simular.")
    return cfg


# ---------------------------------------------------------------------------------------------
# Análise de preços de mercado
# ---------------------------------------------------------------------------------------------
def analise_precos(dados: pd.DataFrame, custo_variavel_kg: Optional[float] = None,
                   coluna_preco: str = "preco", coluna_quantidade: str = "quantidade") -> Dict[str, Any]:
    """Resume uma amostra de preços (cotações) por percentis e, se houver quantidades, ajusta uma
    curva de demanda linear q = a − b·p por mínimos quadrados. Com o custo variável unitário c,
    o preço que maximiza a margem de contribuição (p − c)·q(p) é p* = (a/b + c)/2.

    Os percentis P10 e P90 podem orientar o mínimo e o máximo da distribuição do preço (Quadro 13).
    """
    p = pd.to_numeric(dados[coluna_preco], errors="coerce").dropna().to_numpy(dtype=float)
    if len(p) < 2:
        raise ValueError("Informe ao menos duas cotações de preço")
    sugestoes = {"Mínimo": float(p.min()), "P10": float(np.percentile(p, 10)), "P25": float(np.percentile(p, 25)),
                 "Mediana (P50)": float(np.percentile(p, 50)), "Média": float(p.mean()),
                 "P75": float(np.percentile(p, 75)), "P90": float(np.percentile(p, 90)), "Máximo": float(p.max())}
    r: Dict[str, Any] = {"n": int(len(p)), "sugestoes": sugestoes, "preco_otimo": None, "elasticidade_media": None,
                         "demanda": None,
                         "distribuicao_sugerida": f"{fmt_num(sugestoes['P10'], 2)}; {fmt_num(sugestoes['P90'], 2)}"}
    if coluna_quantidade in dados.columns:
        d = dados[[coluna_preco, coluna_quantidade]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(d) >= 3 and d[coluna_preco].nunique() > 1:
            inc, intercepto = np.polyfit(d[coluna_preco], d[coluna_quantidade], 1)
            a, b = float(intercepto), float(-inc)
            r["demanda"] = {"a": a, "b": b,
                            "r2": float(np.corrcoef(d[coluna_preco], d[coluna_quantidade])[0, 1] ** 2)}
            if b > 0:
                pm, qm = float(d[coluna_preco].mean()), float(d[coluna_quantidade].mean())
                r["elasticidade_media"] = -b * pm / qm if qm else None
                if custo_variavel_kg is not None:
                    r["preco_otimo"] = (a / b + float(custo_variavel_kg)) / 2
    return r

#### Módulo `vv` — verificação por invariantes e testes automatizados (dados sintéticos de teste)

In [ ]:
%%writefile evteas_py/vv.py
"""EVTEAS-Py — módulo vv (Verificação e Validação).

Verificação em duas camadas (Boehm, 1984): (1) invariantes conferidos a cada execução e
(2) testes automatizados com casos de solução conhecida, executáveis no próprio notebook.
A validação com especialistas é feita por questionários (Apêndices C e D) após a simulação.
"""
from __future__ import annotations

import math
import traceback
from typing import Any, Callable, Dict, List

import numpy as np
import pandas as pd

from .config import EVTEASConfig, safe_div, taxa_anual_para_mensal


def validar_invariantes(cfg: EVTEASConfig, resultado: Dict[str, Any], bruto: Dict[str, Any] = None) -> Dict[str, Any]:
    from .financeiro import vpl as f_vpl
    checks: List[Dict[str, Any]] = []

    def chk(nome: str, ok, detalhe: str = ""):
        checks.append({"Verificação": nome, "Aprovada": bool(ok), "Detalhe": detalhe})

    t, e, a, s, i = (resultado["tecnico"], resultado["economico"], resultado["ambiental"],
                     resultado["social"], resultado["indice"])
    g = lambda d, k: float(np.atleast_1d(np.asarray(d[k], dtype=float))[0])
    # Técnicas (6)
    chk("OEE em [0, 1]", 0 <= g(t, "oee") <= 1, f"OEE = {g(t, 'oee'):.4f}")
    chk("OEE = Disponibilidade × Performance × Qualidade",
        abs(g(t, "oee") - g(t, "disponibilidade") * g(t, "performance") * g(t, "qualidade")) < 1e-12)
    chk("FCR não negativo", g(t, "fcr") >= 0)
    chk("Ciclos/ano compatíveis com a duração do ciclo",
        cfg.tecnico.ciclos_ano <= 365 / max(cfg.tecnico.ciclo_dias, 1) + 1e-9,
        f"{cfg.tecnico.ciclos_ano} ≤ {365 / cfg.tecnico.ciclo_dias:.2f}")
    chk("Biomassa de despesca ≤ capacidade de suporte",
        g(t, "biomassa_despesca_ciclo_kg") <= g(t, "volume_util_m3") * g(t, "capacidade_suporte_kg_m3") * g(t, "disponibilidade") + 1e-6)
    chk("Ração = FCR × ganho de biomassa", abs(g(t, "racao_ciclo_kg") - g(t, "fcr") * g(t, "ganho_biomassa_ciclo_kg")) < 1e-6)

    # Econômicas (10)
    df = resultado["dre_mensal"]
    close = lambda x, y: bool(np.allclose(x, y, rtol=1e-9, atol=1e-6))
    chk("DRE: Receita Líquida = Receita Bruta − Impostos s/ Faturamento",
        close(df["Receita Líquida"], df["Receita Bruta"] - df["Impostos s/ Faturamento"]))
    chk("DRE: Lucro Bruto = Receita Líquida − Custos Variáveis",
        close(df["Lucro Bruto"], df["Receita Líquida"] - df["Custos Variáveis"]))
    chk("DRE: EBITDA = Lucro Bruto − Custos Fixos", close(df["EBITDA"], df["Lucro Bruto"] - df["Custos Fixos"]))
    chk("DRE: Resultado = LAIR − Impostos s/ Lucro", close(df["Resultado Líquido"], df["LAIR"] - df["Impostos s/ Lucro"]))
    chk("Fluxo operacional = Resultado + Depreciação",
        close(df["Fluxo de Caixa Operacional"], df["Resultado Líquido"] + df["Depreciação"]))
    fluxo = np.asarray(resultado["fluxo_caixa"], dtype=float)
    tma_am = float(taxa_anual_para_mensal(cfg.economico.tma_aa_pct / 100))
    vpl_manual = sum(f / (1 + tma_am) ** k for k, f in enumerate(fluxo))
    chk("VPL = recálculo independente (laço explícito)", abs(vpl_manual - e["vpl"]) < 1e-6 * max(1.0, abs(vpl_manual)),
        f"diferença = {abs(vpl_manual - e['vpl']):.2e}")
    tir_am = e["tir_am"]
    if isinstance(tir_am, float) and math.isfinite(tir_am):
        chk("VPL nulo na TIR", abs(float(f_vpl(tir_am, fluxo)[0])) < 1e-3 * max(1.0, abs(fluxo[0])),
            f"VPL(TIR) = {float(f_vpl(tir_am, fluxo)[0]):.2e}")
    else:
        chk("VPL nulo na TIR (TIR indefinida declarada)", e["diagnostico_tir"] == "sem_solucao", str(e["diagnostico_tir"]))
    pbs, pbd = e["payback_simples_meses"], e["payback_descontado_meses"]
    chk("Payback descontado ≥ payback simples", (not math.isfinite(pbd)) or (math.isfinite(pbs) and pbd >= pbs - 1e-9),
        f"simples = {pbs:.1f}; descontado = {pbd:.1f}")
    chk("Sinal do VPL coerente com TIR × TMA",
        (not math.isfinite(e["tir_aa"])) or ((e["vpl"] >= 0) == (e["tir_aa"] >= cfg.economico.tma_aa_pct / 100 - 1e-9))
        or e["diagnostico_tir"] != "convencional")
    chk("Investimento do beneficiário = investimento − fomento",
        abs(e["investimento_beneficiario"] - (e["investimento"] - e["fomento"])) < 1e-6)

    # Ambientais (6)
    chk("Balanço de N: aportado = retido + lançado + removido",
        g(a, "n_aportado_kg") >= g(a, "n_lancado_kg") - 1e-9 and g(a, "n_retido_kg") >= 0)
    chk("Balanço de P: lançado ≤ aportado − retido",
        g(a, "p_lancado_kg") <= max(g(a, "p_aportado_kg") - g(a, "p_retido_kg"), 0) + 1e-9)
    chk("Pegada hídrica cinza = máximo entre N e P",
        abs(g(a, "ph_cinza_m3_t") - max(g(a, "ph_cinza_n_m3_t"), g(a, "ph_cinza_p_m3_t"))) < 1e-6)
    chk("Água consumida (evaporação) ≤ água captada", g(a, "agua_evaporada_m3") <= g(a, "agua_captada_m3_ano") + 1e-6)
    chk("Emissões = ração + energia + diesel",
        abs(g(a, "co2_total_kg") - (g(a, "co2_racao_kg") + g(a, "co2_energia_kg") + g(a, "co2_diesel_kg"))) < 1e-6)
    status = str(np.atleast_1d(a["status_ecoeficiencia"])[0])
    chk("Ecoeficiência coerente com o status declarado",
        (status == "definida") == math.isfinite(g(a, "ecoeficiencia_rs_kgco2e")), status)

    # Sociais e índice (5)
    chk("LSO em [0, 100]", 0 <= s["lso_0_100"] <= 100, f"LSO = {s['lso_0_100']:.1f}")
    chk("Risco social em [0, 1]", 0 <= s["risco_social"] <= 1)
    pesos = i["pesos"]["dimensoes"]
    chk("Pesos dimensionais somam 1", abs(sum(pesos.values()) - 1) < 1e-9, str({k: round(v, 3) for k, v in pesos.items()}))
    chk("Scores e índice em [0, 1]",
        all(0 <= g(i["scores"], d) <= 1 for d in i["scores"]) and 0 <= g(i, "indice_evteas") <= 1,
        f"índice = {g(i, 'indice_evteas'):.4f}")
    chk("Índice = Σ peso × score", abs(g(i, "indice_evteas") - sum(pesos[d] * g(i["scores"], d) for d in pesos)) < 1e-9)

    # Monte Carlo (1)
    if "monte_carlo" in resultado:
        st = resultado["monte_carlo"]["estatisticas"]
        chk("Monte Carlo: P5 ≤ mediana ≤ P95 e P(VPL>0) em [0, 1]",
            st["p5"] <= st["mediana"] <= st["p95"] and 0 <= st["prob_vpl_positivo"] <= 1)
    tabela = pd.DataFrame(checks)
    return {"tabela": tabela, "total": len(tabela), "aprovadas": int(tabela["Aprovada"].sum()),
            "todas_aprovadas": bool(tabela["Aprovada"].all())}


# ---------------------------------------------------------------------------------------------
# Dados sintéticos de teste (verificação do software)
# ---------------------------------------------------------------------------------------------
# ATENÇÃO: os valores abaixo são DADOS SINTÉTICOS usados somente para testar o código (casos com
# resultado conhecido, monotonicidade, reprodutibilidade). Não são entradas do estudo e não devem
# ser citados na dissertação: as entradas do estudo são informadas pelo usuário antes da simulação.
DADOS_SINTETICOS_TESTE = {
    "projeto": "Dados sintéticos de teste", "especie": "Espécie de teste", "sistema_produtivo": "semi-intensivo",
    "area_lamina_m2": "40.000", "profundidade_media_m": "1,4", "numero_tanques": "32", "tanques_ativos": "30",
    "peso_inicial_g": "30", "peso_final_g": "800", "ciclo_dias": "180", "ciclos_ano": "1,8",
    "produtividade_esperada_kg_m2_ciclo": "1,3", "fcr": "1,5", "mortalidade_pct": "10",
    "tipo_organizacao": "cooperativa", "capital_giro": "60000", "valor_residual_pct": "25",
    "fomento_nao_reembolsavel_pct": "70", "preco_venda_kg": "10,50", "custo_racao_kg": "3,10",
    "custo_alevino_milheiro": "280", "tarifa_energia_kwh": "0,75", "outros_variaveis_kg": "0,30",
    "mao_obra_mes": "12000", "encargos_pct": "20", "cooperados_trabalhadores": "6", "perc_sobras_trabalhadores": "100",
    "manutencao_pct_capex_ano": "2", "administrativo_mes": "2500", "seguros_outros_fixos_mes": "1200",
    "tributos_faturamento_pct": "2,3", "tributos_resultado_pct": "0", "tma_aa_pct": "10", "horizonte_anos": "10",
    "rampa_inicial_pct": "70", "rampa_meses": "12", "ano_regime": "3",
    "renovacao_agua_pct_dia": "3", "enchimentos_ano": "1,8", "evaporacao_mm_dia": "3,75", "infiltracao_mm_dia": "0,5",
    "metodo_tratamento": "bacia de sedimentação", "remocao_n_tratamento_pct": "40", "remocao_p_tratamento_pct": "60",
    "proteina_racao_pct": "32", "fosforo_racao_pct": "1", "nitrogenio_peixe_pct": "2,7", "fosforo_peixe_pct": "0,7",
    "classe_corpo_receptor": "classe 2, lótico", "n_max_mg_l": "2,18", "n_natural_mg_l": "0,3", "p_max_mg_l": "0,1",
    "p_natural_mg_l": "0,02", "fonte_energia": "rede", "fracao_renovavel_pct": "0", "consumo_kwh_kg": "0,6",
    "fator_emissao_rede_kg_kwh": "0,0817", "fator_emissao_racao_kgco2e_kg": "1,1", "diesel_l_ano": "600",
    "fator_emissao_diesel_kg_l": "2,68", "distancia_app_m": "50", "distancia_minima_app_m": "30",
    "amb_licenca_ambiental": "sim", "amb_outorga_agua": "sim", "amb_app_respeitada": "sim",
    "amb_tratamento_efluentes": "sim", "amb_monitoramento_agua": "sim", "amb_car": "sim", "amb_plano_residuos": "não",
    "amb_prevencao_escape": "sim", "amb_energia_renovavel": "não",
    "empregos_diretos": "6", "empregos_indiretos": "10", "mao_obra_local_pct": "100", "compras_locais_pct": "25",
    "salario_minimo": "1518", "aderencia_nr": "sim", "programa_capacitacao": "sim", "participacao_mulheres_pct": "33",
    "relacao_comunidade": "parcial", "canais_formais_comunicacao": "sim", "reunioes_comunidade_ano": "6",
    "conflitos_registrados_ano": "1",
    "gov_estatuto_registrado": "sim", "gov_assembleias_regulares": "sim", "gov_conselho_fiscal": "sim",
    "gov_prestacao_contas_publica": "não", "gov_plano_negocios": "sim", "gov_registro_premissas_auditavel": "sim",
    "gov_canal_denuncia": "não",
    "fcr_referencia": "1,4", "mortalidade_referencia_pct": "5", "consumo_kwh_kg_referencia": "0,4",
    "capex_01": "Viveiros; 380.000; 20", "capex_02": "Abastecimento e drenagem; 95.000; 15",
    "capex_03": "Aeradores e quadro elétrico; 85.000; 8", "capex_04": "Galpão e depósito; 90.000; 20",
    "capex_05": "Equipamentos de manejo; 35.000; 5", "capex_06": "Veículo utilitário; 40.000; 10",
    "capex_07": "Licenciamento e projeto; 20.000; 10",
    "dist_preco_venda_kg": "9; 11,5", "dist_custo_racao_kg": "2,8; 3,7", "dist_custo_alevino_milheiro": "230; 350",
    "dist_tarifa_energia_kwh": "0,65; 0,95", "dist_custos_fixos_fator": "0,95; 1,15", "dist_capex_fator": "0,7; 1,5",
    "dist_fcr": "1,3; 1,85", "dist_mortalidade_pct": "5; 20", "dist_desempenho_crescimento_pct": "85; 105",
    "alt_B_nome": "Alternativa sintética B", "alt_B_alteracoes": "fonte_energia = solar; fracao_renovavel_pct = 90; "
    "remocao_n_tratamento_pct = 70; remocao_p_tratamento_pct = 80; amb_energia_renovavel = sim",
    "alt_B_capex_adicional": "Usina solar; 120.000; 25 | Wetland construído; 60.000; 20",
    "alt_C_nome": "Alternativa sintética C", "alt_C_alteracoes": "produtividade_esperada_kg_m2_ciclo = 1,8; "
    "capacidade_suporte_kg_m3 = 1,6; consumo_kwh_kg = 0,85; empregos_diretos = 8",
    "alt_C_capex_adicional": "Aeradores adicionais; 90.000; 8",
}


def config_sintetica_teste():
    """Configuração com os dados sintéticos de teste (somente para a verificação do software)."""
    from .entradas import config_de_formulario
    return config_de_formulario(DADOS_SINTETICOS_TESTE)


# ---------------------------------------------------------------------------------------------
# Testes automatizados (segunda camada da verificação)
# ---------------------------------------------------------------------------------------------
def testes_automatizados(rapido: bool = True) -> pd.DataFrame:
    """Executa a suíte de testes e devolve uma tabela (nome, aprovado, detalhe).

    Pode ser rodada no Colab sem pytest. Com ``rapido=True`` usa menos iterações de Monte Carlo.
    """
    from . import entradas, financeiro, incerteza, interface, modelo, pipeline, ponderacao, relatorios
    from .config import Distribuicao, carregar_config, copiar, salvar_config
    resultados = []

    def teste(nome: str):
        def deco(fn: Callable[[], Any]):
            try:
                det = fn()
                resultados.append({"Teste": nome, "Aprovado": True, "Detalhe": det or ""})
            except Exception as exc:  # noqa: BLE001
                resultados.append({"Teste": nome, "Aprovado": False,
                                   "Detalhe": f"{type(exc).__name__}: {exc} | {traceback.format_exc(limit=2)[-300:]}"})
            return fn
        return deco

    N = 2000 if rapido else 10000

    @teste("VPL com solução analítica (perpetuidade truncada)")
    def _():
        f = [-1000] + [100] * 10
        esperado = -1000 + 100 * (1 - 1.05 ** -10) / 0.05
        assert abs(financeiro.vpl(0.05, f)[0] - esperado) < 1e-9

    @teste("VPL e TIR conferidos com numpy-financial (se instalado)")
    def _():
        try:
            import numpy_financial as npf
        except ImportError:
            return "numpy-financial não instalado: conferido contra solução analítica"
        f = np.array([-805000] + [9000] * 119 + [400000])
        assert abs(financeiro.vpl(0.008, f)[0] - npf.npv(0.008, f)) < 1e-4
        r, d = financeiro.tir(f)
        assert abs(r[0] - npf.irr(f)) < 1e-7 and d[0] == "convencional"

    @teste("TIR de fluxo simples (−100, +110) = 10%")
    def _():
        r, d = financeiro.tir([-100, 110])
        assert abs(r[0] - 0.10) < 1e-10 and d[0] == "convencional"

    @teste("Fluxo sem TIR declara 'sem_solucao' (não força o limite numérico)")
    def _():
        r, d = financeiro.tir([-100, -50, -10])
        assert np.isnan(r[0]) and d[0] == "sem_solucao"

    @teste("Fluxo com duas TIR é diagnosticado como 'multiplas'")
    def _():
        r, d = financeiro.tir([-1600, 10000, -10000])     # raízes em 25% e 400%
        r2, d2 = financeiro.tir([-1600, 10000, -10000], hi=5.0, pontos=400)
        assert d2[0] == "multiplas"

    @teste("Payback simples e descontado")
    def _():
        assert abs(financeiro.payback([-100, 50, 50, 50])[0] - 2.0) < 1e-12
        assert abs(financeiro.payback([-100, 40, 40, 40])[0] - 2.5) < 1e-12
        assert np.isnan(financeiro.payback([-100, 10, 10])[0])
        assert financeiro.payback([-100, 60, 60], 0.10)[0] > financeiro.payback([-100, 60, 60])[0]

    @teste("Pegada hídrica cinza confere com cálculo manual")
    def _():
        cfg = config_sintetica_teste()
        r = modelo.motor(cfg)
        t, a = r["tecnico"], r["ambiental"]
        c = cfg.ambiental
        p_in = t["racao_ano_kg"][0] * c.fosforo_racao_pct / 100
        p_ret = t["ganho_biomassa_ano_kg"][0] * c.fosforo_peixe_pct / 100
        p_lanc = (p_in - p_ret) * (1 - c.remocao_p_tratamento_pct / 100)
        manual_p = p_lanc * 1000 / (c.p_max_mg_l - c.p_natural_mg_l) / (t["producao_ano_kg"][0] / 1000)
        assert abs(a["ph_cinza_p_m3_t"][0] - manual_p) < 1e-6

    @teste("Monte Carlo degenerado na moda reproduz o determinístico")
    def _():
        cfg = config_sintetica_teste()
        cfg.distribuicoes = {k: Distribuicao("triangular", d.mais_provavel, d.mais_provavel, d.mais_provavel)
                             for k, d in cfg.distribuicoes.items()}
        mc = incerteza.monte_carlo(cfg, n=50, seed=1)
        det = modelo.motor(cfg)["economico"]["vpl"][0]
        assert np.allclose(mc["vpl"], det, rtol=0, atol=1e-6)

    @teste("Reprodutibilidade por semente")
    def _():
        cfg = config_sintetica_teste()
        a = incerteza.monte_carlo(cfg, n=500, seed=7)["vpl"]
        b = incerteza.monte_carlo(cfg, n=500, seed=7)["vpl"]
        c = incerteza.monte_carlo(cfg, n=500, seed=8)["vpl"]
        assert np.array_equal(a, b) and not np.array_equal(a, c)

    @teste("Vetos: VPL negativo impede 'VIÁVEL'")
    def _():
        cfg = config_sintetica_teste()
        assert ponderacao.classificar(cfg, -1.0, 0.95, 80, [])["classificacao"] == "NÃO VIÁVEL"

    @teste("Vetos: item ambiental eliminatório e LSO baixa")
    def _():
        cfg = config_sintetica_teste()
        assert ponderacao.classificar(cfg, 1e6, 0.95, 80, ["Licença ambiental"])["classificacao"] == "NÃO VIÁVEL"
        assert ponderacao.classificar(cfg, 1e6, 0.95, 20, [])["classificacao"] == "NÃO VIÁVEL"
        assert ponderacao.classificar(cfg, 1e6, 0.95, 80, [], 0.9)["classificacao"] == "VIÁVEL"
        assert ponderacao.classificar(cfg, 1e6, 0.55, 80, [], 0.9)["classificacao"] == "VIÁVEL COM RESSALVAS"

    @teste("Licença ambiental ausente gera veto no pipeline")
    def _():
        cfg = config_sintetica_teste()
        cfg.ambiental.checklist["licenca_ambiental"] = False
        r = pipeline.executar_evteas(cfg, executar_mc=False, executar_sens=False)
        assert r["decisao"]["classificacao"] == "NÃO VIÁVEL"

    @teste("Normalização: benefício, custo e saturação")
    def _():
        assert ponderacao.normalizar(5, "benefício", 0, 10) == 0.5
        assert ponderacao.normalizar(2.5, "custo", 1, 2.5) == 0.0
        assert ponderacao.normalizar(99, "benefício", 0, 10) == 1.0

    @teste("Pesos Likert somam 1 e concordância é calculada")
    def _():
        k = list(ponderacao.KPIS)
        p = ponderacao.pesos_likert({"A": {x: 4 for x in k}, "B": {x: 2 for x in k}})
        assert abs(sum(p["dimensoes"].values()) - 1) < 1e-12 and abs(p["concordancia_media"] - 0.5) < 1e-12

    @teste("AHP: matriz consistente tem RC ≈ 0 e pesos corretos")
    def _():
        w = np.array([0.4, 0.3, 0.2, 0.1])
        A = w[:, None] / w[None, :]
        p = ponderacao.pesos_ahp(A)
        assert np.allclose(list(p["dimensoes"].values()), w) and p["razao_consistencia"] < 1e-9

    @teste("TOPSIS ordena alternativa dominante em 1º")
    def _():
        c = ponderacao.topsis([[10, 1], [5, 5], [1, 10]], [1, 1], ["benefício", "custo"])
        assert c[0] == 1.0 and c[2] == 0.0

    @teste("Invariantes aprovados com os dados sintéticos de teste")
    def _():
        r = pipeline.executar_evteas(config_sintetica_teste(), executar_mc=True, executar_sens=False, n_mc=500)
        assert r["vv"]["todas_aprovadas"], r["vv"]["tabela"][~r["vv"]["tabela"]["Aprovada"]].to_string()
        return f"{r['vv']['aprovadas']}/{r['vv']['total']} invariantes"

    @teste("Monotonicidade: preço maior aumenta o VPL; FCR maior reduz")
    def _():
        cfg = config_sintetica_teste()
        b = modelo.motor(cfg)["economico"]["vpl"][0]
        assert modelo.motor(cfg, {"economico.preco_venda_kg": np.array([11.0])})["economico"]["vpl"][0] > b
        assert modelo.motor(cfg, {"tecnico.fcr": np.array([1.7])})["economico"]["vpl"][0] < b

    @teste("Mortalidade maior reduz a biomassa despescada (estocagem fixa)")
    def _():
        cfg = config_sintetica_teste()
        a = modelo.motor(cfg)["tecnico"]["biomassa_despesca_ciclo_kg"][0]
        b = modelo.motor(cfg, {"tecnico.mortalidade_pct": np.array([20.0])})["tecnico"]["biomassa_despesca_ciclo_kg"][0]
        assert b < a

    @teste("Valor crítico do preço zera o VPL")
    def _():
        cfg = config_sintetica_teste()
        vc = incerteza.valores_criticos(cfg, ["economico.preco_venda_kg"])
        p = vc["Valor crítico (VPL = 0)"][0]
        assert abs(modelo.motor(cfg, {"economico.preco_venda_kg": np.array([p])})["economico"]["vpl"][0]) < 1.0

    @teste("Fomento aumenta o VPL do beneficiário exatamente no valor aportado")
    def _():
        e = modelo.motor(config_sintetica_teste())["economico"]
        assert abs((e["vpl_beneficiario"][0] - e["vpl"][0]) - e["fomento"][0]) < 1e-6

    @teste("Salvar e carregar configuração em JSON preserva o resultado")
    def _():
        import os
        import tempfile
        cfg = config_sintetica_teste()
        caminho = os.path.join(tempfile.mkdtemp(), "cfg.json")
        salvar_config(cfg, caminho)
        cfg2 = carregar_config(caminho)
        assert abs(modelo.motor(cfg)["economico"]["vpl"][0] - modelo.motor(cfg2)["economico"]["vpl"][0]) < 1e-9

    @teste("Entradas em branco impedem a execução (nenhum valor predefinido)")
    def _():
        cfg = entradas.config_vazia()
        faltam = entradas.entradas_faltantes(cfg)
        assert len(faltam) > 80, len(faltam)
        try:
            pipeline.executar_evteas(cfg, executar_mc=False, executar_sens=False)
        except entradas.EntradasInvalidas as exc:
            return f"{len(exc.erros)} entradas obrigatórias apontadas"
        raise AssertionError("a execução deveria ser bloqueada")

    @teste("Formulário: vírgula decimal, milhar com ponto, sim/não e opção sem acento")
    def _():
        assert entradas.ler_numero("1.234,56") == 1234.56 and entradas.ler_numero("40.000") == 40000
        assert entradas.ler_numero("1,5") == 1.5 and entradas.ler_numero("0.0817") == 0.0817
        assert entradas.ler_bool("Não") is False and entradas.ler_bool("sim") is True
        assert entradas.converter(entradas.POR_VAR["tipo_organizacao"], "associacao") == "associação"
        dados = dict(DADOS_SINTETICOS_TESTE, profundidade_media_m="abc", mortalidade_pct="120")
        cfg, erros = entradas.ler_formulario(dados)
        assert any("E-T04" in e for e in erros) and any("E-T15" in e for e in erros), erros

    @teste("Moda da distribuição deve coincidir com o valor determinístico")
    def _():
        dados = dict(DADOS_SINTETICOS_TESTE, dist_preco_venda_kg="9; 10; 11,5")
        cfg, erros = entradas.ler_formulario(dados)
        assert any("moda" in e for e in erros), erros

    @teste("Planilha de entradas: gerar, preencher e ler reproduz o formulário")
    def _():
        import os
        import tempfile
        cfg = config_sintetica_teste()
        caminho = os.path.join(tempfile.mkdtemp(), "entradas.xlsx")
        entradas.gerar_planilha_entradas(caminho, cfg)
        cfg2 = entradas.config_de_planilha(caminho)
        assert abs(modelo.motor(cfg)["economico"]["vpl"][0] - modelo.motor(cfg2)["economico"]["vpl"][0]) < 1e-6
        vazia = os.path.join(tempfile.mkdtemp(), "vazia.xlsx")
        entradas.gerar_planilha_entradas(vazia)
        try:
            entradas.config_de_planilha(vazia)
        except entradas.EntradasInvalidas:
            return "planilha em branco recusada; planilha preenchida aceita"
        raise AssertionError("planilha em branco deveria ser recusada")

    @teste("Assistente interativo com respostas simuladas")
    def _():
        dados = DADOS_SINTETICOS_TESTE
        respostas = []
        for e in entradas.ENTRADAS:
            if e.metodo:
                continue
            if e.var == "profundidade_media_m":
                respostas += ["abc", "-3"]          # inválidas: devem ser recusadas
            respostas.append(str(dados.get(e.var, "")))
        respostas += [dados[f"capex_{i:02d}"] for i in range(1, 8)] + [""]
        respostas += [dados.get(d[1], "") for d in entradas.DISTRIBUICOES]
        it = iter(respostas)
        cfg = interface.wizard_evteas(entrada=lambda _: next(it), silencioso=True)
        ref = config_sintetica_teste()
        assert abs(modelo.motor(cfg)["economico"]["vpl"][0] - modelo.motor(ref)["economico"]["vpl"][0]) < 1e-6

    @teste("Assistente com configuração-base: Enter mantém os valores")
    def _():
        it = iter([""] * 1000)
        base = config_sintetica_teste()
        cfg = interface.wizard_evteas(entrada=lambda _: next(it), base=base, silencioso=True)
        assert abs(modelo.motor(cfg)["economico"]["vpl"][0] - modelo.motor(base)["economico"]["vpl"][0]) < 1e-9

    @teste("Análise de preços por percentis e curva de demanda")
    def _():
        df = pd.DataFrame({"preco": [9, 10, 11, 12, 13], "quantidade": [1000, 900, 800, 700, 600]})
        r = interface.analise_precos(df, custo_variavel_kg=6.0)
        assert r["sugestoes"]["Mediana (P50)"] == 11 and r["preco_otimo"] is not None

    @teste("Comparação de alternativas por TOPSIS")
    def _():
        alts = entradas.alternativas_de_formulario(config_sintetica_teste(), DADOS_SINTETICOS_TESTE)
        df = pipeline.comparar_alternativas(alts, n_mc=300)
        assert len(df) == 3 and df["TOPSIS"].between(0, 1).all()

    @teste("Exportação (Excel, JSON, Markdown e marcadores)")
    def _():
        import os
        import tempfile
        cfg = config_sintetica_teste()
        r = pipeline.executar_evteas(cfg, n_mc=300)
        d = tempfile.mkdtemp()
        arqs = relatorios.exportar_tudo(r, cfg, d, figuras=False, alternativas=None)
        mk = relatorios.marcadores(r, cfg)
        assert {"E-T03", "E-T03F", "R-E04", "R-M06", "R-I08"} <= set(mk["Marcador"])
        assert all(os.path.exists(p) for p in arqs.values()), arqs
        return f"{len(arqs)} arquivos"

    return pd.DataFrame(resultados)

#### Módulo `__init__` — interface pública do pacote

In [ ]:
%%writefile evteas_py/__init__.py
"""EVTEAS-Py — framework computacional para Estudos de Viabilidade Técnica, Econômica, Ambiental e
Social (EVTEAS) de projetos de piscicultura.

Fluxo de uso:
    1. informar as entradas do projeto (formulário do Colab, planilha ou ``wizard_evteas``);
    2. ``executar_evteas(cfg)`` — análise determinística, Monte Carlo, sensibilidade e decisão;
    3. ``marcadores(...)`` / ``exportar_tudo(...)`` — valores [E-xx] e [R-xx] para o texto;
    4. ``preencher_docx(...)`` — substitui os marcadores no .docx da dissertação (opcional).
Nenhuma entrada do projeto tem valor predefinido.
"""
from .config import VERSAO, EVTEASConfig, carregar_config, salvar_config, validar_config
from .entradas import (ENTRADAS, EntradasInvalidas, alternativas_de_formulario, coletar_formulario,
                       config_de_formulario, config_de_planilha, config_vazia, entradas_faltantes,
                       gerar_planilha_entradas, ler_formulario, tabela_entradas)
from .interface import analise_precos, wizard_evteas
from .pipeline import comparar_alternativas, executar_evteas
from .relatorios import (exportar_tudo, gerar_figuras, marcadores, preencher_docx, resumo_executivo,
                         tabelas_dissertacao)
from .vv import testes_automatizados, validar_invariantes

__version__ = VERSAO
__all__ = [
    "VERSAO", "EVTEASConfig", "carregar_config", "salvar_config", "validar_config", "ENTRADAS", "EntradasInvalidas",
    "alternativas_de_formulario", "coletar_formulario", "config_de_formulario", "config_de_planilha", "config_vazia",
    "entradas_faltantes", "gerar_planilha_entradas", "ler_formulario", "tabela_entradas", "analise_precos",
    "wizard_evteas", "comparar_alternativas", "executar_evteas", "exportar_tudo", "gerar_figuras", "marcadores",
    "preencher_docx", "resumo_executivo", "tabelas_dissertacao", "testes_automatizados", "validar_invariantes",
]

In [ ]:
#@title 1.4 Carregar o pacote { display-mode: "form" }
import sys
sys.path.insert(0, ".")
for _m in [m for m in sys.modules if m.startswith("evteas_py")]:
    del sys.modules[_m]
import pandas as pd
from IPython.display import HTML, Markdown, display
import evteas_py as ev
from evteas_py import entradas as ent, relatorios as rel, vv

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 500)
try:
    from google.colab import files as colab_files
except ImportError:          # execução fora do Colab (Jupyter local)
    colab_files = None


class Pendencia(Exception):
    """Interrompe a execução com uma mensagem curta (sem rastreamento de pilha)."""
    def _render_traceback_(self):
        return [f"⚠ {self}"]


def baixar(caminho):
    if colab_files is not None:
        colab_files.download(caminho)
    else:
        print("Arquivo gerado em:", os.path.abspath(caminho))


def enviar(extensao, caminho_informado=""):
    if caminho_informado:
        return caminho_informado
    if colab_files is None:
        raise Pendencia(f"Fora do Colab, informe o caminho do arquivo {extensao} no campo correspondente.")
    enviados = colab_files.upload()
    nomes = [n for n in enviados if n.lower().endswith(extensao)]
    if not nomes:
        raise Pendencia(f"Nenhum arquivo {extensao} foi enviado.")
    return nomes[0]


print(f"EVTEAS-Py {ev.VERSAO} carregado. Prossiga para a Seção 2.")

## 2 Entradas do projeto

Todas as entradas começam **em branco**. Os marcadores entre colchetes (por exemplo, **[E-T03]**) são os
mesmos do Quadro de premissas da dissertação (Seção 4.2), preenchido com os valores informados aqui.
Os parâmetros do método (célula 2.10) vêm com os valores documentados no Capítulo 3 e podem ser alterados.

In [ ]:
#@title 2.0 Modo de entrada { display-mode: "form" }
#@markdown **formulário**: preencha as células 2.1 a 2.10. **planilha (.xlsx)**: gere a planilha na célula 2.11,
#@markdown preencha-a (com a origem de cada premissa) e envie-a na Seção 3. **arquivo de premissas (.json)**: reutiliza
#@markdown as entradas de uma execução anterior (EVTEAS_premissas.json). **assistente interativo**: perguntas uma a uma.
MODO_ENTRADA = "formulário"  #@param ["formulário", "planilha (.xlsx)", "arquivo de premissas (.json)", "assistente interativo"]
#@markdown Caminho do arquivo (opcional; em branco, o Colab pede o envio do arquivo):
ARQUIVO_ENTRADAS = ""  #@param {type:"string"}

In [ ]:
#@title 2.1 Identificação e dimensão técnica { display-mode: "form" }
#@markdown Campos com \* são obrigatórios. Use vírgula decimal (1,5); o ponto de milhar é opcional (40.000).
#@markdown Campos opcionais em branco = "não se aplica". Após preencher, execute a célula (▶).
#@markdown ---
#@markdown **[E-I01]** Nome do projeto \*
projeto = ""  #@param {type:"string"}
#@markdown **[E-I02]** Localização (município/UF)
localizacao = ""  #@param {type:"string"}
#@markdown **[E-I03]** Responsável pelo estudo
responsavel = ""  #@param {type:"string"}
#@markdown **[E-T01]** Espécie cultivada \*
especie = ""  #@param {type:"string"}
#@markdown **[E-T02]** Sistema produtivo \*
sistema_produtivo = ""  #@param ["", "extensivo", "semi-intensivo", "intensivo", "superintensivo"]
#@markdown **[E-T03]** Área de lâmina d'água — m² \*
area_lamina_m2 = ""  #@param {type:"string"}
#@markdown **[E-T04]** Profundidade média — m \*
profundidade_media_m = ""  #@param {type:"string"}
#@markdown **[E-T05]** Número de viveiros/tanques — un. \*
numero_tanques = ""  #@param {type:"string"}
#@markdown **[E-T06]** Viveiros/tanques em operação simultânea — un. \*
tanques_ativos = ""  #@param {type:"string"}
#@markdown **[E-T07]** Capacidade de suporte — kg/m³ (Em branco: valor de referência do sistema produtivo (Kubitza, 2017))
capacidade_suporte_kg_m3 = ""  #@param {type:"string"}
#@markdown **[E-T08]** Peso inicial dos juvenis — g \*
peso_inicial_g = ""  #@param {type:"string"}
#@markdown **[E-T09]** Peso de abate — g \*
peso_final_g = ""  #@param {type:"string"}
#@markdown **[E-T10]** Duração do ciclo de engorda — dias \*
ciclo_dias = ""  #@param {type:"string"}
#@markdown **[E-T11]** Ciclos por ano — ciclos/ano \*
ciclos_ano = ""  #@param {type:"string"}
#@markdown **[E-T12]** Produtividade esperada por área — kg/m²/ciclo (Em branco: sem restrição por área (vale só a capacidade de suporte))
produtividade_esperada_kg_m2_ciclo = ""  #@param {type:"string"}
#@markdown **[E-T13]** Juvenis estocados por ciclo — un./ciclo (Em branco: estocagem dimensionada pelo modelo)
alevinos_por_ciclo = ""  #@param {type:"string"}
#@markdown **[E-T14]** Conversão alimentar aparente (FCR) — kg ração/kg \*
fcr = ""  #@param {type:"string"}
#@markdown **[E-T15]** Mortalidade no ciclo — % \*
mortalidade_pct = ""  #@param {type:"string"}

In [ ]:
#@title 2.2 Dimensão econômica { display-mode: "form" }
#@markdown Campos com \* são obrigatórios. Use vírgula decimal (1,5); o ponto de milhar é opcional (40.000).
#@markdown Campos opcionais em branco = "não se aplica". Após preencher, execute a célula (▶).
#@markdown Mix de produtos (opcional): `nome; participação; fator de preço | ...` (ex.: `Inteira; 0,7; 1 | Filé; 0,3; 2,4`). Em branco = produto único.
#@markdown ---
#@markdown **[E-E01]** Tipo de organização \*
tipo_organizacao = ""  #@param ["", "cooperativa", "associação", "empresa"]
#@markdown **[E-E02]** Capital de giro — R$ \*
capital_giro = ""  #@param {type:"string"}
#@markdown **[E-E03]** Valor residual ao fim do horizonte — % do CAPEX \*
valor_residual_pct = ""  #@param {type:"string"}
#@markdown **[E-E04]** Fomento não reembolsável — % do CAPEX \*
fomento_nao_reembolsavel_pct = ""  #@param {type:"string"}
#@markdown **[E-E05]** Preço de venda — R$/kg \*
preco_venda_kg = ""  #@param {type:"string"}
#@markdown **[E-E06]** Outras receitas (serviços) — R$/mês
receita_servicos_mes = ""  #@param {type:"string"}
#@markdown **[E-E07]** Custo da ração — R$/kg \*
custo_racao_kg = ""  #@param {type:"string"}
#@markdown **[E-E08]** Custo do milheiro de juvenis — R$/mil \*
custo_alevino_milheiro = ""  #@param {type:"string"}
#@markdown **[E-E09]** Tarifa de energia elétrica — R$/kWh \*
tarifa_energia_kwh = ""  #@param {type:"string"}
#@markdown **[E-E10]** Outros custos variáveis (embalagem, gelo, frete, comercialização) — R$/kg \*
outros_variaveis_kg = ""  #@param {type:"string"}
#@markdown **[E-E11]** Retiradas/salários — R$/mês \*
mao_obra_mes = ""  #@param {type:"string"}
#@markdown **[E-E12]** Encargos sobre retiradas/salários — % \*
encargos_pct = ""  #@param {type:"string"}
#@markdown **[E-E13]** Cooperados/trabalhadores — pessoas \*
cooperados_trabalhadores = ""  #@param {type:"string"}
#@markdown **[E-E14]** Sobras distribuídas aos trabalhadores — % \*
perc_sobras_trabalhadores = ""  #@param {type:"string"}
#@markdown **[E-E15]** Manutenção — % do CAPEX/ano \*
manutencao_pct_capex_ano = ""  #@param {type:"string"}
#@markdown **[E-E16]** Despesas administrativas — R$/mês \*
administrativo_mes = ""  #@param {type:"string"}
#@markdown **[E-E17]** Seguros e outros custos fixos — R$/mês \*
seguros_outros_fixos_mes = ""  #@param {type:"string"}
#@markdown **[E-E18]** Tributos sobre o faturamento — % \*
tributos_faturamento_pct = ""  #@param {type:"string"}
#@markdown **[E-E19]** Tributos sobre o resultado — % \*
tributos_resultado_pct = ""  #@param {type:"string"}
#@markdown **[E-E20]** Taxa mínima de atratividade (TMA) — % a.a. \*
tma_aa_pct = ""  #@param {type:"string"}
#@markdown **[E-E21]** Horizonte de análise — anos \*
horizonte_anos = ""  #@param {type:"string"}
#@markdown **[E-E22]** Produção inicial da rampa — % do regime \*
rampa_inicial_pct = ""  #@param {type:"string"}
#@markdown **[E-E23]** Duração da rampa de produção — meses \*
rampa_meses = ""  #@param {type:"string"}
#@markdown **[E-E24]** Ano de regime (indicadores anuais) — ano \*
ano_regime = ""  #@param {type:"string"}
#@markdown **[E-E25]** Mix de produtos (opcional)
mix_produtos = ""  #@param {type:"string"}

In [ ]:
#@title 2.3 Investimento (CAPEX) { display-mode: "form" }
#@markdown Um item por campo, no formato `descrição; valor (R$); vida útil (anos)`, por exemplo
#@markdown `Escavação dos viveiros; 380.000; 20`. Informe ao menos um item. Campos vazios são ignorados.
#@markdown ---
capex_01 = ""  #@param {type:"string"}
capex_02 = ""  #@param {type:"string"}
capex_03 = ""  #@param {type:"string"}
capex_04 = ""  #@param {type:"string"}
capex_05 = ""  #@param {type:"string"}
capex_06 = ""  #@param {type:"string"}
capex_07 = ""  #@param {type:"string"}
capex_08 = ""  #@param {type:"string"}
capex_09 = ""  #@param {type:"string"}
capex_10 = ""  #@param {type:"string"}
capex_11 = ""  #@param {type:"string"}
capex_12 = ""  #@param {type:"string"}

In [ ]:
#@title 2.4 Dimensão ambiental { display-mode: "form" }
#@markdown Campos com \* são obrigatórios. Use vírgula decimal (1,5); o ponto de milhar é opcional (40.000).
#@markdown Campos opcionais em branco = "não se aplica". Após preencher, execute a célula (▶).
#@markdown Limites da Resolução CONAMA nº 357/2005 para águas doces classe 2 (art. 15): P total 0,030 mg/L (lêntico), 0,050 mg/L (intermediário) e 0,1 mg/L (lótico); N total 1,27 mg/L (lêntico) e 2,18 mg/L (lótico), quando o N for limitante. Confira a classe do corpo receptor no enquadramento vigente.
#@markdown ---
#@markdown **[E-A01]** Renovação de água — % do volume/dia \*
renovacao_agua_pct_dia = ""  #@param {type:"string"}
#@markdown **[E-A02]** Enchimentos completos dos viveiros — por ano \*
enchimentos_ano = ""  #@param {type:"string"}
#@markdown **[E-A03]** Evaporação — mm/dia \*
evaporacao_mm_dia = ""  #@param {type:"string"}
#@markdown **[E-A04]** Infiltração — mm/dia \*
infiltracao_mm_dia = ""  #@param {type:"string"}
#@markdown **[E-A05]** Método de tratamento de efluentes \*
metodo_tratamento = ""  #@param {type:"string"}
#@markdown **[E-A06]** Remoção de nitrogênio no tratamento — % \*
remocao_n_tratamento_pct = ""  #@param {type:"string"}
#@markdown **[E-A07]** Remoção de fósforo no tratamento — % \*
remocao_p_tratamento_pct = ""  #@param {type:"string"}
#@markdown **[E-A08]** Proteína bruta da ração — % \*
proteina_racao_pct = ""  #@param {type:"string"}
#@markdown **[E-A09]** Fósforo da ração — % \*
fosforo_racao_pct = ""  #@param {type:"string"}
#@markdown **[E-A10]** Nitrogênio retido na biomassa — % do peso \*
nitrogenio_peixe_pct = ""  #@param {type:"string"}
#@markdown **[E-A11]** Fósforo retido na biomassa — % do peso \*
fosforo_peixe_pct = ""  #@param {type:"string"}
#@markdown **[E-A12]** Classe e ambiente do corpo receptor \* (Ex.: classe 2, ambiente lótico (Resolução CONAMA nº 357/2005))
classe_corpo_receptor = ""  #@param {type:"string"}
#@markdown **[E-A13]** Concentração máxima de N total (padrão da classe) — mg/L \*
n_max_mg_l = ""  #@param {type:"string"}
#@markdown **[E-A14]** Concentração natural de N total — mg/L \*
n_natural_mg_l = ""  #@param {type:"string"}
#@markdown **[E-A15]** Concentração máxima de P total (padrão da classe) — mg/L \*
p_max_mg_l = ""  #@param {type:"string"}
#@markdown **[E-A16]** Concentração natural de P total — mg/L \*
p_natural_mg_l = ""  #@param {type:"string"}
#@markdown **[E-A17]** Fonte principal de energia \*
fonte_energia = ""  #@param ["", "rede", "solar", "biomassa"]
#@markdown **[E-A18]** Energia renovável de fonte própria — % \*
fracao_renovavel_pct = ""  #@param {type:"string"}
#@markdown **[E-A19]** Consumo de energia (aeração e bombeamento) — kWh/kg \*
consumo_kwh_kg = ""  #@param {type:"string"}
#@markdown **[E-A20]** Fator de emissão da rede elétrica — kgCO2e/kWh \*
fator_emissao_rede_kg_kwh = ""  #@param {type:"string"}
#@markdown **[E-A21]** Fator de emissão da ração — kgCO2e/kg \*
fator_emissao_racao_kgco2e_kg = ""  #@param {type:"string"}
#@markdown **[E-A22]** Consumo de diesel — L/ano \*
diesel_l_ano = ""  #@param {type:"string"}
#@markdown **[E-A23]** Fator de emissão do diesel — kgCO2e/L \*
fator_emissao_diesel_kg_l = ""  #@param {type:"string"}
#@markdown **[E-A24]** Distância dos viveiros ao curso d'água — m \*
distancia_app_m = ""  #@param {type:"string"}
#@markdown **[E-A25]** Largura mínima da APP aplicável — m \*
distancia_minima_app_m = ""  #@param {type:"string"}
#@markdown **[E-A26]** Licença ambiental da aquicultura (eliminatório) — sim/não \*
amb_licenca_ambiental = ""  #@param ["", "sim", "não"]
#@markdown **[E-A27]** Outorga de direito de uso da água (eliminatório) — sim/não \*
amb_outorga_agua = ""  #@param ["", "sim", "não"]
#@markdown **[E-A28]** Área de Preservação Permanente respeitada (eliminatório) — sim/não \*
amb_app_respeitada = ""  #@param ["", "sim", "não"]
#@markdown **[E-A29]** Tratamento de efluentes — sim/não \*
amb_tratamento_efluentes = ""  #@param ["", "sim", "não"]
#@markdown **[E-A30]** Monitoramento da qualidade da água — sim/não \*
amb_monitoramento_agua = ""  #@param ["", "sim", "não"]
#@markdown **[E-A31]** Cadastro Ambiental Rural (CAR) — sim/não \*
amb_car = ""  #@param ["", "sim", "não"]
#@markdown **[E-A32]** Plano de gestão de resíduos — sim/não \*
amb_plano_residuos = ""  #@param ["", "sim", "não"]
#@markdown **[E-A33]** Prevenção de escape de peixes — sim/não \*
amb_prevencao_escape = ""  #@param ["", "sim", "não"]
#@markdown **[E-A34]** Uso de energia renovável — sim/não \*
amb_energia_renovavel = ""  #@param ["", "sim", "não"]

In [ ]:
#@title 2.5 Dimensão social e governança { display-mode: "form" }
#@markdown Campos com \* são obrigatórios. Use vírgula decimal (1,5); o ponto de milhar é opcional (40.000).
#@markdown Campos opcionais em branco = "não se aplica". Após preencher, execute a célula (▶).
#@markdown ---
#@markdown **[E-S01]** Empregos diretos — postos \*
empregos_diretos = ""  #@param {type:"string"}
#@markdown **[E-S02]** Empregos indiretos — postos \*
empregos_indiretos = ""  #@param {type:"string"}
#@markdown **[E-S03]** Mão de obra local — % \*
mao_obra_local_pct = ""  #@param {type:"string"}
#@markdown **[E-S04]** Compras de insumos no município/região — % \*
compras_locais_pct = ""  #@param {type:"string"}
#@markdown **[E-S05]** Salário mínimo vigente — R$/mês \*
salario_minimo = ""  #@param {type:"string"}
#@markdown **[E-S06]** Aderência às Normas Regulamentadoras — sim/não \*
aderencia_nr = ""  #@param ["", "sim", "não"]
#@markdown **[E-S07]** Programa de capacitação — sim/não \*
programa_capacitacao = ""  #@param ["", "sim", "não"]
#@markdown **[E-S08]** Participação de mulheres — % \*
participacao_mulheres_pct = ""  #@param {type:"string"}
#@markdown **[E-S09]** Relação com a comunidade \*
relacao_comunidade = ""  #@param ["", "inexistente", "parcial", "consolidada"]
#@markdown **[E-S10]** Canais formais de comunicação — sim/não \*
canais_formais_comunicacao = ""  #@param ["", "sim", "não"]
#@markdown **[E-S11]** Reuniões com a comunidade — por ano \*
reunioes_comunidade_ano = ""  #@param {type:"string"}
#@markdown **[E-S12]** Conflitos registrados — por ano \*
conflitos_registrados_ano = ""  #@param {type:"string"}
#@markdown **[E-G01]** Estatuto social registrado — sim/não \*
gov_estatuto_registrado = ""  #@param ["", "sim", "não"]
#@markdown **[E-G02]** Assembleias regulares — sim/não \*
gov_assembleias_regulares = ""  #@param ["", "sim", "não"]
#@markdown **[E-G03]** Conselho fiscal atuante — sim/não \*
gov_conselho_fiscal = ""  #@param ["", "sim", "não"]
#@markdown **[E-G04]** Prestação de contas pública — sim/não \*
gov_prestacao_contas_publica = ""  #@param ["", "sim", "não"]
#@markdown **[E-G05]** Plano de negócios aprovado — sim/não \*
gov_plano_negocios = ""  #@param ["", "sim", "não"]
#@markdown **[E-G06]** Registro auditável das premissas — sim/não \*
gov_registro_premissas_auditavel = ""  #@param ["", "sim", "não"]
#@markdown **[E-G07]** Canal de denúncia — sim/não \*
gov_canal_denuncia = ""  #@param ["", "sim", "não"]

In [ ]:
#@title 2.6 Referências Lean-Green (boas práticas declaradas) { display-mode: "form" }
#@markdown Valores de referência usados para quantificar desperdícios evitáveis (FCR, mortalidade e energia).
#@markdown ---
#@markdown **[E-L01]** FCR de referência (boa prática) — kg/kg \*
fcr_referencia = ""  #@param {type:"string"}
#@markdown **[E-L02]** Mortalidade de referência — % \*
mortalidade_referencia_pct = ""  #@param {type:"string"}
#@markdown **[E-L03]** Consumo de energia de referência — kWh/kg \*
consumo_kwh_kg_referencia = ""  #@param {type:"string"}

In [ ]:
#@title 2.7 Distribuições de incerteza para o Monte Carlo { display-mode: "form" }
#@markdown Informe `mín; máx` (a moda é o valor determinístico informado nas células anteriores) ou
#@markdown `mín; moda; máx` (a moda deve coincidir com o valor informado). Em branco = premissa determinística.
#@markdown Para os fatores de CAPEX e de custos fixos a moda é 1; para o desempenho de crescimento, 100.
#@markdown Informe ao menos uma distribuição. ---
#@markdown **[E-D01]** Preço de venda — R$/kg
dist_preco_venda_kg = ""  #@param {type:"string"}
#@markdown **[E-D02]** Custo da ração — R$/kg
dist_custo_racao_kg = ""  #@param {type:"string"}
#@markdown **[E-D03]** Custo do milheiro de juvenis — R$/mil
dist_custo_alevino_milheiro = ""  #@param {type:"string"}
#@markdown **[E-D04]** Tarifa de energia — R$/kWh
dist_tarifa_energia_kwh = ""  #@param {type:"string"}
#@markdown **[E-D05]** Fator de custos fixos (moda = 1) — fator
dist_custos_fixos_fator = ""  #@param {type:"string"}
#@markdown **[E-D06]** Fator de CAPEX (moda = 1) — fator
dist_capex_fator = ""  #@param {type:"string"}
#@markdown **[E-D07]** Conversão alimentar (FCR) — kg/kg
dist_fcr = ""  #@param {type:"string"}
#@markdown **[E-D08]** Mortalidade — %
dist_mortalidade_pct = ""  #@param {type:"string"}
#@markdown **[E-D09]** Desempenho de crescimento (moda = 100) — %
dist_desempenho_crescimento_pct = ""  #@param {type:"string"}

In [ ]:
#@title 2.8 Alternativas de projeto para comparação por TOPSIS (opcional) { display-mode: "form" }
#@markdown A alternativa A é o projeto informado acima. Para B e C, informe um nome, as alterações
#@markdown no formato `variável = valor; variável = valor` (nomes dos campos dos formulários, por exemplo
#@markdown `fonte_energia = solar; fracao_renovavel_pct = 90`) e, se houver, o CAPEX adicional no formato
#@markdown `descrição; valor; vida útil | descrição; valor; vida útil`. Em branco = alternativa não avaliada.
#@markdown ---
#@markdown **[E-XB]** Alternativa B
alt_B_nome = ""  #@param {type:"string"}
alt_B_alteracoes = ""  #@param {type:"string"}
alt_B_capex_adicional = ""  #@param {type:"string"}
#@markdown **[E-XC]** Alternativa C
alt_C_nome = ""  #@param {type:"string"}
alt_C_alteracoes = ""  #@param {type:"string"}
alt_C_capex_adicional = ""  #@param {type:"string"}

In [ ]:
#@title 2.9 Origem das premissas (rastreabilidade, Seção 3.6) { display-mode: "form" }
#@markdown Formato: `variável = categoria (referência); ...`, por exemplo
#@markdown `preco_venda_kg = cotação (CEASA-RJ, jun. 2026); fcr = parâmetro bibliográfico (Kubitza, 2017)`.
#@markdown Categorias: dado histórico; dado secundário; cotação; valor normativo; parâmetro bibliográfico; empreendimento análogo; especialista; estimativa do autor.
#@markdown Premissas sem origem aparecem como "não informada" no Quadro de premissas. Na planilha .xlsx há
#@markdown uma coluna própria para a origem de cada premissa.
fontes = ""  #@param {type:"string"}

In [ ]:
#@title 2.10 Parâmetros do método (Capítulo 3) { display-mode: "form" }
#@markdown Valores documentados no Capítulo 3. Altere somente se a mudança for justificada no texto.
#@markdown ---
#@markdown **[E-M01]** Iterações do Monte Carlo — iterações
mc_iteracoes = "10000"  #@param {type:"string"}
#@markdown **[E-M02]** Semente do gerador pseudoaleatório
mc_semente = "20260612"  #@param {type:"string"}
#@markdown **[E-M03]** Nível de confiança — fração
mc_nivel_confianca = "0,95"  #@param {type:"string"}
#@markdown **[E-M04]** Método de ponderação
pesos_metodo = "preset"  #@param ["preset", "likert", "ahp"]
#@markdown **[E-M05]** Peso da dimensão técnica (preset) — fração
peso_tecnica = "0,25"  #@param {type:"string"}
#@markdown **[E-M06]** Peso da dimensão econômica (preset) — fração
peso_economica = "0,35"  #@param {type:"string"}
#@markdown **[E-M07]** Peso da dimensão ambiental (preset) — fração
peso_ambiental = "0,2"  #@param {type:"string"}
#@markdown **[E-M08]** Peso da dimensão social (preset) — fração
peso_social = "0,2"  #@param {type:"string"}
#@markdown **[E-M09]** Aplicar vetos não compensatórios — sim/não
aplicar_vetos = "sim"  #@param ["sim", "não"]
#@markdown **[E-M10]** LSO mínima — 0–100
lso_minimo = "40"  #@param {type:"string"}
#@markdown **[E-M11]** Limiar do índice para 'viável' — 0–1
limiar_indice_viavel = "0,6"  #@param {type:"string"}
#@markdown **[E-M12]** Limiar do índice para 'com ressalvas' — 0–1
limiar_indice_ressalvas = "0,5"  #@param {type:"string"}
#@markdown **[E-M13]** Probabilidade mínima de VPL > 0 — fração
prob_vpl_positivo_minima = "0,7"  #@param {type:"string"}

In [ ]:
#@title 2.11 Planilha de entradas em branco (somente no modo planilha) { display-mode: "form" }
if MODO_ENTRADA.startswith("planilha"):
    ent.gerar_planilha_entradas("EVTEAS_entradas.xlsx")
    print("Planilha gerada: preencha as abas Entradas, CAPEX, Distribuições e Alternativas e envie-a na Seção 3.")
    baixar("EVTEAS_entradas.xlsx")
else:
    print("Modo atual:", MODO_ENTRADA, "— esta célula não se aplica.")

## 3 Simulação

In [ ]:
#@title 3.1 Ler e validar as entradas { display-mode: "form" }
valores = ent.coletar_formulario(globals())
fontes_planilha = {}
if MODO_ENTRADA == "formulário":
    cfg, erros = ent.ler_formulario(valores)
elif MODO_ENTRADA.startswith("planilha"):
    v_planilha, fontes_planilha = ent.valores_de_planilha(enviar(".xlsx", ARQUIVO_ENTRADAS))
    valores.update({k: v for k, v in v_planilha.items() if str(v).strip()})
    cfg, erros = ent.ler_formulario(v_planilha)
    cfg.fontes.update(fontes_planilha)
elif MODO_ENTRADA.startswith("arquivo"):
    cfg = ev.carregar_config(enviar(".json", ARQUIVO_ENTRADAS))
    erros = ev.validar_config(cfg)
else:
    cfg = ev.wizard_evteas()
    erros = []
if erros:
    display(HTML("<b>Entradas pendentes ou inválidas</b><ul>" + "".join(f"<li>{e}</li>" for e in erros) + "</ul>"))
    raise Pendencia(f"{len(erros)} entrada(s) em branco ou inválida(s). Corrija-as na Seção 2 e execute esta célula novamente.")
alternativas_cfg = ent.alternativas_de_formulario(cfg, valores)
print(f"Entradas válidas. Alternativas declaradas: {', '.join(alternativas_cfg)}.")
display(ent.tabela_entradas(cfg))

In [ ]:
#@title 3.2 Executar o estudo { display-mode: "form" }
resultado = ev.executar_evteas(cfg)
alternativas = ev.comparar_alternativas(alternativas_cfg) if len(alternativas_cfg) > 1 else None
print(ev.resumo_executivo(resultado, cfg))
print(f"\nTempo de execução: {resultado['tempo_execucao_s']:.1f} s")

In [ ]:
#@title 3.3 Tabelas de resultados (formato da dissertação) { display-mode: "form" }
for nome, tabela in ev.tabelas_dissertacao(resultado, cfg, alternativas).items():
    display(Markdown(f"**{nome}**"))
    display(tabela)
display(Markdown("**Decisão**"))
print(resultado["decisao"])

In [ ]:
#@title 3.4 Figuras { display-mode: "form" }
figuras = ev.gerar_figuras(resultado, cfg, "saidas/figuras", alternativas, mostrar=True)

## 4 Verificação

In [ ]:
#@title 4.1 Invariantes desta execução e testes automatizados { display-mode: "form" }
EXECUTAR_TESTES = True  #@param {type:"boolean"}
display(resultado["vv"]["tabela"])
print(f"Invariantes aprovados: {resultado['vv']['aprovadas']} de {resultado['vv']['total']}")
testes = None
if EXECUTAR_TESTES:
    print("Executando os testes automatizados (dados sintéticos de teste, não são dados do estudo)...")
    testes = ev.testes_automatizados(rapido=True)
    display(testes)
    print(f"Testes aprovados: {int(testes['Aprovado'].sum())} de {len(testes)}")

## 5 Marcadores para o texto da dissertação

Cada marcador do texto (`[E-xx]` = entrada informada; `[R-xx]` = resultado da simulação) recebe o valor
da coluna **Valor**. Os marcadores terminados em `F` trazem a origem declarada da entrada. Os textos
`R-TXTxx` são sugestões geradas por regras explícitas e devem ser revisados antes do uso.

In [ ]:
#@title 5.1 Tabela de marcadores { display-mode: "form" }
SECAO = "todas"  #@param ["todas", "4.2", "4.3", "4.4", "4.5", "4.6", "4.7", "4.8", "4.9", "4.10", "4.11", "4.x"]
marcadores = ev.marcadores(resultado, cfg, alternativas, testes, alternativas_cfg)
display(marcadores if SECAO == "todas" else marcadores[marcadores["Seção"] == SECAO])

## 6 Exportar as saídas

In [ ]:
#@title 6.1 Gerar e baixar o pacote de saídas (.zip) { display-mode: "form" }
arquivos = ev.exportar_tudo(resultado, cfg, "saidas", figuras=True, alternativas=alternativas, testes=testes,
                            configs_alternativas=alternativas_cfg)
for k, v in arquivos.items():
    print(f"{k:>18}: {v}")
baixar(arquivos["zip"])

## 7 Preencher o .docx da dissertação (opcional)

Envie o `.docx` que contém os marcadores `[E-xx]`, `[R-xx]` e `[FIG-...]`. O EVTEAS-Py troca cada marcador pelo
valor desta execução (mantendo a formatação do trecho), insere as figuras e devolve uma cópia preenchida.
Depois, no Word, atualize o sumário e as listas (Ctrl+A, F9).

In [ ]:
#@title 7.1 Preencher marcadores no .docx { display-mode: "form" }
ARQUIVO_DOCX = ""  #@param {type:"string"}
entrada_docx = enviar(".docx", ARQUIVO_DOCX)
saida_docx = os.path.splitext(os.path.basename(entrada_docx))[0] + "_preenchido.docx"
rel_docx = ev.preencher_docx(entrada_docx, saida_docx, marcadores, figuras)
print(rel_docx)
baixar(saida_docx)

## 8 Pesos Likert dos especialistas (somente após os questionários)

Preencha com as notas de 1 a 5 atribuídas pelos especialistas a cada KPI no questionário de validação
(Apêndice C, bloco de relevância dos indicadores). O avaliador 2 é opcional. A célula reexecuta o estudo com
os pesos Likert e gera os marcadores `[R-L01]` a `[R-L05]`, que comparam o resultado com o preset.

In [ ]:
#@title 8.1 Notas Likert por KPI (1 = nada relevante; 5 = extremamente relevante) { display-mode: "form" }
AVALIADOR_1 = ""  #@param {type:"string"}
AVALIADOR_2 = ""  #@param {type:"string"}
#@markdown ---
#@markdown **OEE** (dimensão tecnica)
likert1_oee = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_oee = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **FCR** (dimensão tecnica)
likert1_fcr = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_fcr = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Produtividade relativa** (dimensão tecnica)
likert1_produtividade_relativa = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_produtividade_relativa = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **VPL/investimento** (dimensão economica)
likert1_vpl_investimento = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_vpl_investimento = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **ROI anual** (dimensão economica)
likert1_roi = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_roi = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Margem de segurança** (dimensão economica)
likert1_margem_seguranca = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_margem_seguranca = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **PH azul (m³/t)** (dimensão ambiental)
likert1_ph_azul = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_ph_azul = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **PH cinza (m³/t)** (dimensão ambiental)
likert1_ph_cinza = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_ph_cinza = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Ecoeficiência (R$/kgCO2e)** (dimensão ambiental)
likert1_ecoeficiencia = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_ecoeficiencia = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Intensidade de carbono (kgCO2e/kg)** (dimensão ambiental)
likert1_intensidade_carbono = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_intensidade_carbono = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Conformidade ambiental (0–10)** (dimensão ambiental)
likert1_conformidade = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_conformidade = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **LSO (0–100)** (dimensão social)
likert1_lso = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_lso = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **RVL (%)** (dimensão social)
likert1_rvl = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_rvl = ""  #@param ["", "1", "2", "3", "4", "5"]
#@markdown **Radar Social** (dimensão social)
likert1_radar_social = ""  #@param ["", "1", "2", "3", "4", "5"]
likert2_radar_social = ""  #@param ["", "1", "2", "3", "4", "5"]

In [ ]:
#@title 8.2 Reexecutar com os pesos Likert { display-mode: "form" }
from evteas_py.config import copiar
from evteas_py.ponderacao import KPIS
escores = {}
for n, nome in ((1, AVALIADOR_1), (2, AVALIADOR_2)):
    notas = {k: globals().get(f"likert{n}_{k}", "") for k in KPIS}
    if not nome.strip() and not any(notas.values()):
        continue
    faltam = [KPIS[k][1] for k, v in notas.items() if not str(v).strip()]
    if faltam or not nome.strip():
        raise Pendencia(f"Avaliador {n}: informe o nome e as notas de todos os KPIs (faltam: {', '.join(faltam) or 'nome'}).")
    escores[nome.strip()] = {k: int(v) for k, v in notas.items()}
if not escores:
    raise Pendencia("Preencha as notas de ao menos um avaliador na célula 8.1 (após os questionários).")
cfg_likert = copiar(cfg)
cfg_likert.pesos.metodo = "likert"
cfg_likert.pesos.likert = escores
resultado_likert = ev.executar_evteas(cfg_likert)
comparacao_pesos = rel.comparar_pesos(resultado, resultado_likert)
display(comparacao_pesos)
marcadores = pd.concat([ev.marcadores(resultado, cfg, alternativas, testes, alternativas_cfg), comparacao_pesos],
                       ignore_index=True)
comparacao_pesos.to_csv("saidas/EVTEAS_marcadores_likert.csv", sep=";", index=False, encoding="utf-8-sig")
print("Marcadores [R-Lxx] acrescentados à tabela de marcadores (use a Seção 7 para preencher o .docx).")

## 9 Ferramenta de apoio: análise de preços de mercado (opcional)

In [ ]:
#@title 9.1 Percentis de cotações e curva de demanda { display-mode: "form" }
#@markdown Envie um .csv com a coluna `preco` (R$/kg) e, opcionalmente, `quantidade` (kg). Os percentis P10 e P90
#@markdown orientam o mínimo e o máximo da distribuição do preço (célula 2.7).
ARQUIVO_PRECOS = ""  #@param {type:"string"}
CUSTO_VARIAVEL_KG = ""  #@param {type:"string"}
precos = pd.read_csv(enviar(".csv", ARQUIVO_PRECOS), sep=None, engine="python", decimal=",")
analise = ev.analise_precos(precos, ent.ler_numero(CUSTO_VARIAVEL_KG) if CUSTO_VARIAVEL_KG.strip() else None)
display(pd.Series(analise["sugestoes"], name="R$/kg"))
print("Distribuição sugerida para o preço (P10; P90):", analise["distribuicao_sugerida"])
if analise["preco_otimo"] is not None:
    print(f"Preço que maximiza a margem de contribuição: R$ {ent.fmt_auto(round(analise['preco_otimo'], 2))}/kg")